In [1]:
!pip install -r ../requirements.txt

ERROR: Could not find a version that satisfies the requirement collections (from versions: none)
ERROR: No matching distribution found for collections


## Data processing:

In [2]:
"""
Blood Glucose Forecasting Data Processing
Trains on OhioT1DM, tests on BrisT1D with shared features only
No data leakage: scalers fitted only on training data
Fixed: Smoother only applied to CGM data
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import sys
import subprocess

from smoother.smooth_SMBG_data import smooth_smbg_data

# ============================================================================
# CONFIGURATION
# ============================================================================

# Paths
TRAIN_PATH = "../data/OhioT1DM.csv"
TEST_PATH = "../data/BrisT1D.csv"

# Parameters
N_OUT = 6  # 30-min horizon (6 × 5min)
N_IN = N_OUT * 2  # 60-min history (12 × 5min)
RESAMPLE_RULE = '5T'  # 5-minute intervals
RANDOM_STATE = 42

# SHARED FEATURES ONLY (intersection of both datasets)
# Base features: present in both datasets
SHARED_FEATURES = ['carbs', 'bolus', 'heartrate', 'steps']

# Computed features: derivatives (can be computed from both datasets)
COMPUTED_FEATURES = ['CGM_derivative', 'carbs_derivative']

# All features for model input
MODEL_FEATURES = SHARED_FEATURES + COMPUTED_FEATURES

print(f"Base shared features: {SHARED_FEATURES}")
print(f"Computed features: {COMPUTED_FEATURES}")
print(f"Total model features: {MODEL_FEATURES}")
print(f"History: {N_IN}×5min = {N_IN*5}min")
print(f"Horizon: {N_OUT}×5min = {N_OUT*5}min\n")

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def smooth_with_kalman(dates, values, subject_id):
    """
    Apply Kalman smoothing with fallback to interpolation+median filter
    Returns: pandas Series with smoothed values at original timestamps
    NOTE: This should ONLY be used for CGM data
    """
    try:
        result = smooth_smbg_data(dates, values, outlier_removal=1, dynamic_model=2)
        y_smooth = result.get('y_smoothed_at_tout', result.get('y_smoothed', None))
        tout = result.get('tout', None)
        
        if y_smooth is not None:
            idx = pd.to_datetime(tout) if tout is not None else pd.to_datetime(dates)
            return pd.Series(y_smooth, index=idx)
        else:
            raise ValueError("No smoothed output")
            
    except Exception as e:
        print(f"  [Kalman failed for subject {subject_id}] Using fallback: {e}")
        # Fallback: interpolation + median filter
        s = pd.Series(values, index=pd.to_datetime(dates))
        s = s.interpolate(method='time')
        s = s.rolling(window=3, min_periods=1, center=True).median()
        return s


def process_subject(df_subject):
    """
    Process a single subject's data:
    1. Create regular 5-min grid
    2. Smooth CGM with Kalman filter (CGM ONLY)
    3. Interpolate all other features to grid
    4. Return DataFrame with all features at regular intervals
    """
    subject_id = df_subject['id'].iloc[0]
    
    # Create regular time grid
    t_min = df_subject['date'].min().floor('5T')
    t_max = df_subject['date'].max().ceil('5T')
    grid = pd.date_range(t_min, t_max, freq=RESAMPLE_RULE)
    
    # Smooth CGM ONLY (not carbs or other features)
    cgm_smooth = smooth_with_kalman(
        df_subject['date'].values,
        df_subject['CGM'].values,
        subject_id
    )
    
    # Reindex CGM to regular grid
    cgm_reg = cgm_smooth.reindex(cgm_smooth.index.union(grid)).sort_index()
    cgm_reg = cgm_reg.interpolate(method='time').reindex(grid)
    cgm_reg = cgm_reg.fillna(method='ffill').fillna(method='bfill')
    cgm_reg = cgm_reg.fillna(cgm_smooth.median())
    
    # Process ALL other features (including carbs) with simple interpolation
    df_idx = df_subject.set_index('date')
    features_df = df_idx.reindex(df_idx.index.union(grid))[SHARED_FEATURES].sort_index()
    features_df = features_df.interpolate(method='time').reindex(grid)
    features_df = features_df.fillna(method='ffill').fillna(method='bfill').fillna(0)
    
    # Assemble output
    output = pd.DataFrame(index=grid)
    output['CGM_smoothed'] = cgm_reg.values
    for col in SHARED_FEATURES:
        output[col] = features_df[col].values
    
    # Compute derivatives (shared computed features for both datasets)
    output['CGM_derivative'] = output['CGM_smoothed'].diff().fillna(0)
    output['carbs_derivative'] = output['carbs'].diff().fillna(0)
    
    output = output.reset_index().rename(columns={'index': 'date'})
    return output


def create_sequences(df_subject, n_in, n_out, cgm_scaler, feature_scaler):
    """
    Create input-output sequences from processed subject data
    Returns: X (n_samples, n_in, n_features+1), y (n_samples, n_out, 1)
    Features include: base shared features + computed derivatives
    """
    cgm = df_subject['CGM_smoothed'].values.reshape(-1, 1)
    features = df_subject[MODEL_FEATURES].values  # Includes derivatives
    
    # Scale using provided scalers
    cgm_scaled = cgm_scaler.transform(cgm).flatten()
    features_scaled = feature_scaler.transform(features)
    
    X_list, y_list = [], []
    L = len(df_subject)
    
    for i in range(n_in, L - n_out + 1):
        # Input: combine scaled CGM with scaled features (including derivatives)
        x_in = np.column_stack([
            cgm_scaled[i-n_in:i],
            features_scaled[i-n_in:i]
        ])
        # Output: future CGM values
        y_out = cgm_scaled[i:i+n_out].reshape(n_out, 1)
        
        X_list.append(x_in)
        y_list.append(y_out)
    
    if len(X_list) == 0:
        return np.zeros((0, n_in, 1+len(MODEL_FEATURES))), np.zeros((0, n_out, 1))
    
    return np.stack(X_list), np.stack(y_list)


# ============================================================================
# STEP 1: LOAD AND PROCESS TRAINING DATA (OhioT1DM)
# ============================================================================

print("="*70)
print("STEP 1: Processing Training Data (OhioT1DM)")
print("="*70)

df_train = pd.read_csv(TRAIN_PATH)
df_train['date'] = pd.to_datetime(df_train['date'])
df_train = df_train.sort_values(['id', 'date']).reset_index(drop=True)

# Rename columns to match shared feature names
if 'insulin' in df_train.columns:
    df_train = df_train.rename(columns={'insulin': 'bolus'})

# Process each subject
print("\nProcessing training subjects...")
train_subjects = {}
for subject_id, group in df_train.groupby('id'):
    print(f"  Processing subject {subject_id}...")
    train_subjects[subject_id] = process_subject(group)

# Split subjects into train/validation (80/20) BEFORE fitting scalers
subject_ids = sorted(train_subjects.keys())
train_ids, val_ids = train_test_split(
    subject_ids, 
    test_size=0.2, 
    random_state=RANDOM_STATE
)

print(f"\nSubject split:")
print(f"  Total: {len(subject_ids)}")
print(f"  Train: {len(train_ids)}")
print(f"  Validation: {len(val_ids)}")


# ============================================================================
# STEP 2: FIT SCALERS ON TRAINING SUBJECTS ONLY (NO LEAKAGE)
# ============================================================================

print("\n" + "="*70)
print("STEP 2: Fitting Scalers (Training Subjects Only)")
print("="*70)

# Fit CGM scaler
train_cgm = np.concatenate([
    train_subjects[sid]['CGM_smoothed'].values 
    for sid in train_ids
])
cgm_scaler = MinMaxScaler(feature_range=(0, 1))
cgm_scaler.fit(train_cgm.reshape(-1, 1))
print(f"✓ CGM scaler fitted on {len(train_cgm)} training samples")

# Fit feature scaler (includes base features + derivatives)
train_features = np.vstack([
    train_subjects[sid][MODEL_FEATURES].values 
    for sid in train_ids
])
feature_scaler = StandardScaler()
feature_scaler.fit(train_features)
print(f"✓ Feature scaler fitted on {len(train_features)} training samples")
print(f"  Feature means: {dict(zip(MODEL_FEATURES, feature_scaler.mean_))}")


# ============================================================================
# STEP 3: CREATE SEQUENCES FOR TRAINING AND VALIDATION
# ============================================================================

print("\n" + "="*70)
print("STEP 3: Creating Sequences")
print("="*70)

# Training sequences
train_sequences = {}
for sid in train_ids:
    X, y = create_sequences(train_subjects[sid], N_IN, N_OUT, cgm_scaler, feature_scaler)
    train_sequences[sid] = {'X': X, 'y': y}

# Validation sequences
val_sequences = {}
for sid in val_ids:
    X, y = create_sequences(train_subjects[sid], N_IN, N_OUT, cgm_scaler, feature_scaler)
    val_sequences[sid] = {'X': X, 'y': y}

# Aggregate training data
X_train = np.concatenate([seq['X'] for seq in train_sequences.values() if len(seq['X']) > 0])
y_train = np.concatenate([seq['y'] for seq in train_sequences.values() if len(seq['y']) > 0])

# Aggregate validation data
X_val = np.concatenate([seq['X'] for seq in val_sequences.values() if len(seq['X']) > 0])
y_val = np.concatenate([seq['y'] for seq in val_sequences.values() if len(seq['y']) > 0])

print(f"\nTraining data:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"\nValidation data:")
print(f"  X_val shape: {X_val.shape}")
print(f"  y_val shape: {y_val.shape}")


# ============================================================================
# STEP 4: LOAD AND PROCESS TEST DATA (BrisT1D)
# ============================================================================

print("\n" + "="*70)
print("STEP 4: Processing Test Data (BrisT1D)")
print("="*70)

df_test = pd.read_csv(TEST_PATH)
df_test['date'] = pd.to_datetime(df_test['date'])
df_test = df_test.sort_values(['id', 'date']).reset_index(drop=True)

# Map columns to shared feature names
column_mapping = {
    'insulin': 'bolus',
    'heart_rate': 'heartrate',
    'hr': 'heartrate',
}
df_test = df_test.rename(columns=column_mapping)

# Ensure all shared features exist (fill with NaN if missing)
for feature in SHARED_FEATURES:
    if feature not in df_test.columns:
        print(f"  Warning: '{feature}' not in test data, will impute with training mean")
        df_test[feature] = np.nan

# Process test subjects
print("\nProcessing test subjects...")
test_subjects = {}
for subject_id, group in df_test.groupby('id'):
    print(f"  Processing subject {subject_id}...")
    processed = process_subject(group)
    
    # Impute missing base features with training means
    # Note: derivatives are computed, so we only need to impute base features
    for i, feature in enumerate(SHARED_FEATURES):
        if processed[feature].isna().any():
            impute_value = feature_scaler.mean_[i]
            processed[feature] = processed[feature].fillna(impute_value)
            print(f"    Imputed {feature} with training mean: {impute_value:.3f}")
    
    test_subjects[subject_id] = processed


# ============================================================================
# STEP 5: CREATE TEST SEQUENCES (USING FITTED SCALERS)
# ============================================================================

print("\n" + "="*70)
print("STEP 5: Creating Test Sequences (Using Fitted Scalers)")
print("="*70)

# Create test sequences
test_sequences = {}
for sid in test_subjects.keys():
    X, y = create_sequences(test_subjects[sid], N_IN, N_OUT, cgm_scaler, feature_scaler)
    test_sequences[sid] = {'X': X, 'y': y}

# Aggregate test data
X_test = np.concatenate([seq['X'] for seq in test_sequences.values() if len(seq['X']) > 0])
y_test = np.concatenate([seq['y'] for seq in test_sequences.values() if len(seq['y']) > 0])

print(f"\nTest data:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")


# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*70)
print("PROCESSING COMPLETE - NO DATA LEAKAGE")
print("="*70)
print(f"""
Summary:
  ✓ Shared features: {SHARED_FEATURES}
  ✓ Kalman smoother applied ONLY to CGM data (not carbs)
  ✓ Scalers fitted ONLY on training subjects from OhioT1DM
  ✓ Same scalers applied to validation and test data
  ✓ No information from validation/test used in scaling
  
Available variables:
  - X_train, y_train: Training data from OhioT1DM
  - X_val, y_val: Validation data from OhioT1DM
  - X_test, y_test: Test data from BrisT1D
  - cgm_scaler: MinMaxScaler fitted on training CGM
  - feature_scaler: StandardScaler fitted on training features
  - train_sequences: Per-subject training sequences (dict)
  - val_sequences: Per-subject validation sequences (dict)
  - test_sequences: Per-subject test sequences (dict)

Ready for model training and evaluation!
""")

Base shared features: ['carbs', 'bolus', 'heartrate', 'steps']
Computed features: ['CGM_derivative', 'carbs_derivative']
Total model features: ['carbs', 'bolus', 'heartrate', 'steps', 'CGM_derivative', 'carbs_derivative']
History: 12×5min = 60min
Horizon: 6×5min = 30min

STEP 1: Processing Training Data (OhioT1DM)

Processing training subjects...
  Processing subject 540...
Autodetected mg/dL as unit


C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:88: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  t_min = df_subject['date'].min().floor('5T')
C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:89: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  t_max = df_subject['date'].max().ceil('5T')
C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:90: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  grid = pd.date_range(t_min, t_max, freq=RESAMPLE_RULE)


Smoother flagged measurement 1743 as outlier: t = 10045.0, y = 8.769008769008769 [mmol/L].
Smoother flagged measurement 8689 as outlier: t = 47370.0, y = 6.382506382506382 [mmol/L].
Smoother flagged measurement 14727 as outlier: t = 80995.0, y = 16.87201687201687 [mmol/L].
Smoother flagged measurement 14728 as outlier: t = 81000.0, y = 12.654012654012654 [mmol/L].
Smoother needs a second pass due to outliers detected. Total # outliers in input data: 4


C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:102: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  cgm_reg = cgm_reg.fillna(method='ffill').fillna(method='bfill')
C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:109: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  features_df = features_df.fillna(method='ffill').fillna(method='bfill').fillna(0)
C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:88: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  t_min = df_subject['date'].min().floor('5T')
C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:89: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  t_max = df_subject['date'].max().ceil('5T')
C:\Users\msbdj\AppData

  Processing subject 544...
Autodetected mg/dL as unit
Smoother flagged measurement 975 as outlier: t = 5055.0, y = 4.828504828504828 [mmol/L].
Smoother flagged measurement 1563 as outlier: t = 7995.0, y = 15.54001554001554 [mmol/L].
Smoother flagged measurement 1564 as outlier: t = 8000.0, y = 14.263514263514264 [mmol/L].
Smoother flagged measurement 1566 as outlier: t = 8010.0, y = 9.268509268509268 [mmol/L].
Smoother flagged measurement 1574 as outlier: t = 8050.0, y = 13.32001332001332 [mmol/L].
Smoother flagged measurement 1575 as outlier: t = 8055.0, y = 12.543012543012543 [mmol/L].
Smoother flagged measurement 1577 as outlier: t = 8065.0, y = 8.935508935508935 [mmol/L].
Smoother flagged measurement 1608 as outlier: t = 8220.0, y = 9.213009213009213 [mmol/L].
Smoother flagged measurement 1610 as outlier: t = 8230.0, y = 6.715506715506716 [mmol/L].
Smoother flagged measurement 1615 as outlier: t = 8255.0, y = 9.046509046509046 [mmol/L].
Smoother flagged measurement 1617 as outlier

...

  Processing subject P03...
Autodetected mg/dL as unit
Smoother flagged measurement 2988 as outlier: t = 15130.0, y = 8.0 [mmol/L].
Smoother flagged measurement 4538 as outlier: t = 23015.0, y = 8.4 [mmol/L].
Smoother flagged measurement 4548 as outlier: t = 23065.0, y = 10.5 [mmol/L].
Smoother flagged measurement 4638 as outlier: t = 23515.0, y = 7.700000000000001 [mmol/L].
Smoother flagged measurement 7413 as outlier: t = 37530.0, y = 8.5 [mmol/L].
Smoother flagged measurement 10325 as outlier: t = 52220.0, y = 8.7 [mmol/L].
Smoother flagged measurement 10469 as outlier: t = 52940.0, y = 3.4 [mmol/L].
Smoother flagged measurement 11092 as outlier: t = 56055.0, y = 5.1 [mmol/L].
Smoother flagged measurement 11200 as outlier: t = 56595.0, y = 6.0 [mmol/L].
Smoother flagged measurement 22980 as outlier: t = 116185.0, y = 4.9 [mmol/L].
Smoother needs a second pass due to outliers detected. Total # outliers in input data: 10


C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:102: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  cgm_reg = cgm_reg.fillna(method='ffill').fillna(method='bfill')
C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:109: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  features_df = features_df.fillna(method='ffill').fillna(method='bfill').fillna(0)
C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:88: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  t_min = df_subject['date'].min().floor('5T')
C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:89: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  t_max = df_subject['date'].max().ceil('5T')
C:\Users\msbdj\AppData

  Processing subject P04...
Autodetected mg/dL as unit
Smoother flagged measurement 2228 as outlier: t = 11275.0, y = 7.9 [mmol/L].
Smoother flagged measurement 4636 as outlier: t = 23370.0, y = 4.3 [mmol/L].
Smoother flagged measurement 4784 as outlier: t = 24110.0, y = 3.6 [mmol/L].
Smoother flagged measurement 4791 as outlier: t = 24280.0, y = 5.5 [mmol/L].
Smoother flagged measurement 4793 as outlier: t = 24290.0, y = 2.8 [mmol/L].
Smoother flagged measurement 4806 as outlier: t = 24365.0, y = 6.3 [mmol/L].
Smoother flagged measurement 4856 as outlier: t = 24665.0, y = 3.2 [mmol/L].
Smoother flagged measurement 4857 as outlier: t = 24675.0, y = 8.0 [mmol/L].
Smoother flagged measurement 7717 as outlier: t = 39130.0, y = 6.6 [mmol/L].
Smoother flagged measurement 7735 as outlier: t = 39220.0, y = 6.8 [mmol/L].
Smoother flagged measurement 10521 as outlier: t = 53280.0, y = 4.4 [mmol/L].
Smoother flagged measurement 10522 as outlier: t = 53285.0, y = 5.3 [mmol/L].
Smoother flagged me

C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:102: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  cgm_reg = cgm_reg.fillna(method='ffill').fillna(method='bfill')
C:\Users\msbdj\AppData\Local\Temp\ipykernel_420\1107450577.py:109: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  features_df = features_df.fillna(method='ffill').fillna(method='bfill').fillna(0)



STEP 5: Creating Test Sequences (Using Fitted Scalers)

Test data:
  X_test shape: (209715, 12, 7)
  y_test shape: (209715, 6, 1)

PROCESSING COMPLETE - NO DATA LEAKAGE

Summary:
  ✓ Shared features: ['carbs', 'bolus', 'heartrate', 'steps']
  ✓ Kalman smoother applied ONLY to CGM data (not carbs)
  ✓ Scalers fitted ONLY on training subjects from OhioT1DM
  ✓ Same scalers applied to validation and test data
  ✓ No information from validation/test used in scaling
  
Available variables:
  - X_train, y_train: Training data from OhioT1DM
  - X_val, y_val: Validation data from OhioT1DM
  - X_test, y_test: Test data from BrisT1D
  - cgm_scaler: MinMaxScaler fitted on training CGM
  - feature_scaler: StandardScaler fitted on training features
  - train_sequences: Per-subject training sequences (dict)
  - val_sequences: Per-subject validation sequences (dict)
  - test_sequences: Per-subject test sequences (dict)

Ready for model training and evaluation!



## Zero order - Persistance Model:

In [4]:
"""
Zero-Order (Persistence) Baseline Model
Predicts that future CGM values will remain constant at the last observed value
No training required - purely a baseline for comparison
"""

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ============================================================================
# PERSISTENCE MODEL FUNCTIONS
# ============================================================================

def persistence_predict(X):
    """
    Zero-order persistence prediction
    
    Args:
        X: Input sequences (n_samples, n_in, n_features+1)
           First feature (X[:, :, 0]) is scaled CGM history
    
    Returns:
        Predictions (n_samples, n_out, 1) - last CGM repeated N_OUT times
    """
    # Extract last observed CGM value from each sequence
    last_cgm = X[:, -1, 0]  # Shape: (n_samples,)
    
    # Repeat this value for all future timesteps
    predictions = np.tile(
        last_cgm.reshape(-1, 1, 1),  # Reshape to (n_samples, 1, 1)
        (1, N_OUT, 1)                 # Tile to (n_samples, N_OUT, 1)
    )
    
    return predictions


def evaluate_persistence(sequences_dict, dataset_name="Dataset"):
    """
    Evaluate persistence model on a set of sequences
    
    Args:
        sequences_dict: Dictionary {subject_id: {'X': X_array, 'y': y_array}}
        dataset_name: Name for printing results
    
    Returns:
        Dictionary with aggregated metrics
    """
    all_predictions = []
    all_targets = []
    per_subject_metrics = {}
    
    for subject_id, data in sequences_dict.items():
        X, y = data['X'], data['y']
        
        if len(X) == 0:
            continue
        
        # Make predictions
        y_pred = persistence_predict(X)
        
        # Inverse transform to original CGM scale
        y_pred_orig = cgm_scaler.inverse_transform(y_pred.reshape(-1, 1)).reshape(y_pred.shape)
        y_orig = cgm_scaler.inverse_transform(y.reshape(-1, 1)).reshape(y.shape)
        
        # Calculate per-subject metrics
        rmse = np.sqrt(mean_squared_error(y_orig.flatten(), y_pred_orig.flatten()))
        mae = mean_absolute_error(y_orig.flatten(), y_pred_orig.flatten())
        
        per_subject_metrics[subject_id] = {
            'rmse': rmse,
            'mae': mae,
            'n_samples': len(X)
        }
        
        # Accumulate for overall metrics
        all_predictions.append(y_pred_orig)
        all_targets.append(y_orig)
    
    # Calculate overall metrics
    all_predictions = np.concatenate(all_predictions)
    all_targets = np.concatenate(all_targets)
    
    overall_rmse = np.sqrt(mean_squared_error(all_targets.flatten(), all_predictions.flatten()))
    overall_mae = mean_absolute_error(all_targets.flatten(), all_predictions.flatten())
    
    # Print results
    print(f"\n{dataset_name} Results:")
    print("="*70)
    print(f"Overall Metrics:")
    print(f"  RMSE: {overall_rmse:.2f} mg/dL")
    print(f"  MAE:  {overall_mae:.2f} mg/dL")
    
    print(f"\nPer-Subject Metrics:")
    for subject_id, metrics in sorted(per_subject_metrics.items()):
        print(f"  Subject {subject_id}: RMSE={metrics['rmse']:.2f}, MAE={metrics['mae']:.2f} "
              f"(n={metrics['n_samples']})")
    
    return {
        'overall_rmse': overall_rmse,
        'overall_mae': overall_mae,
        'per_subject': per_subject_metrics,
        'predictions': all_predictions,
        'targets': all_targets
    }


# ============================================================================
# EVALUATE ON VALIDATION SET (OhioT1DM)
# ============================================================================

print("="*70)
print("ZERO-ORDER PERSISTENCE MODEL EVALUATION")
print("="*70)
print(f"\nModel Description:")
print(f"  Predicts future CGM = last observed CGM")
print(f"  No training required - purely baseline")
print(f"  Prediction horizon: {N_OUT} steps ({N_OUT*5} minutes)")

val_results = evaluate_persistence(val_sequences, "Validation (OhioT1DM)")


# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*70)
print("SUMMARY - PERSISTENCE BASELINE")
print("="*70)
print(f"""
Validation Performance (OhioT1DM):
  RMSE: {val_results['overall_rmse']:.2f} mg/dL
  MAE:  {val_results['overall_mae']:.2f} mg/dL

This baseline represents the minimum performance threshold.
Any predictive model should significantly outperform these metrics.

Note: Test set (BrisT1D) reserved for final model evaluation.
""")

ZERO-ORDER PERSISTENCE MODEL EVALUATION

Model Description:
  Predicts future CGM = last observed CGM
  No training required - purely baseline
  Prediction horizon: 6 steps (30 minutes)

Validation (OhioT1DM) Results:
Overall Metrics:
  RMSE: 16.72 mg/dL
  MAE:  10.68 mg/dL

Per-Subject Metrics:
  Subject 540: RMSE=17.12, MAE=11.08 (n=16297)
  Subject 588: RMSE=15.80, MAE=10.19 (n=16111)
  Subject 591: RMSE=17.22, MAE=10.75 (n=15823)

SUMMARY - PERSISTENCE BASELINE

Validation Performance (OhioT1DM):
  RMSE: 16.72 mg/dL
  MAE:  10.68 mg/dL

This baseline represents the minimum performance threshold.
Any predictive model should significantly outperform these metrics.

Note: Test set (BrisT1D) reserved for final model evaluation.



## Vanilla-LSTM model without Fine-tuning:

In [5]:
"""
Simple Vanilla LSTM with Nested Cross-Validation for Blood Glucose Forecasting
Outer loop: K-fold CV for model evaluation
Inner loop: Validation split for early stopping
Single-phase training: RMSE loss only
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import KFold

# ============================================================================
# CONFIGURATION
# ============================================================================

# Model architecture
LSTM_UNITS_1 = 48
LSTM_UNITS_2 = 48
DROPOUT_RATE = 0.2
BATCH_SIZE = 32

# Training parameters
EPOCHS = 1
LEARNING_RATE = 0.001

# Nested CV parameters
N_OUTER_FOLDS = 2  # Outer loop for evaluation
VALIDATION_SPLIT = 0.2  # Inner loop for early stopping
RANDOM_STATE = 42

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

# Extract scaler parameters
MIN_CGM = float(cgm_scaler.data_min_[0])
MAX_CGM = float(cgm_scaler.data_max_[0])

def _to_mgdl(tensor_scaled):
    """Convert scaled [0,1] values back to mg/dL"""
    return tensor_scaled * (MAX_CGM - MIN_CGM) + MIN_CGM

@tf.function
def rmse_loss(y_true, y_pred):
    """Root Mean Squared Error"""
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-8)

def create_model(n_in, n_features, n_out):
    """Create LSTM model with given architecture"""
    tf.keras.backend.clear_session()
    
    model = Sequential([
        LSTM(LSTM_UNITS_1, return_sequences=True, input_shape=(n_in, n_features)),
        Dropout(DROPOUT_RATE),
        LSTM(LSTM_UNITS_2, return_sequences=False),
        Dropout(DROPOUT_RATE),
        Dense(n_out, activation='relu')
    ])
    return model

def train_model(model, X_tr, y_tr, X_v, y_v):
    """Single-phase training with RMSE loss"""
    
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=rmse_loss,
        metrics=['mae']
    )
    
    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, 
                          min_lr=1e-6, verbose=0),
        EarlyStopping(monitor='val_loss', patience=8, 
                      restore_best_weights=True, verbose=0)
    ]
    
    model.fit(
        X_tr, y_tr,
        validation_data=(X_v, y_v),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=0
    )
    
    return model

# ============================================================================
# NESTED CROSS-VALIDATION
# ============================================================================

print("="*70)
print("NESTED CROSS-VALIDATION")
print("="*70)
print(f"Outer folds: {N_OUTER_FOLDS}")
print(f"Validation split: {VALIDATION_SPLIT}")
print()

# Get dimensions
n_samples, n_in, n_features = X_train.shape
n_out = y_train.shape[1]

# Flatten targets
y_train_flat = y_train.reshape((y_train.shape[0], n_out))

# Storage for fold results
fold_results = []
fold_models = []

# Outer CV loop
kfold = KFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for fold_idx, (train_idx, test_idx) in enumerate(kfold.split(X_train)):
    print(f"{'='*70}")
    print(f"FOLD {fold_idx + 1}/{N_OUTER_FOLDS}")
    print(f"{'='*70}")
    
    # Split data for this fold
    X_fold_train = X_train[train_idx]
    y_fold_train = y_train_flat[train_idx]
    X_fold_test = X_train[test_idx]
    y_fold_test = y_train_flat[test_idx]
    
    # Inner validation split for early stopping
    val_size = int(len(X_fold_train) * VALIDATION_SPLIT)
    X_fold_val = X_fold_train[:val_size]
    y_fold_val = y_fold_train[:val_size]
    X_fold_train = X_fold_train[val_size:]
    y_fold_train = y_fold_train[val_size:]
    
    print(f"Train samples: {len(X_fold_train)}")
    print(f"Validation samples: {len(X_fold_val)}")
    print(f"Test samples: {len(X_fold_test)}")
    
    # Create and train model
    model = create_model(n_in, n_features, n_out)
    print("\nTraining model with RMSE loss...")
    model = train_model(model, X_fold_train, y_fold_train, 
                       X_fold_val, y_fold_val)
    
    # Evaluate on fold test set
    y_fold_pred = model.predict(X_fold_test, batch_size=64, verbose=0)
    
    # Convert to mg/dL
    y_fold_test_mgdl = _to_mgdl(tf.constant(y_fold_test)).numpy()
    y_fold_pred_mgdl = _to_mgdl(tf.constant(y_fold_pred)).numpy()
    
    # Calculate metrics
    fold_rmse = np.sqrt(np.mean((y_fold_test_mgdl - y_fold_pred_mgdl)**2))
    fold_mae = np.mean(np.abs(y_fold_test_mgdl - y_fold_pred_mgdl))
    
    print(f"\nFold {fold_idx + 1} Results (mg/dL):")
    print(f"  RMSE: {fold_rmse:.2f}")
    print(f"  MAE: {fold_mae:.2f}")
    
    # Store results
    fold_results.append({
        'fold': fold_idx + 1,
        'rmse': fold_rmse,
        'mae': fold_mae,
        'y_true': y_fold_test_mgdl,
        'y_pred': y_fold_pred_mgdl
    })
    fold_models.append(model)
    print()

# ============================================================================
# AGGREGATE RESULTS
# ============================================================================

print("="*70)
print("NESTED CV SUMMARY")
print("="*70)

rmse_scores = [r['rmse'] for r in fold_results]
mae_scores = [r['mae'] for r in fold_results]

print(f"\nRMSE across folds:")
print(f"  Mean: {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f}")
print(f"  Range: [{np.min(rmse_scores):.2f}, {np.max(rmse_scores):.2f}]")

print(f"\nMAE across folds:")
print(f"  Mean: {np.mean(mae_scores):.2f} ± {np.std(mae_scores):.2f}")
print(f"  Range: [{np.min(mae_scores):.2f}, {np.max(mae_scores):.2f}]")

# ============================================================================
# FINAL MODEL TRAINING (FULL TRAINING SET)
# ============================================================================

print("\n" + "="*70)
print("TRAINING FINAL MODEL ON FULL TRAINING SET")
print("="*70)

final_model = create_model(n_in, n_features, n_out)
y_val_flat = y_val.reshape((y_val.shape[0], n_out))

print("Training final model...")
final_model = train_model(final_model, X_train, y_train_flat, 
                          X_val, y_val_flat)

# Validation predictions
y_val_pred_flat = final_model.predict(X_val, batch_size=64, verbose=0)
y_val_pred = y_val_pred_flat.reshape((-1, n_out, 1))

y_val_mgdl = _to_mgdl(tf.constant(y_val)).numpy()
y_val_pred_mgdl = _to_mgdl(tf.constant(y_val_pred)).numpy()

val_rmse = np.sqrt(np.mean((y_val_mgdl - y_val_pred_mgdl)**2))
val_mae = np.mean(np.abs(y_val_mgdl - y_val_pred_mgdl))

print(f"\nFinal Model - Validation Metrics (mg/dL):")
print(f"  RMSE: {val_rmse:.2f}")
print(f"  MAE: {val_mae:.2f}")

# ============================================================================
# EXPORTS
# ============================================================================

lstm_model = final_model
cv_results = {
    'fold_results': fold_results,
    'fold_models': fold_models,
    'mean_rmse': np.mean(rmse_scores),
    'std_rmse': np.std(rmse_scores),
    'mean_mae': np.mean(mae_scores),
    'std_mae': np.std(mae_scores)
}

val_predictions = {
    'y_true': y_val,
    'y_pred': y_val_pred,
    'y_true_mgdl': y_val_mgdl,
    'y_pred_mgdl': y_val_pred_mgdl
}

print("\n" + "="*70)
print("NESTED CV COMPLETE!")
print("="*70)
print("""
Available variables:
  - lstm_model: Final trained model (full training set)
  - cv_results: Cross-validation results and fold models
  - val_predictions: Validation set predictions
  - fold_results: Individual fold results
  
Ready for hyperparameter tuning or test evaluation!
""")

NESTED CROSS-VALIDATION
Outer folds: 2
Validation split: 0.2

FOLD 1/2
Train samples: 57012
Validation samples: 14253
Test samples: 71266



c:\Users\msbdj\anaconda3\envs\ttk4250\lib\site-packages\keras\src\layers\rnn\rnn.py:205: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training model with RMSE loss...

Fold 1 Results (mg/dL):
  RMSE: 11.69
  MAE: 7.75

FOLD 2/2
Train samples: 57013
Validation samples: 14253
Test samples: 71265

Training model with RMSE loss...

Fold 2 Results (mg/dL):
  RMSE: 11.44
  MAE: 7.31

NESTED CV SUMMARY

RMSE across folds:
  Mean: 11.56 ± 0.12
  Range: [11.44, 11.69]

MAE across folds:
  Mean: 7.53 ± 0.22
  Range: [7.31, 7.75]

TRAINING FINAL MODEL ON FULL TRAINING SET
Training final model...

Final Model - Validation Metrics (mg/dL):
  RMSE: 9.87
  MAE: 6.01

NESTED CV COMPLETE!

Available variables:
  - lstm_model: Final trained model (full training set)
  - cv_results: Cross-validation results and fold models
  - val_predictions: Validation set predictions
  - fold_results: Individual fold results
  
Ready for hyperparameter tuning or test evaluation!



## Vanilla LSTM - Fine-tuned - Optuna + Post-training:

In [6]:
"""
Optuna Hyperparameter Tuning with Nested Cross-Validation
1. Find best hyperparameters using Optuna (on validation set)
2. Evaluate best hyperparameters with nested CV
3. Train final model on full training set
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import KFold
import optuna
from optuna.pruners import MedianPruner

# ============================================================================
# CONFIGURATION
# ============================================================================

N_TRIALS = 2  # Number of Optuna trials
N_OUTER_FOLDS = 2  # Nested CV folds after finding best params
VALIDATION_SPLIT = 0.2  # Inner validation split for early stopping
PRETRAIN_EPOCHS = 1
FINETUNE_EPOCHS = 1
RANDOM_STATE = 42

# Set random seeds for reproducibility
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

MIN_CGM = float(cgm_scaler.data_min_[0])
MAX_CGM = float(cgm_scaler.data_max_[0])

def _to_mgdl(tensor_scaled):
    """Convert scaled [0,1] values back to mg/dL"""
    return tensor_scaled * (MAX_CGM - MIN_CGM) + MIN_CGM

@tf.function
def compound_glucose_tf_loss(y_true_scaled, y_pred_scaled):
    """Compound loss: RMSE + Temporal Gain surrogate + G-Mean surrogate"""
    y_true = _to_mgdl(y_true_scaled)
    y_pred = _to_mgdl(y_pred_scaled)
    
    rmse = tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-8)
    
    dt_true = y_true[:, 1:] - y_true[:, :-1]
    dt_pred = y_pred[:, 1:] - y_pred[:, :-1]
    temporal_penalty = tf.reduce_mean(tf.square(dt_pred - dt_true))
    
    hypo_thr = tf.constant(70.2, tf.float32)
    hyper_thr = tf.constant(180.0, tf.float32)
    
    p_hypo_true = tf.sigmoid(-(y_true - hypo_thr) / 10.0)
    p_hyper_true = tf.sigmoid((y_true - hyper_thr) / 10.0)
    p_norm_true = 1.0 - p_hypo_true - p_hyper_true
    
    p_hypo_pred = tf.sigmoid(-(y_pred - hypo_thr) / 10.0)
    p_hyper_pred = tf.sigmoid((y_pred - hyper_thr) / 10.0)
    p_norm_pred = 1.0 - p_hypo_pred - p_hyper_pred
    
    eps = 1e-8
    recall_hypo = tf.reduce_mean(p_hypo_pred * p_hypo_true)
    recall_norm = tf.reduce_mean(p_norm_pred * p_norm_true)
    recall_hyper = tf.reduce_mean(p_hyper_pred * p_hyper_true)
    
    g_mean = tf.exp(
        (tf.math.log(recall_hypo + eps) +
         tf.math.log(recall_norm + eps) +
         tf.math.log(recall_hyper + eps)) / 3.0
    )
    
    compound_loss = (rmse / 100.0) + temporal_penalty + (1.0 - g_mean)
    return compound_loss

@tf.function
def rmse_loss(y_true, y_pred):
    """Root Mean Squared Error"""
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-8)

def create_model(n_in, n_features, n_out, params):
    """Create LSTM model with given hyperparameters"""
    tf.keras.backend.clear_session()
    
    model = Sequential([
        LSTM(params['lstm_units_1'], return_sequences=True, 
             input_shape=(n_in, n_features)),
        Dropout(params['dropout_rate']),
        LSTM(params['lstm_units_2'], return_sequences=False),
        Dropout(params['dropout_rate']),
        Dense(n_out, activation='relu')
    ])
    return model

def train_two_phase(model, X_tr, y_tr, X_v, y_v, params):
    """Two-phase training: RMSE pretraining → Compound loss fine-tuning"""
    
    # Phase 1: RMSE Pretraining
    model.compile(
        optimizer=Adam(learning_rate=params['lr_pretrain']),
        loss=rmse_loss,
        metrics=['mae']
    )
    
    callbacks_pretrain = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, 
                          min_lr=1e-6, verbose=0),
        EarlyStopping(monitor='val_loss', patience=5, 
                      restore_best_weights=True, verbose=0)
    ]
    
    model.fit(
        X_tr, y_tr,
        validation_data=(X_v, y_v),
        epochs=PRETRAIN_EPOCHS,
        batch_size=params['batch_size'],
        callbacks=callbacks_pretrain,
        verbose=0
    )
    
    # Phase 2: Compound Loss Fine-tuning
    model.compile(
        optimizer=Adam(learning_rate=params['lr_finetune']),
        loss=compound_glucose_tf_loss,
        metrics=['mae']
    )
    
    callbacks_finetune = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                          min_lr=1e-7, verbose=0),
        EarlyStopping(monitor='val_loss', patience=5,
                      restore_best_weights=True, verbose=0)
    ]
    
    model.fit(
        X_tr, y_tr,
        validation_data=(X_v, y_v),
        epochs=FINETUNE_EPOCHS,
        batch_size=params['batch_size'],
        callbacks=callbacks_finetune,
        verbose=0
    )
    
    return model

# ============================================================================
# DATA PREPARATION
# ============================================================================

n_samples, n_in, n_features = X_train.shape
n_out = y_train.shape[1]

y_train_flat = y_train.reshape((y_train.shape[0], n_out))
y_val_flat = y_val.reshape((y_val.shape[0], n_out))

print(f"Input shape: ({n_in}, {n_features})")
print(f"Output shape: ({n_out},)")
print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print()

# ============================================================================
# STEP 1: OPTUNA HYPERPARAMETER SEARCH
# ============================================================================

def objective(trial):
    """Optuna objective function - Returns: validation RMSE"""
    
    # Suggest hyperparameters
    params = {
        'lstm_units_1': trial.suggest_int('lstm_units_1', 32, 128, step=16),
        'lstm_units_2': trial.suggest_int('lstm_units_2', 32, 128, step=16),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.0, 0.5, step=0.1),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32, 64]),
        'lr_pretrain': trial.suggest_float('lr_pretrain', 1e-4, 1e-2, log=True),
        'lr_finetune': trial.suggest_float('lr_finetune', 1e-5, 1e-3, log=True)
    }
    
    # Build and train model
    model = create_model(n_in, n_features, n_out, params)
    model = train_two_phase(model, X_train, y_train_flat, X_val, y_val_flat, params)
    
    # Evaluate on validation set
    y_val_pred = model.predict(X_val, batch_size=64, verbose=0)
    y_val_mgdl = _to_mgdl(tf.constant(y_val_flat)).numpy()
    y_val_pred_mgdl = _to_mgdl(tf.constant(y_val_pred)).numpy()
    val_rmse = np.sqrt(np.mean((y_val_mgdl - y_val_pred_mgdl)**2))
    
    return val_rmse

print("="*70)
print("STEP 1: OPTUNA HYPERPARAMETER SEARCH")
print("="*70)
print(f"Number of trials: {N_TRIALS}")
print(f"Objective: Minimize validation RMSE")
print()

study = optuna.create_study(
    direction='minimize',
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10),
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

print("Starting optimization...")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
best_val_rmse = study.best_value

print("\n" + "="*70)
print("HYPERPARAMETER SEARCH COMPLETE")
print("="*70)
print(f"\nBest Validation RMSE: {best_val_rmse:.2f} mg/dL")
print("\nBest Hyperparameters:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

# ============================================================================
# STEP 2: NESTED CROSS-VALIDATION WITH BEST HYPERPARAMETERS
# ============================================================================

print("\n" + "="*70)
print("STEP 2: NESTED CV WITH BEST HYPERPARAMETERS")
print("="*70)
print(f"Outer folds: {N_OUTER_FOLDS}")
print(f"Inner validation split: {VALIDATION_SPLIT}")
print()

fold_results = []
fold_models = []

kfold = KFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for fold_idx, (train_idx, test_idx) in enumerate(kfold.split(X_train)):
    print(f"{'='*70}")
    print(f"FOLD {fold_idx + 1}/{N_OUTER_FOLDS}")
    print(f"{'='*70}")
    
    # Split data for this fold
    X_fold_train = X_train[train_idx]
    y_fold_train = y_train_flat[train_idx]
    X_fold_test = X_train[test_idx]
    y_fold_test = y_train_flat[test_idx]
    
    # Inner validation split
    val_size = int(len(X_fold_train) * VALIDATION_SPLIT)
    X_fold_val = X_fold_train[:val_size]
    y_fold_val = y_fold_train[:val_size]
    X_fold_train = X_fold_train[val_size:]
    y_fold_train = y_fold_train[val_size:]
    
    print(f"Train samples: {len(X_fold_train)}")
    print(f"Validation samples: {len(X_fold_val)}")
    print(f"Test samples: {len(X_fold_test)}")
    
    # Train with best hyperparameters
    model = create_model(n_in, n_features, n_out, best_params)
    print("\nTraining with best hyperparameters...")
    model = train_two_phase(model, X_fold_train, y_fold_train, 
                           X_fold_val, y_fold_val, best_params)
    
    # Evaluate on fold test set
    y_fold_pred = model.predict(X_fold_test, batch_size=64, verbose=0)
    
    # Convert to mg/dL
    y_fold_test_mgdl = _to_mgdl(tf.constant(y_fold_test)).numpy()
    y_fold_pred_mgdl = _to_mgdl(tf.constant(y_fold_pred)).numpy()
    
    # Calculate metrics
    fold_rmse = np.sqrt(np.mean((y_fold_test_mgdl - y_fold_pred_mgdl)**2))
    fold_mae = np.mean(np.abs(y_fold_test_mgdl - y_fold_pred_mgdl))
    
    print(f"\nFold {fold_idx + 1} Results (mg/dL):")
    print(f"  RMSE: {fold_rmse:.2f}")
    print(f"  MAE: {fold_mae:.2f}")
    
    fold_results.append({
        'fold': fold_idx + 1,
        'rmse': fold_rmse,
        'mae': fold_mae,
        'y_true': y_fold_test_mgdl,
        'y_pred': y_fold_pred_mgdl
    })
    fold_models.append(model)
    print()

# Aggregate CV results
rmse_scores = [r['rmse'] for r in fold_results]
mae_scores = [r['mae'] for r in fold_results]

print("="*70)
print("NESTED CV SUMMARY (WITH BEST HYPERPARAMETERS)")
print("="*70)
print(f"\nRMSE across folds:")
print(f"  Mean: {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f}")
print(f"  Range: [{np.min(rmse_scores):.2f}, {np.max(rmse_scores):.2f}]")
print(f"\nMAE across folds:")
print(f"  Mean: {np.mean(mae_scores):.2f} ± {np.std(mae_scores):.2f}")
print(f"  Range: [{np.min(mae_scores):.2f}, {np.max(mae_scores):.2f}]")

# ============================================================================
# STEP 3: TRAIN FINAL MODEL ON FULL TRAINING SET
# ============================================================================

print("\n" + "="*70)
print("STEP 3: TRAINING FINAL MODEL (FULL TRAINING SET)")
print("="*70)

final_model = create_model(n_in, n_features, n_out, best_params)

print("\nFinal Model Architecture:")
final_model.summary()

# Phase 1: RMSE Pretraining
print("\nPhase 1: RMSE Pretraining...")
final_model.compile(
    optimizer=Adam(learning_rate=best_params['lr_pretrain']),
    loss=rmse_loss,
    metrics=['mae']
)

callbacks_pretrain = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, 
                      min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='val_loss', patience=8, 
                  restore_best_weights=True, verbose=1)
]

history_pretrain = final_model.fit(
    X_train, y_train_flat,
    validation_data=(X_val, y_val_flat),
    epochs=PRETRAIN_EPOCHS,
    batch_size=best_params['batch_size'],
    callbacks=callbacks_pretrain,
    verbose=2
)

# Phase 2: Compound Loss Fine-tuning
print("\nPhase 2: Compound Loss Fine-tuning...")
final_model.compile(
    optimizer=Adam(learning_rate=best_params['lr_finetune']),
    loss=compound_glucose_tf_loss,
    metrics=['mae']
)

callbacks_finetune = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                      min_lr=1e-7, verbose=1),
    EarlyStopping(monitor='val_loss', patience=8,
                  restore_best_weights=True, verbose=1)
]

history_finetune = final_model.fit(
    X_train, y_train_flat,
    validation_data=(X_val, y_val_flat),
    epochs=FINETUNE_EPOCHS,
    batch_size=best_params['batch_size'],
    callbacks=callbacks_finetune,
    verbose=2
)

# Final validation predictions
y_val_pred_flat = final_model.predict(X_val, batch_size=64, verbose=1)
y_val_pred = y_val_pred_flat.reshape((-1, n_out, 1))

y_val_mgdl = _to_mgdl(tf.constant(y_val)).numpy()
y_val_pred_mgdl = _to_mgdl(tf.constant(y_val_pred)).numpy()

final_val_rmse = np.sqrt(np.mean((y_val_mgdl - y_val_pred_mgdl)**2))
final_val_mae = np.mean(np.abs(y_val_mgdl - y_val_pred_mgdl))

print(f"\nFinal Model - Validation Metrics (mg/dL):")
print(f"  RMSE: {final_val_rmse:.2f}")
print(f"  MAE: {final_val_mae:.2f}")

# ============================================================================
# EXPORTS
# ============================================================================

tuned_lstm_model = final_model
tuning_results = {
    'study': study,
    'best_params': best_params,
    'best_val_rmse': best_val_rmse,
    'cv_fold_results': fold_results,
    'cv_fold_models': fold_models,
    'cv_mean_rmse': np.mean(rmse_scores),
    'cv_std_rmse': np.std(rmse_scores),
    'cv_mean_mae': np.mean(mae_scores),
    'cv_std_mae': np.std(mae_scores),
    'all_trials': study.trials_dataframe()
}

val_predictions_tuned = {
    'y_true': y_val,
    'y_pred': y_val_pred,
    'y_true_mgdl': y_val_mgdl,
    'y_pred_mgdl': y_val_pred_mgdl
}

print("\n" + "="*70)
print("COMPLETE PIPELINE FINISHED!")
print("="*70)
print(f"""
Summary:
  ✓ Optuna found best hyperparameters (validation RMSE: {best_val_rmse:.2f})
  ✓ Nested CV validated performance (mean RMSE: {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f})
  ✓ Final model trained on full training set

Available variables:
  - tuned_lstm_model: Final model trained with best hyperparameters
  - tuning_results: Complete results (Optuna + CV)
  - best_params: Dictionary of best hyperparameters
  - val_predictions_tuned: Validation predictions
  - fold_results: Individual CV fold results
  
Ready for test evaluation on BrisT1D!
""")



[I 2025-12-14 12:18:04,643] A new study created in memory with name: no-name-410db0b9-dd17-40eb-9bf4-3d44e610aa23


Input shape: (12, 7)
Output shape: (6,)
Training samples: 142531
Validation samples: 48231

STEP 1: OPTUNA HYPERPARAMETER SEARCH
Number of trials: 2
Objective: Minimize validation RMSE

Starting optimization...


  0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\msbdj\anaconda3\envs\ttk4250\lib\site-packages\keras\src\layers\rnn\rnn.py:205: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[I 2025-12-14 12:21:44,074] Trial 0 finished with value: 17.1095056623079 and parameters: {'lstm_units_1': 64, 'lstm_units_2': 128, 'dropout_rate': 0.4, 'batch_size': 16, 'lr_pretrain': 0.00013066739238053285, 'lr_finetune': 0.0005399484409787432}. Best is trial 0 with value: 17.1095056623079.
[I 2025-12-14 12:25:08,142] Trial 1 finished with value: 9.164691865662041 and parameters: {'lstm_units_1': 96, 'lstm_units_2': 96, 'dropout_rate': 0.0, 'batch_size': 16, 'lr_pretrain': 0.0002310201887845295, 'lr_finetune': 2.3270677083837777e-05}. Best is trial 1 with value: 9.164691865662041.

HYPERPARAMETER SEARCH COMPLETE

Best Validation RMSE: 9.16 mg/dL

Best Hyperparameters:
  lstm_units_1: 96
  lstm_units_2: 96
  dropout_rate: 0.0
  batch_size: 16
  lr_pretrain: 0.0002310201887845295
  lr_finetune: 2.3270677083837777e-05

STEP 2: NESTED CV WITH BEST HYPERPARAMETERS
Outer folds: 2
Inner validation split: 0.2

FOLD 1/2
Train samples: 57012
Validation samples: 14253
Test samples: 71266

Trai

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 12, 96)         │        39,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 96)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 96)             │        74,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │           582 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 114,630 (447.77 KB)

 Trainable params: 114,630 (447.77 KB)

 Non-trainable params: 0 (0.00 B)


Phase 1: RMSE Pretraining...
8909/8909 - 97s - 11ms/step - loss: 0.0298 - mae: 0.0200 - val_loss: 0.0211 - val_mae: 0.0151 - learning_rate: 2.3102e-04
Restoring model weights from the end of the best epoch: 1.

Phase 2: Compound Loss Fine-tuning...
8909/8909 - 100s - 11ms/step - loss: 10.6268 - mae: 0.0124 - val_loss: 13.4442 - val_mae: 0.0139 - learning_rate: 2.3271e-05
Restoring model weights from the end of the best epoch: 1.
754/754 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step

Final Model - Validation Metrics (mg/dL):
  RMSE: 9.05
  MAE: 5.28

COMPLETE PIPELINE FINISHED!

Summary:
  ✓ Optuna found best hyperparameters (validation RMSE: 9.16)
  ✓ Nested CV validated performance (mean RMSE: 10.64 ± 0.30)
  ✓ Final model trained on full training set

Available variables:
  - tuned_lstm_model: Final model trained with best hyperparameters
  - tuning_results: Complete results (Optuna + CV)
  - best_params: Dictionary of best hyperparameters
  - val_predictions_tuned: Validation predictions
  - f

## Bergman Minimal Model - Simulation:

In [7]:
"""
Bergman Minimal Model for Blood Glucose Forecasting
Fits on training data from OhioT1DM, evaluates on validation data
"""

import numpy as np
from scipy.optimize import least_squares

# ============================================================================
# CONFIGURATION
# ============================================================================

dt_min = 5.0  # Time step in minutes
dt_hr = dt_min / 60.0  # Time step in hours

print("="*70)
print("BERGMAN MINIMAL MODEL")
print("="*70)

# ============================================================================
# STEP 1: PREPARE TRAINING SEQUENCES
# ============================================================================

print("\nStep 1: Preparing training sequences...")

train_data = []
for sid in train_ids:
    subj_df = train_subjects[sid]
    L = len(subj_df)
    
    for i in range(N_IN, L - N_OUT + 1):
        G_init = subj_df['CGM_smoothed'].iloc[i-1]
        bolus_seq = subj_df['bolus'].iloc[i:i+N_OUT].values
        carbs_seq = subj_df['carbs'].iloc[i:i+N_OUT].values
        y_true = subj_df['CGM_smoothed'].iloc[i:i+N_OUT].values
        
        # Only keep sequences with valid data
        if (np.isfinite(G_init) and 
            np.all(np.isfinite(bolus_seq)) and 
            np.all(np.isfinite(carbs_seq)) and 
            np.all(np.isfinite(y_true))):
            train_data.append({
                'G_init': float(G_init),
                'bolus': bolus_seq.astype(float),
                'carbs': carbs_seq.astype(float),
                'y_true': y_true.astype(float)
            })

print(f"  Training sequences: {len(train_data)}")

# ============================================================================
# STEP 2: DEFINE BERGMAN MODEL
# ============================================================================

def simulate_bergman(params, G_init, bolus_seq, carbs_seq):
    """
    Simulate glucose dynamics using Bergman minimal model
    
    Parameters:
        p1: glucose effectiveness (1/hr)
        p2: insulin sensitivity (1/hr per U/L)
        p3: carb absorption rate (mg/dL per g)
        k_i: insulin decay rate (1/hr)
        Gb: basal glucose (mg/dL)
    """
    p1, p2, p3, k_i, Gb = params
    
    N = len(bolus_seq)
    G = float(G_init)
    I = 0.0  # Active insulin
    
    predictions = np.zeros(N)
    
    for t in range(N):
        # Update active insulin with decay and new bolus
        I = I * np.exp(-k_i * dt_hr) + bolus_seq[t]
        
        # Meal effect from carbs
        meal_effect = p3 * carbs_seq[t]
        
        # Glucose dynamics: decay from baseline, insulin action, meal absorption
        dG = -p1 * (G - Gb) - p2 * I + meal_effect
        G = G + dG * dt_hr
        
        predictions[t] = G
    
    return predictions


def compute_residuals(params):
    """Compute residuals across all training sequences"""
    all_residuals = []
    
    for seq in train_data:
        y_pred = simulate_bergman(
            params, 
            seq['G_init'], 
            seq['bolus'], 
            seq['carbs']
        )
        residuals = y_pred - seq['y_true']
        all_residuals.append(residuals)
    
    return np.concatenate(all_residuals)


# ============================================================================
# STEP 3: FIT MODEL TO TRAINING DATA
# ============================================================================

print("\nStep 2: Fitting Bergman model parameters...")

# Initial parameter guess
p0 = np.array([
    0.01,   # p1: glucose effectiveness
    0.001,  # p2: insulin sensitivity  
    0.01,   # p3: carb absorption
    0.5,    # k_i: insulin decay
    100.0   # Gb: basal glucose
])

# Parameter bounds (physiologically plausible ranges)
bounds = (
    [1e-5, 1e-6, 0.0, 1e-3, 40.0],   # lower bounds
    [1.0, 1.0, 1.0, 5.0, 200.0]      # upper bounds
)

# Optimize parameters
result = least_squares(
    compute_residuals, 
    p0, 
    bounds=bounds,
    verbose=1,
    max_nfev=10
)

bergman_params = result.x
print(f"\n✓ Optimized parameters:")
print(f"  p1 (glucose effectiveness): {bergman_params[0]:.6f}")
print(f"  p2 (insulin sensitivity): {bergman_params[1]:.6f}")
print(f"  p3 (carb absorption): {bergman_params[2]:.6f}")
print(f"  k_i (insulin decay): {bergman_params[3]:.6f}")
print(f"  Gb (basal glucose): {bergman_params[4]:.2f} mg/dL")


# ============================================================================
# STEP 4: GENERATE PREDICTIONS ON VALIDATION DATA
# ============================================================================

print("\n" + "="*70)
print("VALIDATION PREDICTIONS")
print("="*70)

def predict_subject(subject_df, params):
    """Generate predictions for a subject using fitted Bergman model"""
    predictions = []
    L = len(subject_df)
    
    for i in range(N_IN, L - N_OUT + 1):
        G_init = subject_df['CGM_smoothed'].iloc[i-1]
        bolus_seq = subject_df['bolus'].iloc[i:i+N_OUT].values
        carbs_seq = subject_df['carbs'].iloc[i:i+N_OUT].values
        
        # Simulate glucose trajectory
        y_pred = simulate_bergman(params, G_init, bolus_seq, carbs_seq)
        
        # Scale predictions to match model input format
        y_pred_scaled = cgm_scaler.transform(y_pred.reshape(-1, 1))
        predictions.append(y_pred_scaled.reshape(N_OUT, 1))
    
    if len(predictions) == 0:
        return np.zeros((0, N_OUT, 1))
    
    return np.array(predictions)


# Generate predictions for each validation subject
print("\nGenerating predictions for validation subjects...")
bergman_val_predictions = {}

for sid in val_ids:
    subject_df = train_subjects[sid]
    preds = predict_subject(subject_df, bergman_params)
    bergman_val_predictions[sid] = preds
    print(f"  Subject {sid}: {preds.shape[0]} predictions")

# Aggregate all validation predictions
y_pred_bergman = np.concatenate([
    bergman_val_predictions[sid] 
    for sid in val_ids 
    if bergman_val_predictions[sid].shape[0] > 0
])

print(f"\n✓ Total validation predictions: {y_pred_bergman.shape}")
print(f"  Shape: (n_samples={y_pred_bergman.shape[0]}, horizon={y_pred_bergman.shape[1]}, features={y_pred_bergman.shape[2]})")


# ============================================================================
# STEP 5: COMPUTE VALIDATION METRICS
# ============================================================================

print("\n" + "="*70)
print("VALIDATION METRICS")
print("="*70)

from sklearn.metrics import mean_squared_error, mean_absolute_error

# Unscale predictions and ground truth for interpretable metrics
y_val_unscaled = cgm_scaler.inverse_transform(y_val.reshape(-1, 1)).reshape(-1, N_OUT)
y_pred_unscaled = cgm_scaler.inverse_transform(y_pred_bergman.reshape(-1, 1)).reshape(-1, N_OUT)

# Compute metrics for each horizon
print("\nMetrics by prediction horizon:")
print(f"{'Horizon':<10} {'RMSE (mg/dL)':<15} {'MAE (mg/dL)':<15}")
print("-" * 40)

for h in range(N_OUT):
    rmse = np.sqrt(mean_squared_error(y_val_unscaled[:, h], y_pred_unscaled[:, h]))
    mae = mean_absolute_error(y_val_unscaled[:, h], y_pred_unscaled[:, h])
    print(f"{(h+1)*5:2d} min     {rmse:>12.2f}     {mae:>12.2f}")

# Overall metrics
rmse_overall = np.sqrt(mean_squared_error(y_val_unscaled.flatten(), y_pred_unscaled.flatten()))
mae_overall = mean_absolute_error(y_val_unscaled.flatten(), y_pred_unscaled.flatten())

print("\n" + "-" * 40)
print(f"{'Overall':<10} {rmse_overall:>12.2f}     {mae_overall:>12.2f}")


# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*70)
print("BERGMAN MODEL READY")
print("="*70)
print(f"""
Available variables:
  - bergman_params: Fitted model parameters
  - bergman_val_predictions: Per-subject validation predictions (dict)
  - y_pred_bergman: All validation predictions (scaled)
  - predict_subject(): Function to generate predictions for new subjects
  
Validation Performance:
  - RMSE: {rmse_overall:.2f} mg/dL
  - MAE: {mae_overall:.2f} mg/dL
  
Next steps:
  - Compare with ML models (LSTM, Transformer, etc.)
  - Use predict_subject(train_subjects[sid], bergman_params) for test predictions when ready
""")

BERGMAN MINIMAL MODEL

Step 1: Preparing training sequences...
  Training sequences: 142531

Step 2: Fitting Bergman model parameters...
`ftol` termination condition is satisfied.
Function evaluations 9, initial cost 9.9464e+07, final cost 9.7516e+07, first-order optimality 3.72e+00.

✓ Optimized parameters:
  p1 (glucose effectiveness): 0.118938
  p2 (insulin sensitivity): 0.000001
  p3 (carb absorption): 0.002628
  k_i (insulin decay): 5.000000
  Gb (basal glucose): 161.63 mg/dL

VALIDATION PREDICTIONS

Generating predictions for validation subjects...
  Subject 591: 15823 predictions
  Subject 588: 16111 predictions
  Subject 540: 16297 predictions

✓ Total validation predictions: (48231, 6, 1)
  Shape: (n_samples=48231, horizon=6, features=1)

VALIDATION METRICS

Metrics by prediction horizon:
Horizon    RMSE (mg/dL)    MAE (mg/dL)    
----------------------------------------
 5 min             4.80             3.37
10 min             9.39             6.61
15 min            13.65  

## Hybrid Model - Residual Corrector - Without Fine-tuning:

In [11]:
"""
Hybrid Residual Corrector with Nested Cross-Validation - OPTIMIZED
Key optimizations:
- Stride-based sampling for residual data (10x speedup)
- Stride-based evaluation (5x speedup)
- Reduced Bergman iterations (2x speedup)
- Larger batch size (1.5x speedup)
- Vectorized operations where possible
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from scipy.optimize import differential_evolution
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

# ============================================================================
# BERGMAN MINIMAL MODEL - OPTIMIZED
# ============================================================================

class BergmanMinimalModel:
    def __init__(self):
        self.p1_bg = None
        self.p2_bg = None
        self.p3_bg = None
        self.SI_bg = None
        self.Vg_bg = None
        
    def simulate(self, G_init, insulin_seq, carbs_seq, dt=5.0):
        n_steps = len(insulin_seq)
        G_bg = G_init
        X_bg = 0.0
        I_bg = 0.0
        G_trajectory = []
        
        for t in range(n_steps):
            Ra_carbs = carbs_seq[t] * 5.0 / self.Vg_bg if self.Vg_bg > 0 else 0
            I_input = insulin_seq[t] * 1000.0 / self.Vg_bg if self.Vg_bg > 0 else 0
            
            dG = (-self.p1_bg * G_bg - X_bg * G_bg + Ra_carbs) * dt
            dX = (-self.p2_bg * X_bg + self.p3_bg * self.SI_bg * I_bg) * dt
            dI = (-self.p2_bg * I_bg + I_input) * dt
            
            G_bg = max(20, G_bg + dG)
            X_bg = max(0, X_bg + dX)
            I_bg = max(0, I_bg + dI)
            
            G_trajectory.append(G_bg)
        
        return np.array(G_trajectory)
    
    def fit(self, train_subjects, train_sids, n_in, n_out, cgm_scaler):
        print("  Fitting Bergman model...")
        
        samples = []
        for sid in train_sids:
            subj_data = train_subjects[sid]
            n_samples = len(subj_data) - n_in - n_out + 1
            
            # OPTIMIZATION: Increased stride from 10 to 20 for faster fitting
            for idx in range(0, n_samples, 20):
                window_start = n_in + idx
                G_init = subj_data['CGM_smoothed'].values[window_start - 1]
                bolus = subj_data['bolus'].values[window_start:window_start + n_out]
                carbs = subj_data['carbs'].values[window_start:window_start + n_out]
                cgm_future = subj_data['CGM_smoothed'].values[window_start:window_start + n_out]
                
                samples.append({
                    'G_init': G_init,
                    'bolus': bolus,
                    'carbs': carbs,
                    'y_true': cgm_future
                })
        
        def objective(params):
            self.p1_bg, self.p2_bg, self.p3_bg, self.SI_bg, self.Vg_bg = params
            errors = []
            # OPTIMIZATION: Reduced from 500 to 200 samples for faster convergence
            for sample in samples[:200]:
                pred = self.simulate(sample['G_init'], sample['bolus'], sample['carbs'])
                error = np.mean((pred - sample['y_true']) ** 2)
                errors.append(error)
            return np.mean(errors)
        
        bounds = [
            (0.001, 0.05),
            (0.001, 0.1),
            (0.0001, 0.01),
            (0.0001, 0.01),
            (50, 300)
        ]
        
        # OPTIMIZATION: Reduced maxiter from 5 to 3 and popsize from 10 to 8
        result = differential_evolution(objective, bounds, maxiter=3, popsize=8, 
                                       seed=42, disp=False, workers=1)
        
        self.p1_bg, self.p2_bg, self.p3_bg, self.SI_bg, self.Vg_bg = result.x
        print(f"    Bergman MSE: {result.fun:.2f}")

# ============================================================================
# HELPER FUNCTIONS - OPTIMIZED
# ============================================================================
@tf.function
def rmse_loss(y_true, y_pred):
    """Root Mean Squared Error"""
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-8)


def build_residual_data_from_subjects(subject_dict, subject_ids, bergman_model, 
                                      n_in, n_out, cgm_scaler, model_features, 
                                      feature_scaler, stride=5):
    """
    OPTIMIZATION: Added stride parameter to sample every Nth window
    stride=5 gives 5x speedup with minimal impact on model quality
    """
    X_residual = []
    y_residual = []
    
    for sid in subject_ids:
        subj_data = subject_dict[sid]
        n_samples = len(subj_data) - n_in - n_out + 1
        
        # OPTIMIZATION: Stride through samples instead of using all
        for idx in range(0, n_samples, stride):
            window_start = n_in + idx
            
            # Build input features
            cgm_window = subj_data['CGM_smoothed'].values[window_start-n_in:window_start]
            features_window = subj_data[model_features].values[window_start-n_in:window_start]
            
            cgm_scaled = cgm_scaler.transform(cgm_window.reshape(-1, 1)).flatten()
            features_scaled = feature_scaler.transform(features_window)
            
            x_input = np.column_stack([cgm_scaled, features_scaled])
            
            # Bergman prediction
            G_init = subj_data['CGM_smoothed'].values[window_start - 1]
            bolus = subj_data['bolus'].values[window_start:window_start + n_out]
            carbs = subj_data['carbs'].values[window_start:window_start + n_out]
            
            berg_pred = bergman_model.simulate(G_init, bolus, carbs)
            berg_pred_scaled = cgm_scaler.transform(berg_pred.reshape(-1, 1)).flatten()
            
            # True values
            y_true = subj_data['CGM_smoothed'].values[window_start:window_start + n_out]
            y_true_scaled = cgm_scaler.transform(y_true.reshape(-1, 1)).flatten()
            
            # Residual
            residual = y_true_scaled - berg_pred_scaled
            
            X_residual.append(x_input)
            y_residual.append(residual)
    
    return np.array(X_residual, dtype=np.float32), np.array(y_residual, dtype=np.float32)


def train_lstm_corrector(X_train, y_train, X_val, y_val, n_in, n_out, n_features, 
                        lstm_units=40, epochs=50, learning_rate=0.001):
    """
    Train LSTM corrector with single-phase MSE loss
    OPTIMIZATION: Increased batch size for faster training
    """
    tf.keras.backend.clear_session()
    
    model = Sequential([
        LSTM(lstm_units, return_sequences=False, input_shape=(n_in, n_features)),
        Dense(n_out, activation='linear')
    ])
    
    optimizer = Adam(learning_rate=learning_rate, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss=rmse_loss, metrics=['mae'])
    
    # OPTIMIZATION: More aggressive early stopping
    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, 
                         verbose=0, min_lr=1e-6),
        EarlyStopping(monitor='val_loss', patience=5, 
                     restore_best_weights=True, verbose=0)
    ]
    
    # OPTIMIZATION: Increased batch_size from 32 to 64 for faster training
    history = model.fit(
        X_train, y_train, 
        epochs=epochs, 
        batch_size=64,
        validation_data=(X_val, y_val), 
        callbacks=callbacks, 
        verbose=0
    )
    
    return model


def evaluate_hybrid_predictions(subject_dict, subject_ids, bergman_model, 
                                lstm_corrector, n_in, n_out, cgm_scaler, 
                                model_features, feature_scaler, stride=10):
    """
    OPTIMIZATION: Added stride parameter for faster evaluation
    stride=10 gives 10x speedup with representative sampling
    """
    all_y_true = []
    all_y_pred = []
    
    for sid in subject_ids:
        subj_data = subject_dict[sid]
        n_samples = len(subj_data) - n_in - n_out + 1
        
        # OPTIMIZATION: Stride through samples for evaluation
        for idx in range(0, n_samples, stride):
            window_start = n_in + idx
            
            # Build input
            cgm_window = subj_data['CGM_smoothed'].values[window_start-n_in:window_start]
            features_window = subj_data[model_features].values[window_start-n_in:window_start]
            
            cgm_scaled = cgm_scaler.transform(cgm_window.reshape(-1, 1)).flatten()
            features_scaled = feature_scaler.transform(features_window)
            
            x_input = np.column_stack([cgm_scaled, features_scaled]).reshape(1, n_in, -1)
            
            # Bergman prediction
            G_init = subj_data['CGM_smoothed'].values[window_start - 1]
            bolus = subj_data['bolus'].values[window_start:window_start + n_out]
            carbs = subj_data['carbs'].values[window_start:window_start + n_out]
            
            berg_pred = bergman_model.simulate(G_init, bolus, carbs)
            berg_pred_scaled = cgm_scaler.transform(berg_pred.reshape(-1, 1)).flatten()
            
            # LSTM correction
            residual_pred = lstm_corrector.predict(x_input, verbose=0)[0]
            
            # Combine
            y_hybrid = berg_pred_scaled + residual_pred
            
            # True values
            y_true = subj_data['CGM_smoothed'].values[window_start:window_start + n_out]
            y_true_scaled = cgm_scaler.transform(y_true.reshape(-1, 1)).flatten()
            
            all_y_true.append(y_true_scaled)
            all_y_pred.append(y_hybrid)
    
    y_true = np.array(all_y_true)
    y_pred = np.array(all_y_pred)
    
    # Convert to mg/dL
    y_true_mgdl = cgm_scaler.inverse_transform(y_true.reshape(-1, 1)).reshape(y_true.shape)
    y_pred_mgdl = cgm_scaler.inverse_transform(y_pred.reshape(-1, 1)).reshape(y_pred.shape)
    
    rmse = np.sqrt(mean_squared_error(y_true_mgdl, y_pred_mgdl))
    mae = mean_absolute_error(y_true_mgdl, y_pred_mgdl)
    
    return rmse, mae


# ============================================================================
# NESTED CROSS-VALIDATION - OPTIMIZED
# ============================================================================

def nested_cross_validation(train_subjects, n_in, n_out, cgm_scaler, 
                            feature_scaler, model_features, 
                            n_outer_folds=5, n_inner_folds=3,
                            lstm_units=40, epochs=50, learning_rate=0.001):
    """
    Nested cross-validation with OPTIMIZATIONS for faster execution
    """
    
    print("="*70)
    print("NESTED CROSS-VALIDATION - OPTIMIZED VERSION")
    print("="*70)
    print(f"Outer folds: {n_outer_folds}")
    print(f"Inner folds: {n_inner_folds}")
    print(f"LSTM units: {lstm_units}")
    print(f"Max epochs: {epochs}")
    print(f"Learning rate: {learning_rate}")
    print(f"OPTIMIZATIONS: Stride sampling for 5-10x speedup")
    print()
    
    # Get all subject IDs
    all_subject_ids = sorted(train_subjects.keys())
    n_subjects = len(all_subject_ids)
    print(f"Total subjects: {n_subjects}\n")
    
    # Outer CV
    outer_kf = KFold(n_splits=n_outer_folds, shuffle=True, random_state=42)
    
    outer_results = []
    
    for outer_fold, (train_idx, test_idx) in enumerate(outer_kf.split(all_subject_ids), 1):
        print(f"\n{'='*70}")
        print(f"OUTER FOLD {outer_fold}/{n_outer_folds}")
        print(f"{'='*70}")
        
        # Split subjects for outer fold
        outer_train_sids = [all_subject_ids[i] for i in train_idx]
        outer_test_sids = [all_subject_ids[i] for i in test_idx]
        
        print(f"Train subjects: {len(outer_train_sids)}")
        print(f"Test subjects: {len(outer_test_sids)}")
        
        # Inner CV
        inner_kf = KFold(n_splits=n_inner_folds, shuffle=True, random_state=42)
        
        best_inner_rmse = float('inf')
        best_model = None
        best_bergman = None
        
        for inner_fold, (inner_train_idx, inner_val_idx) in enumerate(inner_kf.split(outer_train_sids), 1):
            print(f"\n  Inner Fold {inner_fold}/{n_inner_folds}")
            
            # Split subjects
            inner_train_sids = [outer_train_sids[i] for i in inner_train_idx]
            inner_val_sids = [outer_train_sids[i] for i in inner_val_idx]
            
            print(f"    Inner train: {len(inner_train_sids)} subjects")
            print(f"    Inner val: {len(inner_val_sids)} subjects")
            
            # Fit Bergman model
            bergman = BergmanMinimalModel()
            bergman.fit(train_subjects, inner_train_sids, n_in, n_out, cgm_scaler)
            
            # Build residual data with STRIDE=5
            X_train_res, y_train_res = build_residual_data_from_subjects(
                train_subjects, inner_train_sids, bergman, n_in, n_out, 
                cgm_scaler, model_features, feature_scaler, stride=5
            )
            
            X_val_res, y_val_res = build_residual_data_from_subjects(
                train_subjects, inner_val_sids, bergman, n_in, n_out, 
                cgm_scaler, model_features, feature_scaler, stride=5
            )
            
            print(f"    Train samples: {len(X_train_res)} (strided)")
            print(f"    Val samples: {len(X_val_res)} (strided)")
            
            # Train LSTM corrector
            n_features = X_train_res.shape[2]
            lstm_model = train_lstm_corrector(
                X_train_res, y_train_res, X_val_res, y_val_res,
                n_in, n_out, n_features, lstm_units, epochs, learning_rate
            )
            
            # Evaluate on inner validation with STRIDE=10
            val_rmse, val_mae = evaluate_hybrid_predictions(
                train_subjects, inner_val_sids, bergman, lstm_model,
                n_in, n_out, cgm_scaler, model_features, feature_scaler, stride=10
            )
            
            print(f"    Val RMSE: {val_rmse:.2f} mg/dL, MAE: {val_mae:.2f} mg/dL")
            
            # Track best model
            if val_rmse < best_inner_rmse:
                best_inner_rmse = val_rmse
                best_model = lstm_model
                best_bergman = bergman
                print(f"    ✓ New best model")
        
        # Evaluate best model on outer test fold with STRIDE=10
        print(f"\n  Evaluating best model on outer test fold...")
        test_rmse, test_mae = evaluate_hybrid_predictions(
            train_subjects, outer_test_sids, best_bergman, best_model,
            n_in, n_out, cgm_scaler, model_features, feature_scaler, stride=10
        )
        
        outer_results.append({
            'fold': outer_fold,
            'test_rmse': test_rmse,
            'test_mae': test_mae,
            'n_test_subjects': len(outer_test_sids)
        })
        
        print(f"\n  OUTER FOLD {outer_fold} RESULTS:")
        print(f"    Test RMSE: {test_rmse:.2f} mg/dL")
        print(f"    Test MAE: {test_mae:.2f} mg/dL")
    
    # Summary statistics
    print(f"\n{'='*70}")
    print("NESTED CV SUMMARY")
    print(f"{'='*70}")
    
    rmse_values = [r['test_rmse'] for r in outer_results]
    mae_values = [r['test_mae'] for r in outer_results]
    
    print(f"\nRMSE across {n_outer_folds} folds:")
    print(f"  Mean: {np.mean(rmse_values):.2f} ± {np.std(rmse_values):.2f} mg/dL")
    print(f"  Min: {np.min(rmse_values):.2f} mg/dL")
    print(f"  Max: {np.max(rmse_values):.2f} mg/dL")
    
    print(f"\nMAE across {n_outer_folds} folds:")
    print(f"  Mean: {np.mean(mae_values):.2f} ± {np.std(mae_values):.2f} mg/dL")
    print(f"  Min: {np.min(mae_values):.2f} mg/dL")
    print(f"  Max: {np.max(mae_values):.2f} mg/dL")
    
    print(f"\nPer-fold results:")
    for r in outer_results:
        print(f"  Fold {r['fold']}: RMSE={r['test_rmse']:.2f}, MAE={r['test_mae']:.2f}")
    
    return outer_results


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Verify required variables
    required_vars = ['train_subjects', 'cgm_scaler', 'feature_scaler', 
                     'N_IN', 'N_OUT', 'MODEL_FEATURES']
    missing = [v for v in required_vars if v not in globals()]
    if missing:
        raise RuntimeError(f"Missing variables: {missing}. Run data processing first.")
    
    # Run nested cross-validation
    cv_results = nested_cross_validation(
        train_subjects=train_subjects,
        n_in=N_IN,
        n_out=N_OUT,
        cgm_scaler=cgm_scaler,
        feature_scaler=feature_scaler,
        model_features=MODEL_FEATURES,
        n_outer_folds=2,
        n_inner_folds=2,
        lstm_units=40,
        epochs=1,
        learning_rate=0.001
    )
    
    print("\n✓ Nested cross-validation complete!")
    print("  Results stored in: cv_results")

NESTED CROSS-VALIDATION - OPTIMIZED VERSION
Outer folds: 2
Inner folds: 2
LSTM units: 40
Max epochs: 1
Learning rate: 0.001
OPTIMIZATIONS: Stride sampling for 5-10x speedup

Total subjects: 12


OUTER FOLD 1/2
Train subjects: 6
Test subjects: 6

  Inner Fold 1/2
    Inner train: 3 subjects
    Inner val: 3 subjects
  Fitting Bergman model...
    Bergman MSE: 185.78
    Train samples: 9305 (strided)
    Val samples: 9618 (strided)


c:\Users\msbdj\anaconda3\envs\ttk4250\lib\site-packages\keras\src\layers\rnn\rnn.py:205: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


    Val RMSE: 34.52 mg/dL, MAE: 26.14 mg/dL
    ✓ New best model

  Inner Fold 2/2
    Inner train: 3 subjects
    Inner val: 3 subjects
  Fitting Bergman model...
    Bergman MSE: 208.49
    Train samples: 9618 (strided)
    Val samples: 9305 (strided)


c:\Users\msbdj\anaconda3\envs\ttk4250\lib\site-packages\keras\src\layers\rnn\rnn.py:205: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


    Val RMSE: 14.95 mg/dL, MAE: 9.44 mg/dL
    ✓ New best model

  Evaluating best model on outer test fold...

  OUTER FOLD 1 RESULTS:
    Test RMSE: 16.57 mg/dL
    Test MAE: 10.14 mg/dL

OUTER FOLD 2/2
Train subjects: 6
Test subjects: 6

  Inner Fold 1/2
    Inner train: 3 subjects
    Inner val: 3 subjects
  Fitting Bergman model...
    Bergman MSE: 227.36
    Train samples: 9769 (strided)
    Val samples: 9466 (strided)


c:\Users\msbdj\anaconda3\envs\ttk4250\lib\site-packages\keras\src\layers\rnn\rnn.py:205: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


    Val RMSE: 13.19 mg/dL, MAE: 8.60 mg/dL
    ✓ New best model

  Inner Fold 2/2
    Inner train: 3 subjects
    Inner val: 3 subjects
  Fitting Bergman model...
    Bergman MSE: 354.55
    Train samples: 9466 (strided)
    Val samples: 9769 (strided)


c:\Users\msbdj\anaconda3\envs\ttk4250\lib\site-packages\keras\src\layers\rnn\rnn.py:205: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


    Val RMSE: 15.66 mg/dL, MAE: 9.29 mg/dL

  Evaluating best model on outer test fold...

  OUTER FOLD 2 RESULTS:
    Test RMSE: 13.79 mg/dL
    Test MAE: 8.71 mg/dL

NESTED CV SUMMARY

RMSE across 2 folds:
  Mean: 15.18 ± 1.39 mg/dL
  Min: 13.79 mg/dL
  Max: 16.57 mg/dL

MAE across 2 folds:
  Mean: 9.42 ± 0.72 mg/dL
  Min: 8.71 mg/dL
  Max: 10.14 mg/dL

Per-fold results:
  Fold 1: RMSE=16.57, MAE=10.14
  Fold 2: RMSE=13.79, MAE=8.71

✓ Nested cross-validation complete!
  Results stored in: cv_results


## Hybrid - Residual Corrector - Fine-tuned - Optuna

In [13]:
"""
Hybrid Residual Corrector with Optuna Hyperparameter Tuning
Features:
- Two-phase training (RMSE pre-training + compound loss fine-tuning)
- Optuna hyperparameter optimization
- Optimized with stride-based sampling
- Saves final model as 'tuned_hybrid_corrector_model'
"""

import numpy as np
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from scipy.optimize import differential_evolution
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

# ============================================================================
# BERGMAN MINIMAL MODEL
# ============================================================================

class BergmanMinimalModel:
    def __init__(self):
        self.p1_bg = None
        self.p2_bg = None
        self.p3_bg = None
        self.SI_bg = None
        self.Vg_bg = None
        
    def simulate(self, G_init, insulin_seq, carbs_seq, dt=5.0):
        n_steps = len(insulin_seq)
        G_bg = G_init
        X_bg = 0.0
        I_bg = 0.0
        G_trajectory = []
        
        for t in range(n_steps):
            Ra_carbs = carbs_seq[t] * 5.0 / self.Vg_bg if self.Vg_bg > 0 else 0
            I_input = insulin_seq[t] * 1000.0 / self.Vg_bg if self.Vg_bg > 0 else 0
            
            dG = (-self.p1_bg * G_bg - X_bg * G_bg + Ra_carbs) * dt
            dX = (-self.p2_bg * X_bg + self.p3_bg * self.SI_bg * I_bg) * dt
            dI = (-self.p2_bg * I_bg + I_input) * dt
            
            G_bg = max(20, G_bg + dG)
            X_bg = max(0, X_bg + dX)
            I_bg = max(0, I_bg + dI)
            
            G_trajectory.append(G_bg)
        
        return np.array(G_trajectory)
    
    def fit(self, train_subjects, train_sids, n_in, n_out, cgm_scaler):
        print("  Fitting Bergman model...")
        
        samples = []
        for sid in train_sids:
            subj_data = train_subjects[sid]
            n_samples = len(subj_data) - n_in - n_out + 1
            
            for idx in range(0, n_samples, 20):
                window_start = n_in + idx
                G_init = subj_data['CGM_smoothed'].values[window_start - 1]
                bolus = subj_data['bolus'].values[window_start:window_start + n_out]
                carbs = subj_data['carbs'].values[window_start:window_start + n_out]
                cgm_future = subj_data['CGM_smoothed'].values[window_start:window_start + n_out]
                
                samples.append({
                    'G_init': G_init,
                    'bolus': bolus,
                    'carbs': carbs,
                    'y_true': cgm_future
                })
        
        def objective(params):
            self.p1_bg, self.p2_bg, self.p3_bg, self.SI_bg, self.Vg_bg = params
            errors = []
            for sample in samples[:200]:
                pred = self.simulate(sample['G_init'], sample['bolus'], sample['carbs'])
                error = np.mean((pred - sample['y_true']) ** 2)
                errors.append(error)
            return np.mean(errors)
        
        bounds = [
            (0.001, 0.05),
            (0.001, 0.1),
            (0.0001, 0.01),
            (0.0001, 0.01),
            (50, 300)
        ]
        
        result = differential_evolution(objective, bounds, maxiter=3, popsize=8, 
                                       seed=42, disp=False, workers=1)
        
        self.p1_bg, self.p2_bg, self.p3_bg, self.SI_bg, self.Vg_bg = result.x
        print(f"    Bergman MSE: {result.fun:.2f}")


# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

@tf.function
def rmse_loss(y_true, y_pred):
    """Root Mean Squared Error"""
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-8)


def create_compound_glucose_loss(cgm_scaler):
    """
    Creates compound loss function with proper scaling
    """
    MIN_CGM = float(cgm_scaler.data_min_[0])
    MAX_CGM = float(cgm_scaler.data_max_[0])
    
    def _to_mgdl(tensor_scaled):
        """Convert scaled [0,1] values back to mg/dL"""
        return tensor_scaled * (MAX_CGM - MIN_CGM) + MIN_CGM
    
    @tf.function
    def compound_glucose_tf_loss(y_true_scaled, y_pred_scaled):
        """Compound loss: RMSE + Temporal + G-Mean"""
        y_true = _to_mgdl(y_true_scaled)
        y_pred = _to_mgdl(y_pred_scaled)
        
        # RMSE component
        rmse = tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-8)
        
        # Temporal consistency
        dt_true = y_true[:, 1:] - y_true[:, :-1]
        dt_pred = y_pred[:, 1:] - y_pred[:, :-1]
        temporal_penalty = tf.reduce_mean(tf.square(dt_pred - dt_true))
        
        # G-Mean surrogate for zone classification
        hypo_thr = tf.constant(70.2, tf.float32)
        hyper_thr = tf.constant(180.0, tf.float32)
        
        p_hypo_true = tf.sigmoid(-(y_true - hypo_thr) / 10.0)
        p_hyper_true = tf.sigmoid((y_true - hyper_thr) / 10.0)
        p_norm_true = 1.0 - p_hypo_true - p_hyper_true
        
        p_hypo_pred = tf.sigmoid(-(y_pred - hypo_thr) / 10.0)
        p_hyper_pred = tf.sigmoid((y_pred - hyper_thr) / 10.0)
        p_norm_pred = 1.0 - p_hypo_pred - p_hyper_pred
        
        eps = 1e-8
        recall_hypo = tf.reduce_mean(p_hypo_pred * p_hypo_true)
        recall_norm = tf.reduce_mean(p_norm_pred * p_norm_true)
        recall_hyper = tf.reduce_mean(p_hyper_pred * p_hyper_true)
        
        g_mean = tf.exp(
            (tf.math.log(recall_hypo + eps) +
             tf.math.log(recall_norm + eps) +
             tf.math.log(recall_hyper + eps)) / 3.0
        )
        
        compound_loss = (rmse / 100.0) + temporal_penalty + (1.0 - g_mean)
        return compound_loss
    
    return compound_glucose_tf_loss


# ============================================================================
# DATA BUILDING - OPTIMIZED
# ============================================================================

def build_residual_data_from_subjects(subject_dict, subject_ids, bergman_model, 
                                      n_in, n_out, cgm_scaler, model_features, 
                                      feature_scaler, stride=5):
    """
    Build residual training data with stride-based sampling
    """
    X_residual = []
    y_residual = []
    
    for sid in subject_ids:
        subj_data = subject_dict[sid]
        n_samples = len(subj_data) - n_in - n_out + 1
        
        for idx in range(0, n_samples, stride):
            window_start = n_in + idx
            
            # Build input features
            cgm_window = subj_data['CGM_smoothed'].values[window_start-n_in:window_start]
            features_window = subj_data[model_features].values[window_start-n_in:window_start]
            
            cgm_scaled = cgm_scaler.transform(cgm_window.reshape(-1, 1)).flatten()
            features_scaled = feature_scaler.transform(features_window)
            
            x_input = np.column_stack([cgm_scaled, features_scaled])
            
            # Bergman prediction
            G_init = subj_data['CGM_smoothed'].values[window_start - 1]
            bolus = subj_data['bolus'].values[window_start:window_start + n_out]
            carbs = subj_data['carbs'].values[window_start:window_start + n_out]
            
            berg_pred = bergman_model.simulate(G_init, bolus, carbs)
            berg_pred_scaled = cgm_scaler.transform(berg_pred.reshape(-1, 1)).flatten()
            
            # True values
            y_true = subj_data['CGM_smoothed'].values[window_start:window_start + n_out]
            y_true_scaled = cgm_scaler.transform(y_true.reshape(-1, 1)).flatten()
            
            # Residual
            residual = y_true_scaled - berg_pred_scaled
            
            X_residual.append(x_input)
            y_residual.append(residual)
    
    return np.array(X_residual, dtype=np.float32), np.array(y_residual, dtype=np.float32)


def evaluate_hybrid_predictions(subject_dict, subject_ids, bergman_model, 
                                lstm_corrector, n_in, n_out, cgm_scaler, 
                                model_features, feature_scaler, stride=10):
    """
    Evaluate hybrid model with stride-based sampling
    """
    all_y_true = []
    all_y_pred = []
    
    for sid in subject_ids:
        subj_data = subject_dict[sid]
        n_samples = len(subj_data) - n_in - n_out + 1
        
        for idx in range(0, n_samples, stride):
            window_start = n_in + idx
            
            # Build input
            cgm_window = subj_data['CGM_smoothed'].values[window_start-n_in:window_start]
            features_window = subj_data[model_features].values[window_start-n_in:window_start]
            
            cgm_scaled = cgm_scaler.transform(cgm_window.reshape(-1, 1)).flatten()
            features_scaled = feature_scaler.transform(features_window)
            
            x_input = np.column_stack([cgm_scaled, features_scaled]).reshape(1, n_in, -1)
            
            # Bergman prediction
            G_init = subj_data['CGM_smoothed'].values[window_start - 1]
            bolus = subj_data['bolus'].values[window_start:window_start + n_out]
            carbs = subj_data['carbs'].values[window_start:window_start + n_out]
            
            berg_pred = bergman_model.simulate(G_init, bolus, carbs)
            berg_pred_scaled = cgm_scaler.transform(berg_pred.reshape(-1, 1)).flatten()
            
            # LSTM correction
            residual_pred = lstm_corrector.predict(x_input, verbose=0)[0]
            
            # Combine
            y_hybrid = berg_pred_scaled + residual_pred
            
            # True values
            y_true = subj_data['CGM_smoothed'].values[window_start:window_start + n_out]
            y_true_scaled = cgm_scaler.transform(y_true.reshape(-1, 1)).flatten()
            
            all_y_true.append(y_true_scaled)
            all_y_pred.append(y_hybrid)
    
    y_true = np.array(all_y_true)
    y_pred = np.array(all_y_pred)
    
    # Convert to mg/dL
    y_true_mgdl = cgm_scaler.inverse_transform(y_true.reshape(-1, 1)).reshape(y_true.shape)
    y_pred_mgdl = cgm_scaler.inverse_transform(y_pred.reshape(-1, 1)).reshape(y_pred.shape)
    
    rmse = np.sqrt(mean_squared_error(y_true_mgdl, y_pred_mgdl))
    mae = mean_absolute_error(y_true_mgdl, y_pred_mgdl)
    
    return rmse, mae


# ============================================================================
# TWO-PHASE TRAINING WITH OPTUNA
# ============================================================================

def create_optuna_objective(X_train, y_train, X_val, y_val, n_in, n_out, 
                            n_features, cgm_scaler):
    """
    Creates objective function for Optuna optimization
    Two-phase training: RMSE → Compound loss
    """
    
    def objective(trial):
        tf.keras.backend.clear_session()
        
        # Hyperparameters
        lstm_units = trial.suggest_int('lstm_units', 30, 80, step=10)
        dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.3)
        
        # Phase 1: RMSE pre-training
        lr_phase1 = trial.suggest_float('lr_phase1', 1e-4, 1e-2, log=True)
        batch_size_phase1 = trial.suggest_categorical('batch_size_phase1', [32, 64, 128])
        epochs_phase1 = trial.suggest_int('epochs_phase1', 20, 50)
        
        # Phase 2: Compound loss fine-tuning
        lr_phase2 = trial.suggest_float('lr_phase2', 1e-6, 1e-4, log=True)
        batch_size_phase2 = trial.suggest_categorical('batch_size_phase2', [32, 64, 128])
        epochs_phase2 = trial.suggest_int('epochs_phase2', 10, 30)
        
        clipnorm = trial.suggest_float('clipnorm', 0.5, 2.0)
        
        # Build model
        model = Sequential([
            LSTM(lstm_units, return_sequences=False, input_shape=(n_in, n_features)),
            Dropout(dropout_rate),
            Dense(n_out, activation='linear')
        ])
        
        # PHASE 1: RMSE PRE-TRAINING
        optimizer_pre = Adam(learning_rate=lr_phase1, clipnorm=clipnorm)
        model.compile(optimizer=optimizer_pre, loss=rmse_loss, metrics=['mae'])
        
        callbacks_pre = [
            EarlyStopping(monitor='val_loss', patience=10, 
                         restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, 
                            verbose=0, min_lr=1e-7)
        ]
        
        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs_phase1,
            batch_size=batch_size_phase1,
            callbacks=callbacks_pre,
            verbose=0
        )
        
        # PHASE 2: COMPOUND LOSS FINE-TUNING
        compound_loss = create_compound_glucose_loss(cgm_scaler)
        optimizer_post = Adam(learning_rate=lr_phase2, clipnorm=clipnorm)
        model.compile(optimizer=optimizer_post, loss=compound_loss)
        
        callbacks_post = [
            EarlyStopping(monitor='val_loss', patience=8, 
                         restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, 
                            verbose=0, min_lr=1e-7)
        ]
        
        history_post = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs_phase2,
            batch_size=batch_size_phase2,
            callbacks=callbacks_post,
            verbose=0
        )
        
        val_loss = min(history_post.history['val_loss'])
        
        # Pruning
        for epoch in range(len(history_post.history['val_loss'])):
            trial.report(history_post.history['val_loss'][epoch], epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        return val_loss
    
    return objective


def optimize_hyperparameters(X_train, y_train, X_val, y_val, n_in, n_out, 
                             cgm_scaler, n_trials=50):
    """
    Run Optuna hyperparameter optimization
    """
    
    print("="*70)
    print("OPTUNA HYPERPARAMETER OPTIMIZATION")
    print("="*70)
    print(f"Number of trials: {n_trials}")
    print(f"Phase 1: RMSE pre-training")
    print(f"Phase 2: Compound loss fine-tuning")
    print()
    
    n_features = X_train.shape[2]
    
    objective = create_optuna_objective(
        X_train, y_train, X_val, y_val,
        n_in, n_out, n_features, cgm_scaler
    )
    
    study = optuna.create_study(
        direction='minimize',
        study_name='hybrid_model_optimization',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )
    
    print("Starting optimization...")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print("\n" + "="*70)
    print("OPTIMIZATION COMPLETE")
    print("="*70)
    print(f"\nBest validation loss: {study.best_value:.6f}")
    print(f"Best trial: {study.best_trial.number}")
    print(f"\nBest hyperparameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    return study, study.best_params


def train_final_model_with_best_params(X_train, y_train, X_val, y_val, 
                                       best_params, n_in, n_out, cgm_scaler):
    """
    Train final model using best hyperparameters
    """
    
    print("\n" + "="*70)
    print("TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
    print("="*70)
    
    tf.keras.backend.clear_session()
    
    n_features = X_train.shape[2]
    
    # Build model
    model = Sequential([
        LSTM(best_params['lstm_units'], return_sequences=False, 
             input_shape=(n_in, n_features)),
        Dropout(best_params['dropout_rate']),
        Dense(n_out, activation='linear')
    ])
    
    print("\nModel architecture:")
    model.summary()
    
    # PHASE 1: RMSE
    print("\nPhase 1: RMSE pre-training...")
    optimizer_pre = Adam(learning_rate=best_params['lr_phase1'], 
                        clipnorm=best_params['clipnorm'])
    model.compile(optimizer=optimizer_pre, loss=rmse_loss, metrics=['mae'])
    
    callbacks_pre = [
        EarlyStopping(monitor='val_loss', patience=10, 
                     restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, 
                        verbose=1, min_lr=1e-7)
    ]
    
    history_pre = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=best_params['epochs_phase1'],
        batch_size=best_params['batch_size_phase1'],
        callbacks=callbacks_pre,
        verbose=2
    )
    
    # PHASE 2: COMPOUND LOSS
    print("\nPhase 2: Compound loss fine-tuning...")
    compound_loss = create_compound_glucose_loss(cgm_scaler)
    optimizer_post = Adam(learning_rate=best_params['lr_phase2'], 
                         clipnorm=best_params['clipnorm'])
    model.compile(optimizer=optimizer_post, loss=compound_loss)
    
    callbacks_post = [
        EarlyStopping(monitor='val_loss', patience=8, 
                     restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, 
                        verbose=1, min_lr=1e-7)
    ]
    
    history_post = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=best_params['epochs_phase2'],
        batch_size=best_params['batch_size_phase2'],
        callbacks=callbacks_post,
        verbose=2
    )
    
    print("\n✓ Final model training complete!")
    
    training_history = {
        'phase1_rmse': history_pre.history,
        'phase2_compound': history_post.history
    }
    
    return model, training_history


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("="*70)
    print("HYBRID RESIDUAL CORRECTOR WITH OPTUNA")
    print("="*70)
    
    # Verify required variables
    required_vars = ['train_subjects', 'train_ids', 'val_ids', 
                     'cgm_scaler', 'feature_scaler', 'N_IN', 'N_OUT', 
                     'MODEL_FEATURES']
    missing = [v for v in required_vars if v not in globals()]
    if missing:
        raise RuntimeError(f"Missing variables: {missing}. Run data processing first.")
    
    # ========================================
    # STEP 1: FIT BERGMAN MODEL
    # ========================================
    
    print("\n" + "="*70)
    print("STEP 1: Fitting Bergman Model on Training Data")
    print("="*70)
    
    bergman_model = BergmanMinimalModel()
    bergman_model.fit(train_subjects, train_ids, N_IN, N_OUT, cgm_scaler)
    
    # ========================================
    # STEP 2: BUILD RESIDUAL DATA
    # ========================================
    
    print("\n" + "="*70)
    print("STEP 2: Building Residual Training/Validation Data")
    print("="*70)
    
    val_subjects_dict = {sid: train_subjects[sid] for sid in val_ids}
    
    X_train_res, y_train_res = build_residual_data_from_subjects(
        train_subjects, train_ids, bergman_model, N_IN, N_OUT, 
        cgm_scaler, MODEL_FEATURES, feature_scaler, stride=5
    )
    
    X_val_res, y_val_res = build_residual_data_from_subjects(
        val_subjects_dict, val_ids, bergman_model, N_IN, N_OUT,
        cgm_scaler, MODEL_FEATURES, feature_scaler, stride=5
    )
    
    print(f"\nTraining samples: {len(X_train_res)} (stride=5)")
    print(f"Validation samples: {len(X_val_res)} (stride=5)")
    print(f"Feature dimensions: {X_train_res.shape[2]}")
    
    # ========================================
    # STEP 3: RUN OPTUNA OPTIMIZATION
    # ========================================
    
    study, best_params = optimize_hyperparameters(
        X_train_res, y_train_res,
        X_val_res, y_val_res,
        N_IN, N_OUT,
        cgm_scaler,
        n_trials=1  # Adjust based on computational budget
    )
    
    # ========================================
    # STEP 4: TRAIN FINAL MODEL
    # ========================================
    
    tuned_hybrid_corrector_model, training_history = train_final_model_with_best_params(
        X_train_res, y_train_res,
        X_val_res, y_val_res,
        best_params,
        N_IN, N_OUT,
        cgm_scaler
    )
    
    # ========================================
    # STEP 5: EVALUATE ON VALIDATION SET
    # ========================================
    
    print("\n" + "="*70)
    print("STEP 5: Evaluating Model on Validation Set")
    print("="*70)
    
    val_rmse, val_mae = evaluate_hybrid_predictions(
        val_subjects_dict, val_ids, bergman_model, tuned_hybrid_corrector_model,
        N_IN, N_OUT, cgm_scaler, MODEL_FEATURES, feature_scaler, stride=10
    )
    
    print(f"\nValidation Performance:")
    print(f"  RMSE: {val_rmse:.2f} mg/dL")
    print(f"  MAE: {val_mae:.2f} mg/dL")
    
    # ========================================
    # SUMMARY
    # ========================================
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE")
    print("="*70)
    print("\nSaved variables:")
    print("  - tuned_hybrid_corrector_model: Final trained LSTM corrector")
    print("  - bergman_model: Fitted Bergman minimal model")
    print("  - study: Optuna study object with all trials")
    print("  - best_params: Best hyperparameters found")
    print("  - training_history: Training history (both phases)")
    print("\nModel ready for deployment!")
    print("\nTo make predictions:")
    print("  1. Get Bergman baseline: bergman_pred = bergman_model.simulate(...)")
    print("  2. Predict residual: residual = tuned_hybrid_corrector_model.predict(...)")
    print("  3. Combine: final_pred = bergman_pred + residual")

HYBRID RESIDUAL CORRECTOR WITH OPTUNA

STEP 1: Fitting Bergman Model on Training Data
  Fitting Bergman model...
    Bergman MSE: 379.13

STEP 2: Building Residual Training/Validation Data


[I 2025-12-14 20:50:41,148] A new study created in memory with name: hybrid_model_optimization



Training samples: 28510 (stride=5)
Validation samples: 9648 (stride=5)
Feature dimensions: 7
OPTUNA HYPERPARAMETER OPTIMIZATION
Number of trials: 1
Phase 1: RMSE pre-training
Phase 2: Compound loss fine-tuning

Starting optimization...


  0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\msbdj\anaconda3\envs\ttk4250\lib\site-packages\keras\src\layers\rnn\rnn.py:205: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 40)             │         7,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │           246 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,926 (30.96 KB)

 Trainable params: 7,926 (30.96 KB)

 Non-trainable params: 0 (0.00 B)


Phase 1: RMSE pre-training...
Epoch 1/30
891/891 - 15s - 17ms/step - loss: 0.0312 - mae: 0.0196 - val_loss: 0.0262 - val_mae: 0.0159 - learning_rate: 0.0063
Epoch 2/30
891/891 - 5s - 6ms/step - loss: 0.0244 - mae: 0.0144 - val_loss: 0.0273 - val_mae: 0.0167 - learning_rate: 0.0063
Epoch 3/30
891/891 - 5s - 6ms/step - loss: 0.0236 - mae: 0.0139 - val_loss: 0.0250 - val_mae: 0.0155 - learning_rate: 0.0063
Epoch 4/30
891/891 - 5s - 6ms/step - loss: 0.0230 - mae: 0.0135 - val_loss: 0.0242 - val_mae: 0.0148 - learning_rate: 0.0063
Epoch 5/30
891/891 - 5s - 6ms/step - loss: 0.0227 - mae: 0.0133 - val_loss: 0.0234 - val_mae: 0.0142 - learning_rate: 0.0063
Epoch 6/30
891/891 - 5s - 6ms/step - loss: 0.0223 - mae: 0.0131 - val_loss: 0.0242 - val_mae: 0.0149 - learning_rate: 0.0063
Epoch 7/30
891/891 - 5s - 6ms/step - loss: 0.0221 - mae: 0.0128 - val_loss: 0.0235 - val_mae: 0.0145 - learning_rate: 0.0063
Epoch 8/30
891/891 - 5s - 6ms/step - loss: 0.0219 - mae: 0.0128 - val_loss: 0.0239 - val_mae

## Hybrid Model - Physics-Informed Neural Network (PINNs) - Without Fine-tuning:

In [20]:
"""
Physics-Informed Neural Network with Nested Cross-Validation
Single-phase training with RMSE + Physics + Insulin Consistency losses
Cross-validation on training subjects (OhioT1DM)

DEPENDENCIES:
This code requires the data processing code to be run first:
'Blood Glucose Forecasting Data Processing'

Required variables from data processing:
- train_subjects: Dict of processed subject DataFrames (OhioT1DM)
- train_sequences: Dict of training sequences {'X': X, 'y': y}
- val_sequences: Dict of validation sequences {'X': X, 'y': y}
- train_ids: List of training subject IDs
- val_ids: List of validation subject IDs
- cgm_scaler: Fitted MinMaxScaler for CGM
- feature_scaler: Fitted StandardScaler for features

FIXED: 
- Removed @tf.function from compute_pinn_losses to avoid variable creation issues
- Uses correct features from data processing pipeline:
  ['carbs', 'bolus', 'heartrate', 'steps', 'CGM_derivative', 'carbs_derivative']
- Only uses OhioT1DM training and validation data (NO BrisT1D test data)
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

# ============================================================================
# CONFIGURATION
# ============================================================================

# Data dimensions - MUST MATCH DATA PROCESSING CODE
N_IN = 12  # Input sequence length (60 min history)
N_OUT = 6  # Output prediction horizon (30 min forecast)

# Features from data processing pipeline - DO NOT MODIFY
# These are the shared features available in both OhioT1DM and BrisT1D
SHARED_FEATURES = ['carbs', 'bolus', 'heartrate', 'steps']
COMPUTED_FEATURES = ['CGM_derivative', 'carbs_derivative']
MODEL_FEATURES = SHARED_FEATURES + COMPUTED_FEATURES
# Total: ['carbs', 'bolus', 'heartrate', 'steps', 'CGM_derivative', 'carbs_derivative']

# Cross-validation settings
N_OUTER_FOLDS = 2  # Number of CV folds
RANDOM_SEED = 42

# Model hyperparameters
EPOCHS = 1  # Training epochs
BATCH_SIZE = 64
LEARNING_RATE = 0.001
LAMBDA_PHYSICS = 1.5  # Weight for physics loss
LAMBDA_INSULIN = 0.01  # Weight for insulin consistency loss
LSTM_UNITS = 128
DENSE_UNITS = 32

# Set random seeds
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("="*70)
print("PINN WITH NESTED CROSS-VALIDATION (SINGLE-PHASE)")
print("="*70)

# ============================================================================
# VERIFY DATA PROCESSING HAS BEEN RUN
# ============================================================================
print("\nVerifying required data from data processing code...")
required_vars = {
    'train_subjects': 'Processed training subject data',
    'train_sequences': 'Training sequences (X, y)',
    'val_sequences': 'Validation sequences (X, y)',
    'train_ids': 'Training subject IDs',
    'val_ids': 'Validation subject IDs',
    'cgm_scaler': 'Fitted CGM scaler',
    'feature_scaler': 'Fitted feature scaler'
}

missing_vars = []
for var_name, description in required_vars.items():
    if var_name not in globals():
        missing_vars.append(f"  - {var_name}: {description}")
    else:
        print(f"  ✓ {var_name}: {description}")

if missing_vars:
    print("\n" + "="*70)
    print("ERROR: Missing required variables from data processing!")
    print("="*70)
    print("\nThe following variables are required but not found:")
    for msg in missing_vars:
        print(msg)
    print("\nPlease run the data processing code first.")
    print("This code expects variables from: 'Blood Glucose Forecasting Data Processing'")
    raise RuntimeError("Data processing must be run before PINN training")

print("\n✓ All required data verified!")
print(f"\nData dimensions:")
print(f"  - Input sequence length (N_IN): {N_IN} timesteps")
print(f"  - Output horizon (N_OUT): {N_OUT} timesteps")
print(f"  - Features per timestep: 1 (CGM) + {len(MODEL_FEATURES)} (features) = {1+len(MODEL_FEATURES)}")
print(f"  - Expected X shape: (n_samples, {N_IN}, {1+len(MODEL_FEATURES)})")
print(f"  - Expected y shape: (n_samples, {N_OUT}, 1)")
print(f"\nInput features: {MODEL_FEATURES}")
print(f"Number of features: {len(MODEL_FEATURES)}")
print(f"Outer folds: {N_OUTER_FOLDS}")
print(f"Training on: OhioT1DM train subjects (cross-validation)")
print(f"Final evaluation on: OhioT1DM validation subjects (held-out)")
print(f"NOTE: BrisT1D test data is NOT used in this analysis")

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def inverse_transform_cgm_tf(scaled_cgm):
    """Convert scaled CGM [0,1] to mg/dL (TensorFlow)"""
    cgm_min = tf.constant(float(cgm_scaler.data_min_[0]), dtype=tf.float32)
    cgm_max = tf.constant(float(cgm_scaler.data_max_[0]), dtype=tf.float32)
    result = scaled_cgm * (cgm_max - cgm_min) + cgm_min
    return tf.clip_by_value(result, 20.0, 600.0)

def inverse_transform_cgm_np(scaled_cgm):
    """Convert scaled CGM [0,1] to mg/dL (NumPy)"""
    result = cgm_scaler.inverse_transform(scaled_cgm.reshape(-1, 1)).flatten()
    return np.clip(result, 20.0, 600.0)

def safe_softplus_inverse(value):
    """Compute inverse of softplus"""
    value = float(value)
    if value < 1e-7:
        return -10.0
    elif value > 20.0:
        return float(np.log(value))
    else:
        result = np.log(np.exp(value) - 1.0)
        return float(result) if np.isfinite(result) else float(np.log(value))

# ============================================================================
# BERGMAN MODEL
# ============================================================================

class BergmanModel:
    """Bergman minimal model for glucose-insulin dynamics"""
    def __init__(self, trainable=True, initial_params=None):
        if initial_params is None:
            init_p1, init_p2, init_p3, init_ki, init_Gb = 0.02, 0.015, 0.03, 0.15, 100.0
        else:
            init_p1, init_p2, init_p3, init_ki, init_Gb = initial_params
        
        self.berg_p1_raw = tf.Variable(safe_softplus_inverse(init_p1), dtype=tf.float32, 
                                       trainable=trainable, name='berg_p1_raw')
        self.berg_p2_raw = tf.Variable(safe_softplus_inverse(init_p2), dtype=tf.float32,
                                       trainable=trainable, name='berg_p2_raw')
        self.berg_p3_raw = tf.Variable(safe_softplus_inverse(init_p3), dtype=tf.float32,
                                       trainable=trainable, name='berg_p3_raw')
        self.berg_ki_raw = tf.Variable(safe_softplus_inverse(init_ki), dtype=tf.float32,
                                       trainable=trainable, name='berg_ki_raw')
        self.berg_Gb_raw = tf.Variable(safe_softplus_inverse(init_Gb), dtype=tf.float32,
                                       trainable=trainable, name='berg_Gb_raw')
    
    def get_positive_params(self):
        """Apply softplus to ensure positive parameters"""
        berg_p1 = tf.clip_by_value(tf.nn.softplus(self.berg_p1_raw), 1e-6, 0.1)
        berg_p2 = tf.clip_by_value(tf.nn.softplus(self.berg_p2_raw), 1e-6, 0.1)
        berg_p3 = tf.clip_by_value(tf.nn.softplus(self.berg_p3_raw), 1e-6, 0.5)
        berg_ki = tf.clip_by_value(tf.nn.softplus(self.berg_ki_raw), 0.01, 1.0)
        berg_Gb = tf.clip_by_value(tf.nn.softplus(self.berg_Gb_raw), 20.0, 500.0)
        return berg_p1, berg_p2, berg_p3, berg_ki, berg_Gb
    
    @property
    def trainable_variables(self):
        return [self.berg_p1_raw, self.berg_p2_raw, self.berg_p3_raw, 
                self.berg_ki_raw, self.berg_Gb_raw]
    
    @tf.function
    def simulate(self, G0, bolus_sequence, carbs_sequence):
        """Simulate glucose dynamics using Bergman equations"""
        berg_p1, berg_p2, berg_p3, berg_ki, berg_Gb = self.get_positive_params()
        
        G = tf.clip_by_value(tf.cast(G0, tf.float32), 50.0, 400.0)
        I = tf.zeros_like(G, dtype=tf.float32)
        
        glucose_trajectory = tf.TensorArray(tf.float32, size=N_OUT)
        exp_decay = tf.exp(-berg_ki)
        dt = 1.0
        
        for t in tf.range(N_OUT):
            bolus_t = tf.clip_by_value(tf.cast(bolus_sequence[:, t], tf.float32), 0.0, 50.0)
            I = I * exp_decay + bolus_t
            I = tf.clip_by_value(I, 0.0, 100.0)
            
            carbs_t = tf.clip_by_value(tf.cast(carbs_sequence[:, t], tf.float32), 0.0, 200.0)
            dG = -berg_p1 * (G - berg_Gb) - berg_p2 * I + berg_p3 * carbs_t
            dG = tf.clip_by_value(dG, -50.0, 50.0)
            
            G = G + dG * dt
            G = tf.clip_by_value(G, 40.0, 500.0)
            
            glucose_trajectory = glucose_trajectory.write(t, G)
        
        return tf.transpose(glucose_trajectory.stack(), perm=[1, 0])

# ============================================================================
# PINN MODEL
# ============================================================================

class PINNModel(tf.keras.Model):
    """Physics-Informed Neural Network with glucose and insulin predictions"""
    def __init__(self, n_features, bergman_model, lstm_units, dense_units):
        super().__init__()
        self.lstm_encoder = layers.LSTM(lstm_units, return_sequences=False)
        self.dense_1 = layers.Dense(dense_units, activation='relu')
        self.output_layer = layers.Dense(N_OUT * 2, activation='linear')  # Glucose + Insulin
        self.bergman = bergman_model
    
    def call(self, inputs, training=False):
        x = tf.cast(inputs, tf.float32)
        encoded = self.lstm_encoder(x)
        hidden = self.dense_1(encoded)
        output = self.output_layer(hidden)
        
        batch_size = tf.shape(output)[0]
        reshaped = tf.reshape(output, (batch_size, N_OUT, 2))
        
        glucose_pred = tf.nn.sigmoid(reshaped[:, :, 0])  # Scaled [0,1]
        insulin_pred = tf.nn.tanh(reshaped[:, :, 1])     # Normalized insulin effect
        
        return glucose_pred, insulin_pred
    
    @tf.function
    def compute_insulin_effect(self, bolus_sequence):
        """Compute insulin effect from bolus using Bergman kinetics"""
        _, _, _, berg_ki, _ = self.bergman.get_positive_params()
        
        batch_size = tf.shape(bolus_sequence)[0]
        I = tf.zeros((batch_size,), dtype=tf.float32)
        insulin_trajectory = tf.TensorArray(tf.float32, size=N_OUT)
        exp_decay = tf.exp(-berg_ki)
        
        for t in tf.range(N_OUT):
            bolus_t = tf.clip_by_value(tf.cast(bolus_sequence[:, t], tf.float32), 0.0, 50.0)
            I = I * exp_decay + bolus_t
            I = tf.clip_by_value(I, 0.0, 100.0)
            insulin_trajectory = insulin_trajectory.write(t, I)
        
        return tf.transpose(insulin_trajectory.stack(), perm=[1, 0])

# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

@tf.function
def rmse_loss(y_true, y_pred):
    """Root Mean Squared Error"""
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-7)

# REMOVED @tf.function decorator to fix variable creation issue
def compute_pinn_losses(X_batch, y_batch, G0_batch, bolus_batch, carbs_batch, 
                        model, lambda_phys, lambda_insulin):
    """Compute all three PINN loss components: Data + Physics + Insulin Consistency"""
    eps = 1e-7
    
    G_pred_scaled, I_pred = model(X_batch, training=True)
    
    # 1. Data loss (RMSE between predicted and actual glucose)
    data_loss = rmse_loss(y_batch, G_pred_scaled)
    
    # 2. Physics loss (consistency with Bergman model)
    G_bergman = model.bergman.simulate(G0_batch, bolus_batch, carbs_batch)
    G_pred_mgdl = inverse_transform_cgm_tf(G_pred_scaled)
    
    cgm_range = float(cgm_scaler.data_max_[0] - cgm_scaler.data_min_[0])
    physics_loss = tf.reduce_mean(tf.square(G_pred_mgdl - G_bergman)) / (cgm_range**2 + eps)
    
    # 3. Insulin consistency loss (predicted insulin should match Bergman kinetics)
    I_expected = model.compute_insulin_effect(bolus_batch)
    insulin_loss = tf.reduce_mean(tf.square(I_pred - I_expected))
    
    # Check for NaN and replace with large penalty
    data_loss = tf.where(tf.math.is_finite(data_loss), data_loss, tf.constant(1e6, dtype=tf.float32))
    physics_loss = tf.where(tf.math.is_finite(physics_loss), physics_loss, tf.constant(1e6, dtype=tf.float32))
    insulin_loss = tf.where(tf.math.is_finite(insulin_loss), insulin_loss, tf.constant(1e6, dtype=tf.float32))
    
    # Total loss: weighted combination of all three components
    total_loss = data_loss + lambda_phys * physics_loss + lambda_insulin * insulin_loss
    
    return total_loss, data_loss, physics_loss, insulin_loss

# ============================================================================
# TRAINING FUNCTION
# ============================================================================

def train_pinn_model(train_data, val_data, hyperparams, verbose=True):
    """
    Train PINN model with given hyperparameters (single phase)
    
    Args:
        train_data: dict with X, y, G0, bolus, carbs
        val_data: dict with X, y, G0, bolus, carbs (EVALUATION ONLY, NO TRAINING)
        hyperparams: dict with learning_rate, lambda_phys, lambda_ins, lstm_u, dense_u, epochs
        verbose: whether to print training progress
    
    Returns:
        model, val_rmse
    """
    # Extract hyperparameters
    learning_rate = hyperparams.get('learning_rate', LEARNING_RATE)
    lambda_phys = hyperparams.get('lambda_physics', LAMBDA_PHYSICS)
    lambda_ins = hyperparams.get('lambda_insulin', LAMBDA_INSULIN)
    lstm_units = hyperparams.get('lstm_units', LSTM_UNITS)
    dense_units = hyperparams.get('dense_units', DENSE_UNITS)
    epochs = hyperparams.get('epochs', EPOCHS)
    batch_size = hyperparams.get('batch_size', BATCH_SIZE)
    
    # Initialize Bergman model with data-driven initial guess
    median_G = np.clip(np.median(train_data['G0']), 80.0, 150.0)
    initial_params = (0.02, 0.015, 0.03, 0.15, median_G)
    bergman = BergmanModel(trainable=True, initial_params=initial_params)
    
    # Initialize PINN model
    n_features = train_data['X'].shape[2]
    model = PINNModel(n_features, bergman, lstm_units, dense_units)
    
    # Build the model by calling it once (this creates all variables)
    dummy_input = tf.zeros((1, N_IN, n_features), dtype=tf.float32)
    _ = model(dummy_input, training=False)
    
    # Create training dataset
    dataset = tf.data.Dataset.from_tensor_slices((
        train_data['X'], train_data['y'], train_data['G0'], 
        train_data['bolus'], train_data['carbs']
    )).shuffle(10000, seed=RANDOM_SEED).batch(batch_size).prefetch(2)
    
    # Single-phase training with Adam optimizer
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    
    for epoch in range(epochs):
        epoch_total_loss = 0.0
        epoch_data_loss = 0.0
        epoch_physics_loss = 0.0
        epoch_insulin_loss = 0.0
        n_batches = 0
        
        for X_batch, y_batch, G0_batch, bolus_batch, carbs_batch in dataset:
            with tf.GradientTape() as tape:
                total_loss, data_loss, physics_loss, insulin_loss = compute_pinn_losses(
                    X_batch, y_batch, G0_batch, bolus_batch, carbs_batch,
                    model, lambda_phys, lambda_ins
                )
                
                if not tf.math.is_finite(total_loss):
                    continue
            
            # Compute gradients
            grads = tape.gradient(total_loss, model.trainable_variables)
            if any(g is not None and tf.reduce_any(~tf.math.is_finite(g)) for g in grads):
                continue
            
            # Clip gradients to prevent instability
            grads, _ = tf.clip_by_global_norm(grads, 5.0)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            
            epoch_total_loss += float(total_loss.numpy())
            epoch_data_loss += float(data_loss.numpy())
            epoch_physics_loss += float(physics_loss.numpy())
            epoch_insulin_loss += float(insulin_loss.numpy())
            n_batches += 1
        
        if verbose and n_batches > 0 and (epoch + 1) % 10 == 0:
            avg_total = epoch_total_loss / n_batches
            avg_data = epoch_data_loss / n_batches
            avg_physics = epoch_physics_loss / n_batches
            avg_insulin = epoch_insulin_loss / n_batches
            print(f"  Epoch {epoch+1}/{epochs}: Total={avg_total:.4f}, Data={avg_data:.4f}, "
                  f"Physics={avg_physics:.4f}, Insulin={avg_insulin:.4f}")
    
    # Evaluate on validation data (INFERENCE ONLY - NO TRAINING)
    val_predictions_scaled, _ = model(val_data['X'].astype(np.float32), training=False)
    val_predictions_scaled = val_predictions_scaled.numpy()
    
    val_predictions_mgdl = inverse_transform_cgm_np(val_predictions_scaled)
    val_true_mgdl = inverse_transform_cgm_np(val_data['y'].flatten())
    
    val_predictions_mgdl = val_predictions_mgdl.reshape(-1, N_OUT)
    val_true_mgdl = val_true_mgdl.reshape(-1, N_OUT)
    
    val_rmse = np.sqrt(np.mean((val_predictions_mgdl - val_true_mgdl)**2))
    
    return model, val_rmse

# ============================================================================
# PREPARE DATA FOR CV
# ============================================================================

def prepare_fold_data(subject_ids, use_val_sequences=False):
    """
    Prepare training data for given subject IDs
    Extracts CGM initial conditions, bolus, and carbs from processed subjects
    
    This function uses data created by the data processing code:
    - train_subjects[sid]: DataFrame with columns CGM_smoothed, bolus, carbs, etc.
    - train_sequences[sid] or val_sequences[sid]: Dict with 'X' and 'y' arrays
    
    The sequences already have:
    - X: (n_samples, N_IN, n_features+1) - scaled CGM + scaled features
    - y: (n_samples, N_OUT, 1) - scaled future CGM values
    
    We additionally need for the physics model:
    - G0: Initial glucose values at each prediction window (from CGM_smoothed)
    - bolus: Future bolus values for Bergman model simulation
    - carbs: Future carbs values for Bergman model simulation
    
    Args:
        subject_ids: List of subject IDs to prepare data for
        use_val_sequences: If True, use val_sequences; otherwise use train_sequences
    
    Returns:
        Dict with keys: 'X', 'y', 'G0', 'bolus', 'carbs'
    """
    fold_data = {
        'X': [], 'y': [], 'G0': [], 'bolus': [], 'carbs': []
    }
    
    # Select the appropriate sequences dictionary
    sequences_dict = val_sequences if use_val_sequences else train_sequences
    
    for sid in subject_ids:
        # Check if subject exists in the sequences dictionary
        if sid not in sequences_dict:
            print(f"Warning: Subject {sid} not found in sequences, skipping...")
            continue
            
        subject_data = train_subjects[sid]
        subject_sequences = sequences_dict[sid]
        
        n_windows = subject_sequences['X'].shape[0]
        if n_windows == 0:
            continue
        
        fold_data['X'].append(subject_sequences['X'])
        fold_data['y'].append(subject_sequences['y'])
        
        # Extract initial glucose values (from smoothed CGM)
        G_initial = subject_data['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
        fold_data['G0'].append(G_initial)
        
        # Extract future bolus and carbs for physics model
        for w in range(n_windows):
            start_idx = N_IN + w
            bolus_future = subject_data['bolus'].values[start_idx : start_idx + N_OUT]
            carbs_future = subject_data['carbs'].values[start_idx : start_idx + N_OUT]
            fold_data['bolus'].append(bolus_future)
            fold_data['carbs'].append(carbs_future)
    
    if len(fold_data['X']) == 0:
        raise ValueError(f"No valid sequences found for the given subject IDs")
    
    return {
        'X': np.concatenate(fold_data['X'], axis=0).astype(np.float32),
        'y': np.concatenate(fold_data['y'], axis=0).reshape(-1, N_OUT).astype(np.float32),
        'G0': np.concatenate(fold_data['G0'], axis=0).astype(np.float32),
        'bolus': np.array(fold_data['bolus'], dtype=np.float32),
        'carbs': np.array(fold_data['carbs'], dtype=np.float32)
    }

# ============================================================================
# NESTED CROSS-VALIDATION (ON TRAINING SUBJECTS ONLY)
# ============================================================================

print("\n" + "="*70)
print("NESTED CROSS-VALIDATION (Training Subjects Only)")
print("="*70)

# Get training subject IDs only (val_ids are held out for final evaluation)
cv_subject_ids = np.array(sorted(train_ids))

# Outer CV loop on training subjects only
outer_cv = KFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED)
cv_results = []

for fold_idx, (train_idx, val_idx) in enumerate(outer_cv.split(cv_subject_ids)):
    print(f"\n{'='*70}")
    print(f"CV FOLD {fold_idx + 1}/{N_OUTER_FOLDS}")
    print(f"{'='*70}")
    
    # Split training subjects into fold-train and fold-val
    fold_train_ids = cv_subject_ids[train_idx]
    fold_val_ids = cv_subject_ids[val_idx]
    
    print(f"Fold training subjects: {len(fold_train_ids)}")
    print(f"Fold validation subjects: {len(fold_val_ids)}")
    
    # Prepare data for this fold
    fold_train_data = prepare_fold_data(fold_train_ids)
    fold_val_data = prepare_fold_data(fold_val_ids)
    
    print(f"Training samples: {fold_train_data['X'].shape[0]}")
    print(f"Validation samples: {fold_val_data['X'].shape[0]}")
    
    # Train model with default hyperparameters
    hyperparams = {
        'learning_rate': LEARNING_RATE,
        'lambda_physics': LAMBDA_PHYSICS,
        'lambda_insulin': LAMBDA_INSULIN,
        'lstm_units': LSTM_UNITS,
        'dense_units': DENSE_UNITS,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE
    }
    
    # Train ONLY on fold_train_data, evaluate on fold_val_data (NO training on val data)
    fold_model, fold_val_rmse = train_pinn_model(
        fold_train_data, fold_val_data, hyperparams, verbose=True
    )
    
    print(f"\nFold {fold_idx + 1} Validation RMSE: {fold_val_rmse:.2f} mg/dL")
    
    cv_results.append({
        'fold': fold_idx + 1,
        'model': fold_model,
        'val_rmse': fold_val_rmse,
        'train_ids': fold_train_ids,
        'val_ids': fold_val_ids
    })

# ============================================================================
# CV SUMMARY
# ============================================================================

print("\n" + "="*70)
print("CROSS-VALIDATION RESULTS SUMMARY")
print("="*70)

val_rmses = [r['val_rmse'] for r in cv_results]
print(f"\nValidation RMSE across folds:")
for i, rmse in enumerate(val_rmses):
    print(f"  Fold {i+1}: {rmse:.2f} mg/dL")

print(f"\nMean CV RMSE: {np.mean(val_rmses):.2f} ± {np.std(val_rmses):.2f} mg/dL")
print(f"Best fold: Fold {np.argmin(val_rmses) + 1} (RMSE: {np.min(val_rmses):.2f} mg/dL)")

# Select best model from CV (based on validation performance)
best_fold_idx = np.argmin(val_rmses)
best_model = cv_results[best_fold_idx]['model']

print(f"\nSelected best model from Fold {best_fold_idx + 1} for final evaluation")

# ============================================================================
# FINAL EVALUATION ON VALIDATION SET (OhioT1DM Held-Out Subjects)
# ============================================================================

print("\n" + "="*70)
print("FINAL EVALUATION ON HELD-OUT VALIDATION SET")
print("="*70)
print(f"Dataset: OhioT1DM validation subjects")
print(f"Validation subjects: {len(val_ids)}")
print(f"NOTE: BrisT1D test data is NOT used in this analysis")

# Prepare validation data (NO TRAINING on validation data)
final_val_data = prepare_fold_data(val_ids, use_val_sequences=True)

print(f"\nFinal validation samples: {final_val_data['X'].shape[0]}")

# Make predictions on validation set (INFERENCE ONLY)
final_predictions_scaled, _ = best_model(final_val_data['X'], training=False)
final_predictions_scaled = final_predictions_scaled.numpy()

final_predictions_mgdl = inverse_transform_cgm_np(final_predictions_scaled)
final_true_mgdl = inverse_transform_cgm_np(final_val_data['y'].flatten())

final_predictions_mgdl = final_predictions_mgdl.reshape(-1, N_OUT)
final_true_mgdl = final_true_mgdl.reshape(-1, N_OUT)

# Calculate final validation metrics
final_rmse = np.sqrt(np.mean((final_predictions_mgdl - final_true_mgdl)**2))
final_mae = np.mean(np.abs(final_predictions_mgdl - final_true_mgdl))

print(f"\nFinal Validation Set Metrics (OhioT1DM Held-Out):")
print(f"  RMSE: {final_rmse:.2f} mg/dL")
print(f"  MAE: {final_mae:.2f} mg/dL")

# ============================================================================
# PRINT LEARNED BERGMAN PARAMETERS
# ============================================================================

print("\n" + "="*70)
print("LEARNED BERGMAN PARAMETERS (Best Model)")
print("="*70)

berg_p1, berg_p2, berg_p3, berg_ki, berg_Gb = best_model.bergman.get_positive_params()
print(f"  p1 (glucose effectiveness): {float(berg_p1.numpy()):.6f}")
print(f"  p2 (insulin sensitivity): {float(berg_p2.numpy()):.6f}")
print(f"  p3 (carb absorption rate): {float(berg_p3.numpy()):.6f}")
print(f"  ki (insulin decay rate): {float(berg_ki.numpy()):.6f}")
print(f"  Gb (basal glucose): {float(berg_Gb.numpy()):.2f} mg/dL")

print("\n" + "="*70)
print("NESTED CROSS-VALIDATION COMPLETE")
print("="*70)
print(f"""
Summary:
  ✓ Features used: {MODEL_FEATURES}
  ✓ {N_OUTER_FOLDS}-fold cross-validation on OhioT1DM training subjects
  ✓ Single-phase training with 3 losses:
    - Data loss (RMSE)
    - Physics loss (Bergman model)
    - Insulin consistency loss
  ✓ Mean CV RMSE: {np.mean(val_rmses):.2f} ± {np.std(val_rmses):.2f} mg/dL
  ✓ Final Validation RMSE: {final_rmse:.2f} mg/dL (OhioT1DM held-out)
  ✓ Final Validation MAE: {final_mae:.2f} mg/dL
  ✓ No data leakage: Validation set never used in training/model selection
  ✓ BrisT1D test data NOT used (only OhioT1DM train/val)
  
Data Structure Used:
  FROM DATA PROCESSING CODE:
  - train_subjects[sid]: DataFrame with CGM_smoothed, bolus, carbs, features
  - train_sequences[sid]: Dict with 'X' (input features + CGM), 'y' (target CGM)
  - val_sequences[sid]: Dict with 'X', 'y' for validation subjects
  - cgm_scaler: MinMaxScaler fitted on training CGM only
  - feature_scaler: StandardScaler fitted on training features only
  
  X shape: (n_samples, {N_IN}, {len(MODEL_FEATURES)+1})
    - First column: scaled CGM values
    - Remaining {len(MODEL_FEATURES)} columns: scaled features
  y shape: (n_samples, {N_OUT}, 1)
    - Scaled future CGM values
  
Variables available:
  - cv_results: List of results from each CV fold
  - best_model: Best performing model from CV
  - final_predictions_mgdl: Predictions on held-out validation set
  - final_true_mgdl: True values from held-out validation set
  - train_ids: Training subject IDs (OhioT1DM)
  - val_ids: Held-out validation subject IDs (OhioT1DM)
  
Next steps:
  - To evaluate on BrisT1D test data, use the best_model on test_sequences
  - This will test generalization to a different dataset
""")

PINN WITH NESTED CROSS-VALIDATION (SINGLE-PHASE)

Verifying required data from data processing code...
  ✓ train_subjects: Processed training subject data
  ✓ train_sequences: Training sequences (X, y)
  ✓ val_sequences: Validation sequences (X, y)
  ✓ train_ids: Training subject IDs
  ✓ val_ids: Validation subject IDs
  ✓ cgm_scaler: Fitted CGM scaler
  ✓ feature_scaler: Fitted feature scaler

✓ All required data verified!

Data dimensions:
  - Input sequence length (N_IN): 12 timesteps
  - Output horizon (N_OUT): 6 timesteps
  - Features per timestep: 1 (CGM) + 6 (features) = 7
  - Expected X shape: (n_samples, 12, 7)
  - Expected y shape: (n_samples, 6, 1)

Input features: ['carbs', 'bolus', 'heartrate', 'steps', 'CGM_derivative', 'carbs_derivative']
Number of features: 6
Outer folds: 2
Training on: OhioT1DM train subjects (cross-validation)
Final evaluation on: OhioT1DM validation subjects (held-out)
NOTE: BrisT1D test data is NOT used in this analysis

NESTED CROSS-VALIDATION (Tra

## PINN - Fine-tuned with Optuna

In [23]:
"""
Nested Cross-Validation with Optuna Hyperparameter Tuning for PINN Model
Outer loop: K-fold CV for unbiased performance estimation
Inner loop: Optuna hyperparameter optimization
Uses ONLY OhioT1DM data - NO test data usage
"""

import optuna
from optuna.samplers import TPESampler
import numpy as np
from sklearn.model_selection import KFold

# ============================================================================
# CONFIGURATION
# ============================================================================

# Nested CV settings
N_OUTER_FOLDS = 2  # Outer CV folds for performance estimation
N_INNER_TRIALS = 2  # Optuna trials per outer fold
INNER_VAL_SIZE = 0.2  # Inner validation split for hyperparameter tuning

# Optuna settings
OPTUNA_TIMEOUT = 1800  # 30 minutes per outer fold
RANDOM_SEED = 42

print("="*80)
print("NESTED CROSS-VALIDATION WITH OPTUNA HYPERPARAMETER OPTIMIZATION")
print("="*80)
print(f"Outer folds: {N_OUTER_FOLDS} (for performance estimation)")
print(f"Inner trials per fold: {N_INNER_TRIALS} (for hyperparameter tuning)")
print(f"Inner validation split: {INNER_VAL_SIZE * 100:.0f}%")
print(f"Data: OhioT1DM ONLY (train + validation subjects)")

# ============================================================================
# HELPER FUNCTION: PREPARE FOLD DATA
# ============================================================================

def prepare_fold_data(subject_ids):
    """
    Prepare data for a fold given list of subject IDs
    Returns dict with X, y, G0, bolus, carbs arrays
    """
    fold_data = {
        'X': [], 'y': [], 'G0': [], 'bolus': [], 'carbs': []
    }
    
    for sid in subject_ids:
        # Check if subject exists in train_subjects
        if sid not in train_subjects:
            print(f"  Warning: Subject {sid} not found in train_subjects, skipping")
            continue
            
        # Check if subject has sequences
        if sid not in train_sequences:
            print(f"  Warning: Subject {sid} has no sequences, skipping")
            continue
        
        subject_data = train_subjects[sid]
        subject_sequences = train_sequences[sid]
        
        n_windows = subject_sequences['X'].shape[0]
        if n_windows == 0:
            continue
        
        fold_data['X'].append(subject_sequences['X'])
        fold_data['y'].append(subject_sequences['y'])
        
        # Get initial glucose values
        G_initial = subject_data['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
        fold_data['G0'].append(G_initial)
        
        # Get future bolus and carbs for each window
        for w in range(n_windows):
            start_idx = N_IN + w
            bolus_future = subject_data['bolus'].values[start_idx : start_idx + N_OUT]
            carbs_future = subject_data['carbs'].values[start_idx : start_idx + N_OUT]
            fold_data['bolus'].append(bolus_future)
            fold_data['carbs'].append(carbs_future)
    
    if len(fold_data['X']) == 0:
        raise ValueError(f"No valid sequences found for the given subject IDs: {subject_ids}")
    
    return {
        'X': np.concatenate(fold_data['X'], axis=0).astype(np.float32),
        'y': np.concatenate(fold_data['y'], axis=0).reshape(-1, N_OUT).astype(np.float32),
        'G0': np.concatenate(fold_data['G0'], axis=0).astype(np.float32),
        'bolus': np.array(fold_data['bolus'], dtype=np.float32),
        'carbs': np.array(fold_data['carbs'], dtype=np.float32)
    }

# ============================================================================
# PREPARE DATA - USE ONLY TRAIN SUBJECTS FROM OhioT1DM
# ============================================================================

# Combine train_ids and val_ids for nested CV (all OhioT1DM subjects)
all_train_ids = np.array(sorted(list(set(train_ids) | set(val_ids))))
print(f"\nTotal subjects for nested CV: {len(all_train_ids)}")
print(f"  (Combined train_ids: {len(train_ids)} + val_ids: {len(val_ids)})")

# ============================================================================
# NESTED CROSS-VALIDATION
# ============================================================================

# Initialize outer CV
outer_cv = KFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# Storage for results
outer_fold_results = []
all_best_hyperparams = []

print("\n" + "="*80)
print("STARTING NESTED CROSS-VALIDATION")
print("="*80)

# Outer loop: Performance estimation
for outer_fold, (train_idx, val_idx) in enumerate(outer_cv.split(all_train_ids), 1):
    
    print(f"\n{'='*80}")
    print(f"OUTER FOLD {outer_fold}/{N_OUTER_FOLDS}")
    print(f"{'='*80}")
    
    # Split subjects for this outer fold
    outer_train_ids = all_train_ids[train_idx]
    outer_val_ids = all_train_ids[val_idx]
    
    print(f"Outer train subjects: {len(outer_train_ids)}")
    print(f"Outer validation subjects: {len(outer_val_ids)}")
    
    # ========================================================================
    # INNER LOOP: Hyperparameter Optimization
    # ========================================================================
    
    print(f"\n--- Inner Loop: Hyperparameter Optimization ---")
    
    # Split outer training data for inner optimization
    np.random.seed(RANDOM_SEED + outer_fold)
    shuffled_train_ids = np.random.permutation(outer_train_ids)
    n_inner_train = int(len(shuffled_train_ids) * (1 - INNER_VAL_SIZE))
    inner_train_ids = shuffled_train_ids[:n_inner_train]
    inner_val_ids = shuffled_train_ids[n_inner_train:]
    
    print(f"Inner train subjects: {len(inner_train_ids)}")
    print(f"Inner validation subjects: {len(inner_val_ids)}")
    
    # Prepare inner data
    inner_train_data = prepare_fold_data(inner_train_ids)
    inner_val_data = prepare_fold_data(inner_val_ids)
    
    print(f"Inner train samples: {inner_train_data['X'].shape[0]}")
    print(f"Inner validation samples: {inner_val_data['X'].shape[0]}")
    
    # Define Optuna objective for this fold
    def objective(trial):
        hyperparams = {
            'lr_phase_1': trial.suggest_float('lr_phase_1', 1e-4, 1e-2, log=True),
            'lr_phase_2': trial.suggest_float('lr_phase_2', 1e-5, 1e-3, log=True),
            'lambda_physics': trial.suggest_float('lambda_physics', 0.1, 5.0),
            'lambda_insulin': trial.suggest_float('lambda_insulin', 0.001, 0.1, log=True),
            'lstm_units': trial.suggest_categorical('lstm_units', [64, 128, 256]),
            'dense_units': trial.suggest_categorical('dense_units', [16, 32, 64]),
            'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
            'epochs_phase_1': trial.suggest_int('epochs_phase_1', 20, 100),
            'epochs_phase_2': trial.suggest_int('epochs_phase_2', 10, 50),
        }
        
        _, val_rmse = train_pinn_model(
            inner_train_data, 
            inner_val_data, 
            hyperparams, 
            verbose=False
        )
        
        trial.report(val_rmse, step=0)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        return val_rmse
    
    # Run Optuna optimization
    study = optuna.create_study(
        direction='minimize',
        sampler=TPESampler(seed=RANDOM_SEED + outer_fold),
        pruner=optuna.pruners.MedianPruner(),
        study_name=f'fold_{outer_fold}'
    )
    
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    print(f"\nRunning Optuna optimization ({N_INNER_TRIALS} trials)...")
    study.optimize(
        objective, 
        n_trials=N_INNER_TRIALS,
        timeout=OPTUNA_TIMEOUT,
        show_progress_bar=True
    )
    
    best_trial = study.best_trial
    print(f"\nBest hyperparameters for fold {outer_fold}:")
    print(f"  Validation RMSE: {best_trial.value:.2f} mg/dL")
    for key, value in best_trial.params.items():
        print(f"  {key}: {value}")
    
    all_best_hyperparams.append(best_trial.params)
    
    # ========================================================================
    # Train final model for this fold with best hyperparameters
    # ========================================================================
    
    print(f"\n--- Training Final Model for Fold {outer_fold} ---")
    
    # Use ALL outer training data (no further splitting)
    outer_train_data = prepare_fold_data(outer_train_ids)
    outer_val_data = prepare_fold_data(outer_val_ids)
    
    print(f"Training on {len(outer_train_ids)} subjects ({outer_train_data['X'].shape[0]} samples)")
    print(f"Validating on {len(outer_val_ids)} subjects ({outer_val_data['X'].shape[0]} samples)")
    
    # Train with best hyperparameters
    fold_model, _ = train_pinn_model(
        outer_train_data, 
        outer_val_data,  # Only for monitoring, not for training decisions
        best_trial.params, 
        verbose=False
    )
    
    # Evaluate on outer validation fold
    val_predictions_scaled, _ = fold_model(outer_val_data['X'], training=False)
    val_predictions_scaled = val_predictions_scaled.numpy()
    
    val_predictions_mgdl = inverse_transform_cgm_np(val_predictions_scaled)
    val_true_mgdl = inverse_transform_cgm_np(outer_val_data['y'].flatten())
    
    val_predictions_mgdl = val_predictions_mgdl.reshape(-1, N_OUT)
    val_true_mgdl = val_true_mgdl.reshape(-1, N_OUT)
    
    fold_rmse = np.sqrt(np.mean((val_predictions_mgdl - val_true_mgdl)**2))
    fold_mae = np.mean(np.abs(val_predictions_mgdl - val_true_mgdl))
    
    print(f"\nFold {outer_fold} Validation Performance:")
    print(f"  RMSE: {fold_rmse:.2f} mg/dL")
    print(f"  MAE: {fold_mae:.2f} mg/dL")
    
    # Store results
    outer_fold_results.append({
        'fold': outer_fold,
        'rmse': fold_rmse,
        'mae': fold_mae,
        'best_hyperparams': best_trial.params,
        'n_train_subjects': len(outer_train_ids),
        'n_val_subjects': len(outer_val_ids)
    })

# ============================================================================
# NESTED CV RESULTS
# ============================================================================

print("\n" + "="*80)
print("NESTED CROSS-VALIDATION RESULTS")
print("="*80)

rmse_scores = [r['rmse'] for r in outer_fold_results]
mae_scores = [r['mae'] for r in outer_fold_results]

print(f"\nPer-Fold Validation Performance:")
for result in outer_fold_results:
    print(f"  Fold {result['fold']}: RMSE={result['rmse']:.2f} mg/dL, MAE={result['mae']:.2f} mg/dL")

print(f"\nNested CV Summary Statistics:")
print(f"  RMSE: {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f} mg/dL")
print(f"  MAE:  {np.mean(mae_scores):.2f} ± {np.std(mae_scores):.2f} mg/dL")
print(f"  Min RMSE: {np.min(rmse_scores):.2f} mg/dL")
print(f"  Max RMSE: {np.max(rmse_scores):.2f} mg/dL")

# ============================================================================
# AGGREGATE BEST HYPERPARAMETERS
# ============================================================================

print("\n" + "="*80)
print("AGGREGATED HYPERPARAMETERS ACROSS FOLDS")
print("="*80)

# Aggregate hyperparameters (median for continuous, mode for categorical)
aggregated_hyperparams = {}

# Continuous parameters: use median
continuous_params = ['lr_phase_1', 'lr_phase_2', 'lambda_physics', 'lambda_insulin']
for param in continuous_params:
    values = [h[param] for h in all_best_hyperparams]
    aggregated_hyperparams[param] = float(np.median(values))

# Categorical parameters: use mode (most common)
categorical_params = ['lstm_units', 'dense_units', 'batch_size']
for param in categorical_params:
    values = [h[param] for h in all_best_hyperparams]
    aggregated_hyperparams[param] = max(set(values), key=values.count)

# Integer parameters: use median (rounded)
int_params = ['epochs_phase_1', 'epochs_phase_2']
for param in int_params:
    values = [h[param] for h in all_best_hyperparams]
    aggregated_hyperparams[param] = int(np.median(values))

print("\nAggregated hyperparameters (median/mode across folds):")
for key, value in aggregated_hyperparams.items():
    print(f"  {key}: {value}")

# ============================================================================
# OPTIONAL: TRAIN FINAL MODEL ON ALL DATA
# ============================================================================

print("\n" + "="*80)
print("OPTIONAL: TRAINING FINAL MODEL ON ALL AVAILABLE DATA")
print("="*80)

full_train_data = prepare_fold_data(all_train_ids)
print(f"Training on all {len(all_train_ids)} subjects ({full_train_data['X'].shape[0]} samples)")

# Train final model with aggregated hyperparameters (no validation monitoring)
final_model, _ = train_pinn_model(
    full_train_data, 
    None,  # No validation data
    aggregated_hyperparams, 
    verbose=True
)

print("\nFinal model trained successfully!")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*80)
print("NESTED CROSS-VALIDATION COMPLETE")
print("="*80)
print(f"""
Summary:
  ✓ Nested CV Performance: {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f} mg/dL RMSE
  ✓ {N_OUTER_FOLDS} outer folds completed
  ✓ {N_INNER_TRIALS} hyperparameter trials per fold
  ✓ Data used: OhioT1DM subjects ONLY
  ✓ No test data used (BrisT1D kept completely separate)
  ✓ Unbiased performance estimate from cross-validation
  
Variables available:
  - outer_fold_results: List of all fold results
  - aggregated_hyperparams: Best hyperparameters aggregated across folds
  - final_model: Model trained on all available OhioT1DM data
  - all_train_ids: All subject IDs used in nested CV
  
Next steps:
  - Use 'final_model' for deployment or further analysis
  - Test on BrisT1D separately if needed (completely independent evaluation)
  - Report: {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f} mg/dL as unbiased estimate
""")

NESTED CROSS-VALIDATION WITH OPTUNA HYPERPARAMETER OPTIMIZATION
Outer folds: 2 (for performance estimation)
Inner trials per fold: 2 (for hyperparameter tuning)
Inner validation split: 20%
Data: OhioT1DM ONLY (train + validation subjects)

Total subjects for nested CV: 12
  (Combined train_ids: 9 + val_ids: 3)

STARTING NESTED CROSS-VALIDATION

OUTER FOLD 1/2
Outer train subjects: 6
Outer validation subjects: 6

--- Inner Loop: Hyperparameter Optimization ---
Inner train subjects: 4
Inner validation subjects: 2
Inner train samples: 62928
Inner validation samples: 31676

Running Optuna optimization (2 trials)...


  0%|          | 0/2 [00:00<?, ?it/s]


Best hyperparameters for fold 1:
  Validation RMSE: 14.56 mg/dL
  lr_phase_1: 0.0001698670453505509
  lr_phase_2: 0.00016524680777017762
  lambda_physics: 0.7536157245113426
  lambda_insulin: 0.0030281629390877954
  lstm_units: 128
  dense_units: 64
  batch_size: 64
  epochs_phase_1: 24
  epochs_phase_2: 45

--- Training Final Model for Fold 1 ---
Training on 6 subjects (94604 samples)
Validating on 6 subjects (47927 samples)

Fold 1 Validation Performance:
  RMSE: 15.64 mg/dL
  MAE: 10.01 mg/dL

OUTER FOLD 2/2
Outer train subjects: 6
Outer validation subjects: 6

--- Inner Loop: Hyperparameter Optimization ---
Inner train subjects: 4
Inner validation subjects: 2
Inner train samples: 31701
Inner validation samples: 16226

Running Optuna optimization (2 trials)...


  0%|          | 0/2 [00:00<?, ?it/s]


Best hyperparameters for fold 2:
  Validation RMSE: 16.61 mg/dL
  lr_phase_1: 0.00467395253077256
  lr_phase_2: 1.6202879744486195e-05
  lambda_physics: 3.7487383602296376
  lambda_insulin: 0.005260192921410246
  lstm_units: 128
  dense_units: 64
  batch_size: 32
  epochs_phase_1: 29
  epochs_phase_2: 18

--- Training Final Model for Fold 2 ---
Training on 6 subjects (47927 samples)
Validating on 6 subjects (94604 samples)

Fold 2 Validation Performance:
  RMSE: 200.15 mg/dL
  MAE: 158.64 mg/dL

NESTED CROSS-VALIDATION RESULTS

Per-Fold Validation Performance:
  Fold 1: RMSE=15.64 mg/dL, MAE=10.01 mg/dL
  Fold 2: RMSE=200.15 mg/dL, MAE=158.64 mg/dL

Nested CV Summary Statistics:
  RMSE: 107.89 ± 92.26 mg/dL
  MAE:  84.32 ± 74.31 mg/dL
  Min RMSE: 15.64 mg/dL
  Max RMSE: 200.15 mg/dL

AGGREGATED HYPERPARAMETERS ACROSS FOLDS

Aggregated hyperparameters (median/mode across folds):
  lr_phase_1: 0.0024219097880615555
  lr_phase_2: 9.072484375733191e-05
  lambda_physics: 2.25117704237049
 

TypeError: 'NoneType' object is not subscriptable

In [ ]:
"""
Optuna Hyperparameter Tuning for PINN Model
Uses validation data from OhioT1DM for optimization
Test data (BrisT1D) remains completely untouched
"""

import optuna
from optuna.samplers import TPESampler
import numpy as np
from sklearn.model_selection import train_test_split

# ============================================================================
# CONFIGURATION
# ============================================================================

N_TRIALS = 50  # Number of Optuna trials
N_JOBS = 1  # Parallel jobs (set to -1 for all cores, but TensorFlow may conflict)
TIMEOUT = 3600  # Max time in seconds (1 hour)
PRUNING_ENABLED = True  # Enable early stopping of unpromising trials

# Validation split for hyperparameter tuning
HYPERPARAM_VAL_SIZE = 0.2  # 20% of training data for validation
RANDOM_SEED = 42

print("="*70)
print("OPTUNA HYPERPARAMETER OPTIMIZATION")
print("="*70)
print(f"Number of trials: {N_TRIALS}")
print(f"Validation split: {HYPERPARAM_VAL_SIZE * 100:.0f}% of training data")
print(f"Test data: BrisT1D (completely held out)")

# ============================================================================
# PREPARE DATA FOR HYPERPARAMETER TUNING
# ============================================================================

# Split training subjects into train/val for hyperparameter tuning
all_train_ids = np.array(sorted(train_subjects.keys()))
hp_train_ids, hp_val_ids = train_test_split(
    all_train_ids, 
    test_size=HYPERPARAM_VAL_SIZE, 
    random_state=RANDOM_SEED
)

print(f"\nHyperparameter tuning data split:")
print(f"  Training subjects: {len(hp_train_ids)}")
print(f"  Validation subjects: {len(hp_val_ids)}")

# Prepare data
hp_train_data = prepare_fold_data(hp_train_ids)
hp_val_data = prepare_fold_data(hp_val_ids)

print(f"  Training samples: {hp_train_data['X'].shape[0]}")
print(f"  Validation samples: {hp_val_data['X'].shape[0]}")

# ============================================================================
# OPTUNA OBJECTIVE FUNCTION
# ============================================================================

def objective(trial):
    """
    Optuna objective function for hyperparameter optimization
    
    Args:
        trial: Optuna trial object
    
    Returns:
        validation RMSE (lower is better)
    """
    
    # Sample hyperparameters
    hyperparams = {
        # Learning rates
        'lr_phase_1': trial.suggest_float('lr_phase_1', 1e-4, 1e-2, log=True),
        'lr_phase_2': trial.suggest_float('lr_phase_2', 1e-5, 1e-3, log=True),
        
        # Physics loss weights
        'lambda_physics': trial.suggest_float('lambda_physics', 0.1, 5.0),
        'lambda_insulin': trial.suggest_float('lambda_insulin', 0.001, 0.1, log=True),
        
        # Architecture
        'lstm_units': trial.suggest_categorical('lstm_units', [64, 128, 256]),
        'dense_units': trial.suggest_categorical('dense_units', [16, 32, 64]),
        
        # Training
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'epochs_phase_1': trial.suggest_int('epochs_phase_1', 20, 100),
        'epochs_phase_2': trial.suggest_int('epochs_phase_2', 10, 50),
    }
    
    # Train model with sampled hyperparameters
    try:
        _, val_rmse = train_pinn_model(
            hp_train_data, 
            hp_val_data, 
            hyperparams, 
            verbose=False  # Suppress output for cleaner logs
        )
        
        # Report intermediate value for pruning
        trial.report(val_rmse, step=0)
        
        # Check if trial should be pruned
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        return val_rmse
    
    except Exception as e:
        # Return high error if training fails
        print(f"Trial {trial.number} failed: {e}")
        return 1000.0

# ============================================================================
# RUN OPTUNA OPTIMIZATION
# ============================================================================

print("\n" + "="*70)
print("STARTING HYPERPARAMETER OPTIMIZATION")
print("="*70)

# Create study
sampler = TPESampler(seed=RANDOM_SEED)
pruner = optuna.pruners.MedianPruner() if PRUNING_ENABLED else optuna.pruners.NopPruner()

study = optuna.create_study(
    direction='minimize',  # Minimize RMSE
    sampler=sampler,
    pruner=pruner,
    study_name='pinn_hyperparam_optimization'
)

# Optimize
study.optimize(
    objective, 
    n_trials=N_TRIALS,
    timeout=TIMEOUT,
    n_jobs=N_JOBS,
    show_progress_bar=True
)

# ============================================================================
# RESULTS
# ============================================================================

print("\n" + "="*70)
print("OPTIMIZATION COMPLETE")
print("="*70)

print(f"\nNumber of finished trials: {len(study.trials)}")
print(f"Number of pruned trials: {len(study.get_trials(states=[optuna.trial.TrialState.PRUNED]))}")
print(f"Number of complete trials: {len(study.get_trials(states=[optuna.trial.TrialState.COMPLETE]))}")

print(f"\nBest trial:")
best_trial = study.best_trial
print(f"  Value (RMSE): {best_trial.value:.2f} mg/dL")

print(f"\nBest hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

# ============================================================================
# VISUALIZATION
# ============================================================================

print("\n" + "="*70)
print("GENERATING OPTIMIZATION PLOTS")
print("="*70)

# Plot optimization history
fig1 = optuna.visualization.plot_optimization_history(study)
fig1.update_layout(title="Optimization History")
fig1.show()

# Plot parameter importances
fig2 = optuna.visualization.plot_param_importances(study)
fig2.update_layout(title="Hyperparameter Importances")
fig2.show()

# Plot parallel coordinate
fig3 = optuna.visualization.plot_parallel_coordinate(study)
fig3.update_layout(title="Parallel Coordinate Plot")
fig3.show()

# Plot contour (for top parameters)
if len(best_trial.params) >= 2:
    param_names = list(best_trial.params.keys())[:2]
    fig4 = optuna.visualization.plot_contour(study, params=param_names)
    fig4.update_layout(title=f"Contour Plot: {param_names[0]} vs {param_names[1]}")
    fig4.show()

# ============================================================================
# TRAIN FINAL MODEL WITH BEST HYPERPARAMETERS
# ============================================================================

print("\n" + "="*70)
print("TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
print("="*70)

# Prepare full training data (all OhioT1DM subjects)
full_train_data = prepare_fold_data(all_train_ids)

print(f"Training on all OhioT1DM subjects: {len(all_train_ids)}")
print(f"Total training samples: {full_train_data['X'].shape[0]}")

# Create best hyperparameters dict
best_hyperparams = {
    'lr_phase_1': best_trial.params['lr_phase_1'],
    'lr_phase_2': best_trial.params['lr_phase_2'],
    'lambda_physics': best_trial.params['lambda_physics'],
    'lambda_insulin': best_trial.params['lambda_insulin'],
    'lstm_units': best_trial.params['lstm_units'],
    'dense_units': best_trial.params['dense_units'],
    'batch_size': best_trial.params['batch_size'],
    'epochs_phase_1': best_trial.params['epochs_phase_1'],
    'epochs_phase_2': best_trial.params['epochs_phase_2'],
}

# Prepare test data for final validation
test_data_final = {
    'X': [], 'y': [], 'G0': [], 'bolus': [], 'carbs': []
}

for sid in test_subjects.keys():
    subject_data = test_subjects[sid]
    subject_sequences = test_sequences[sid]
    
    n_windows = subject_sequences['X'].shape[0]
    if n_windows == 0:
        continue
    
    test_data_final['X'].append(subject_sequences['X'])
    test_data_final['y'].append(subject_sequences['y'])
    
    G_initial = subject_data['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
    test_data_final['G0'].append(G_initial)
    
    for w in range(n_windows):
        start_idx = N_IN + w
        bolus_future = subject_data['bolus'].values[start_idx : start_idx + N_OUT]
        carbs_future = subject_data['carbs'].values[start_idx : start_idx + N_OUT]
        test_data_final['bolus'].append(bolus_future)
        test_data_final['carbs'].append(carbs_future)

test_data_final = {
    'X': np.concatenate(test_data_final['X'], axis=0).astype(np.float32),
    'y': np.concatenate(test_data_final['y'], axis=0).reshape(-1, N_OUT).astype(np.float32),
    'G0': np.concatenate(test_data_final['G0'], axis=0).astype(np.float32),
    'bolus': np.array(test_data_final['bolus'], dtype=np.float32),
    'carbs': np.array(test_data_final['carbs'], dtype=np.float32)
}

# Train final model
final_optimized_model, _ = train_pinn_model(
    full_train_data, 
    test_data_final,  # Use test as validation just for monitoring
    best_hyperparams, 
    verbose=True
)

# ============================================================================
# FINAL EVALUATION ON TEST SET
# ============================================================================

print("\n" + "="*70)
print("FINAL EVALUATION ON TEST SET (BrisT1D)")
print("="*70)

# Make predictions
test_predictions_scaled, _ = final_optimized_model(test_data_final['X'], training=False)
test_predictions_scaled = test_predictions_scaled.numpy()

test_predictions_mgdl = inverse_transform_cgm_np(test_predictions_scaled)
test_true_mgdl = inverse_transform_cgm_np(test_data_final['y'].flatten())

test_predictions_mgdl = test_predictions_mgdl.reshape(-1, N_OUT)
test_true_mgdl = test_true_mgdl.reshape(-1, N_OUT)

# Calculate metrics
test_rmse_optimized = np.sqrt(np.mean((test_predictions_mgdl - test_true_mgdl)**2))
test_mae_optimized = np.mean(np.abs(test_predictions_mgdl - test_true_mgdl))

print(f"\nOptimized Model Test Metrics:")
print(f"  RMSE: {test_rmse_optimized:.2f} mg/dL")
print(f"  MAE: {test_mae_optimized:.2f} mg/dL")

# Per-horizon metrics
print(f"\nPer-Horizon Performance:")
for h in range(N_OUT):
    horizon_min = (h + 1) * 5
    horizon_rmse = np.sqrt(np.mean((test_predictions_mgdl[:, h] - test_true_mgdl[:, h])**2))
    horizon_mae = np.mean(np.abs(test_predictions_mgdl[:, h] - test_true_mgdl[:, h]))
    print(f"  {horizon_min:2d} min: RMSE={horizon_rmse:.2f} mg/dL, MAE={horizon_mae:.2f} mg/dL")

print("\n" + "="*70)
print("HYPERPARAMETER OPTIMIZATION COMPLETE")
print("="*70)
print(f"""
Summary:
  ✓ Optimized {len(best_trial.params)} hyperparameters
  ✓ Completed {len(study.trials)} trials
  ✓ Best validation RMSE: {best_trial.value:.2f} mg/dL
  ✓ Test RMSE (BrisT1D): {test_rmse_optimized:.2f} mg/dL
  ✓ No data leakage: Test set never used in optimization
  
Variables available:
  - study: Optuna study object with all results
  - best_trial: Best trial from optimization
  - best_hyperparams: Dictionary of best hyperparameters
  - final_optimized_model: Model trained with best hyperparameters
  - test_predictions_mgdl: Final test predictions
  - test_true_mgdl: True test values
  
Next steps:
  - Use best_hyperparams for nested CV
  - Compare with baseline hyperparameters
  - Analyze parameter importance plots
""")

In [ ]:
"""
Physics-Informed Neural Network (PINN) for Blood Glucose Forecasting
Trains on OhioT1DM with Bergman minimal model constraints
Two-phase training: RMSE loss → Hybrid compound loss
FIXED: Numerical stability issues resolved
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# ============================================================================
# CONFIGURATION
# ============================================================================

# Training hyperparameters
EPOCHS_BERGMAN = 1
EPOCHS_PHASE_1 = 1  # RMSE-based training
EPOCHS_PHASE_2 = 1   # Hybrid loss fine-tuning
BATCH_SIZE = 64
LR_PHASE_1 = 0.0009
LR_PHASE_2 = 0.00009
LAMBDA_PHYSICS = 1.5
LAMBDA_INSULIN = 0.013
LEARN_BERGMAN_IN_PINN = True
RANDOM_SEED = 42

# Model architecture
LSTM_UNITS = 128
DENSE_UNITS = 32

# Set random seeds
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("="*70)
print("PHYSICS-INFORMED NEURAL NETWORK SETUP")
print("="*70)
print(f"Features: {MODEL_FEATURES}")
print(f"History: {N_IN} steps ({N_IN*5} min)")
print(f"Forecast: {N_OUT} steps ({N_OUT*5} min)")
print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

# ============================================================================
# HELPER FUNCTIONS WITH NUMERICAL STABILITY
# ============================================================================

def inverse_transform_cgm_tf(scaled_cgm):
    """Convert scaled CGM [0,1] to mg/dL (TensorFlow) with clipping"""
    cgm_min = tf.constant(float(cgm_scaler.data_min_[0]), dtype=tf.float32)
    cgm_max = tf.constant(float(cgm_scaler.data_max_[0]), dtype=tf.float32)
    result = scaled_cgm * (cgm_max - cgm_min) + cgm_min
    # Clip to reasonable glucose range
    return tf.clip_by_value(result, 20.0, 600.0)

def inverse_transform_cgm_np(scaled_cgm):
    """Convert scaled CGM [0,1] to mg/dL (NumPy) with clipping"""
    result = cgm_scaler.inverse_transform(scaled_cgm.reshape(-1, 1)).flatten()
    return np.clip(result, 20.0, 600.0)

def safe_softplus_inverse(value):
    """Compute inverse of softplus with numerical stability"""
    value = float(value)
    if value < 1e-7:
        return -10.0  # Small value maps to negative raw
    elif value > 20.0:
        return float(np.log(value))  # For large values, softplus ≈ x
    else:
        # Standard formula with safety check
        result = np.log(np.exp(value) - 1.0)
        return float(result) if np.isfinite(result) else float(np.log(value))

# ============================================================================
# BERGMAN MINIMAL MODEL WITH NUMERICAL STABILITY
# ============================================================================

class BergmanModel:
    """
    Bergman minimal model for glucose-insulin dynamics
    With enhanced numerical stability
    """
    def __init__(self, trainable=True, initial_params=None):
        # Initialize with reasonable defaults or provided values
        if initial_params is None:
            init_p1, init_p2, init_p3, init_ki, init_Gb = 0.01, 0.01, 0.01, 0.1, 100.0
        else:
            init_p1, init_p2, init_p3, init_ki, init_Gb = initial_params
        
        # Use safe initialization
        self.berg_p1_raw = tf.Variable(
            safe_softplus_inverse(init_p1), dtype=tf.float32, 
            trainable=trainable, name='berg_p1_raw'
        )
        self.berg_p2_raw = tf.Variable(
            safe_softplus_inverse(init_p2), dtype=tf.float32,
            trainable=trainable, name='berg_p2_raw'
        )
        self.berg_p3_raw = tf.Variable(
            safe_softplus_inverse(init_p3), dtype=tf.float32,
            trainable=trainable, name='berg_p3_raw'
        )
        self.berg_ki_raw = tf.Variable(
            safe_softplus_inverse(init_ki), dtype=tf.float32,
            trainable=trainable, name='berg_ki_raw'
        )
        self.berg_Gb_raw = tf.Variable(
            safe_softplus_inverse(init_Gb), dtype=tf.float32,
            trainable=trainable, name='berg_Gb_raw'
        )
    
    def get_positive_params(self):
        """Apply softplus to ensure all parameters are positive with bounds"""
        berg_p1 = tf.clip_by_value(tf.nn.softplus(self.berg_p1_raw), 1e-6, 0.1)
        berg_p2 = tf.clip_by_value(tf.nn.softplus(self.berg_p2_raw), 1e-6, 0.1)
        berg_p3 = tf.clip_by_value(tf.nn.softplus(self.berg_p3_raw), 1e-6, 0.5)
        berg_ki = tf.clip_by_value(tf.nn.softplus(self.berg_ki_raw), 0.01, 1.0)
        berg_Gb = tf.clip_by_value(tf.nn.softplus(self.berg_Gb_raw), 20.0, 500.0)
        return berg_p1, berg_p2, berg_p3, berg_ki, berg_Gb
    
    @property
    def trainable_variables(self):
        return [self.berg_p1_raw, self.berg_p2_raw, self.berg_p3_raw, 
                self.berg_ki_raw, self.berg_Gb_raw]
    
    @tf.function
    def simulate(self, G0, bolus_sequence, carbs_sequence):
        """
        Simulate glucose dynamics using Bergman equations with stability
        """
        berg_p1, berg_p2, berg_p3, berg_ki, berg_Gb = self.get_positive_params()
        
        batch_size = tf.shape(G0)[0]
        # Clip initial glucose to reasonable range
        G = tf.clip_by_value(tf.cast(G0, tf.float32), 50.0, 400.0)
        I = tf.zeros_like(G, dtype=tf.float32)
        
        glucose_trajectory = tf.TensorArray(tf.float32, size=N_OUT)
        exp_decay = tf.exp(-berg_ki)
        dt = 1.0  # Normalized timestep
        
        for t in tf.range(N_OUT):
            # Insulin dynamics (exponential decay + bolus)
            bolus_t = tf.clip_by_value(tf.cast(bolus_sequence[:, t], tf.float32), 0.0, 50.0)
            I = I * exp_decay + bolus_t
            I = tf.clip_by_value(I, 0.0, 100.0)
            
            # Glucose dynamics
            carbs_t = tf.clip_by_value(tf.cast(carbs_sequence[:, t], tf.float32), 0.0, 200.0)
            dG = -berg_p1 * (G - berg_Gb) - berg_p2 * I + berg_p3 * carbs_t
            
            # Clip rate of change to prevent instability
            dG = tf.clip_by_value(dG, -50.0, 50.0)
            
            G = G + dG * dt
            # Keep glucose in physiological range
            G = tf.clip_by_value(G, 40.0, 500.0)
            
            glucose_trajectory = glucose_trajectory.write(t, G)
        
        return tf.transpose(glucose_trajectory.stack(), perm=[1, 0])

# ============================================================================
# PRE-TRAIN BERGMAN MODEL ON TRAINING DATA
# ============================================================================

print("\n" + "="*70)
print("STEP 1: Pre-training Bergman Model")
print("="*70)

# Prepare data for Bergman pre-training
bergman_train_data = []
for sid in train_ids:
    subject_data = train_subjects[sid]
    subject_sequences = train_sequences[sid]
    
    n_windows = subject_sequences['X'].shape[0]
    if n_windows == 0:
        continue
    
    # Get initial glucose values (last point of history window)
    G_initial = subject_data['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
    
    # Get future bolus and carbs
    for w in range(n_windows):
        start_idx = N_IN + w
        bolus_future = subject_data['bolus'].values[start_idx : start_idx + N_OUT]
        carbs_future = subject_data['carbs'].values[start_idx : start_idx + N_OUT]
        
        # Get true glucose values
        y_scaled = subject_sequences['y'][w].flatten()
        y_mgdl = cgm_scaler.inverse_transform(y_scaled.reshape(-1, 1)).flatten()
        
        bergman_train_data.append({
            'G0': G_initial[w],
            'bolus': bolus_future,
            'carbs': carbs_future,
            'y_true': y_mgdl
        })

if len(bergman_train_data) > 0:
    G0_array = np.array([d['G0'] for d in bergman_train_data], dtype=np.float32)
    bolus_array = np.array([d['bolus'] for d in bergman_train_data], dtype=np.float32)
    carbs_array = np.array([d['carbs'] for d in bergman_train_data], dtype=np.float32)
    y_true_array = np.array([d['y_true'] for d in bergman_train_data], dtype=np.float32)
    
    # Clean data - remove any NaN or extreme values
    valid_mask = (
        np.isfinite(G0_array) & 
        np.all(np.isfinite(y_true_array), axis=1) &
        (G0_array > 40) & (G0_array < 400)
    )
    G0_array = G0_array[valid_mask]
    bolus_array = bolus_array[valid_mask]
    carbs_array = carbs_array[valid_mask]
    y_true_array = y_true_array[valid_mask]
    
    # Initialize with data statistics
    median_G = np.clip(np.median(G0_array), 80.0, 150.0)
    
    print(f"Training Bergman model on {len(G0_array)} valid windows...")
    print(f"Initial baseline glucose (Gb): {median_G:.1f} mg/dL")
    
    # Create Bergman model with reasonable initialization
    initial_params = (0.02, 0.015, 0.03, 0.15, median_G)
    bergman = BergmanModel(trainable=True, initial_params=initial_params)
    
    # Verify initialization
    test_p1, test_p2, test_p3, test_ki, test_Gb = bergman.get_positive_params()
    print(f"Initial parameters: p1={float(test_p1):.5f}, p2={float(test_p2):.5f}, "
          f"p3={float(test_p3):.5f}, ki={float(test_ki):.5f}, Gb={float(test_Gb):.1f}")
    
    # Optimizer for Bergman pre-training
    bergman_optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
    
    # Create dataset
    bergman_dataset = tf.data.Dataset.from_tensor_slices((
        G0_array, bolus_array, carbs_array, y_true_array
    )).shuffle(10000, seed=RANDOM_SEED).batch(BATCH_SIZE).prefetch(2)
    
    # Pre-train Bergman model
    for epoch in range(EPOCHS_BERGMAN):
        epoch_loss = 0.0
        n_batches = 0
        nan_count = 0
        
        for G0_batch, bolus_batch, carbs_batch, y_batch in bergman_dataset:
            with tf.GradientTape() as tape:
                G_pred = bergman.simulate(G0_batch, bolus_batch, carbs_batch)
                loss = tf.reduce_mean(tf.square(G_pred - y_batch))
                
                # Check for NaN
                if not tf.math.is_finite(loss):
                    nan_count += 1
                    continue
            
            grads = tape.gradient(loss, bergman.trainable_variables)
            
            # Check gradients for NaN
            if any(tf.reduce_any(~tf.math.is_finite(g)) for g in grads if g is not None):
                nan_count += 1
                continue
            
            # Clip gradients
            grads, _ = tf.clip_by_global_norm(grads, 1.0)
            bergman_optimizer.apply_gradients(zip(grads, bergman.trainable_variables))
            
            epoch_loss += float(loss.numpy())
            n_batches += 1
        
        if n_batches > 0 and ((epoch + 1) % 10 == 0 or epoch == 0):
            avg_loss = epoch_loss / n_batches
            berg_p1, berg_p2, berg_p3, berg_ki, berg_Gb = bergman.get_positive_params()
            print(f"Epoch {epoch+1}/{EPOCHS_BERGMAN}, Loss: {avg_loss:.2f} mg²/dL²" +
                  (f" (NaN batches: {nan_count})" if nan_count > 0 else ""))
            print(f"  Parameters: p1={float(berg_p1):.5f}, p2={float(berg_p2):.5f}, "
                  f"p3={float(berg_p3):.5f}, ki={float(berg_ki):.5f}, Gb={float(berg_Gb):.1f}")
    
    berg_p1_final, berg_p2_final, berg_p3_final, berg_ki_final, berg_Gb_final = bergman.get_positive_params()
    print(f"\nFinal Bergman parameters:")
    print(f"  p1={float(berg_p1_final):.5f}, p2={float(berg_p2_final):.5f}, "
          f"p3={float(berg_p3_final):.5f}, ki={float(berg_ki_final):.5f}, Gb={float(berg_Gb_final):.1f}")
else:
    print("No training data available for Bergman model")
    bergman = BergmanModel(trainable=True)

# ============================================================================
# PINN MODEL ARCHITECTURE
# ============================================================================

class PINNModel(tf.keras.Model):
    """
    Physics-Informed Neural Network combining LSTM with Bergman dynamics
    """
    def __init__(self, n_features, bergman_model, learn_bergman=True):
        super().__init__()
        
        # Neural network components
        self.lstm_encoder = layers.LSTM(LSTM_UNITS, return_sequences=False)
        self.dense_1 = layers.Dense(DENSE_UNITS, activation='relu')
        self.output_layer = layers.Dense(N_OUT * 2, activation='linear')
        
        # Bergman model (physics component)
        self.bergman = bergman_model
        self.learn_bergman = learn_bergman
    
    def call(self, inputs, training=False):
        """
        Forward pass with numerical stability
        """
        x = tf.cast(inputs, tf.float32)
        
        # LSTM encoding
        encoded = self.lstm_encoder(x)
        hidden = self.dense_1(encoded)
        output = self.output_layer(hidden)
        
        # Split output into glucose and insulin predictions
        batch_size = tf.shape(output)[0]
        reshaped = tf.reshape(output, (batch_size, N_OUT, 2))
        
        # Sigmoid activation for glucose (keeps in [0,1] range)
        glucose_pred = tf.nn.sigmoid(reshaped[:, :, 0])
        # Tanh for insulin (keeps bounded)
        insulin_pred = tf.nn.tanh(reshaped[:, :, 1])
        
        return glucose_pred, insulin_pred
    
    @tf.function
    def compute_insulin_effect(self, bolus_sequence):
        """Compute insulin effect from bolus using Bergman kinetics"""
        _, _, _, berg_ki, _ = self.bergman.get_positive_params()
        
        batch_size = tf.shape(bolus_sequence)[0]
        I = tf.zeros((batch_size,), dtype=tf.float32)
        insulin_trajectory = tf.TensorArray(tf.float32, size=N_OUT)
        exp_decay = tf.exp(-berg_ki)
        
        for t in tf.range(N_OUT):
            bolus_t = tf.clip_by_value(tf.cast(bolus_sequence[:, t], tf.float32), 0.0, 50.0)
            I = I * exp_decay + bolus_t
            I = tf.clip_by_value(I, 0.0, 100.0)
            insulin_trajectory = insulin_trajectory.write(t, I)
        
        return tf.transpose(insulin_trajectory.stack(), perm=[1, 0])

# ============================================================================
# LOSS FUNCTIONS WITH NUMERICAL STABILITY
# ============================================================================

def safe_compound_glucose_loss(y_true_scaled, y_pred_scaled):
    """
    Compound loss with extensive numerical stability checks
    """
    eps = 1e-7
    
    # Convert to mg/dL with clipping
    y_true = inverse_transform_cgm_tf(tf.clip_by_value(y_true_scaled, 0.0, 1.0))
    y_pred = inverse_transform_cgm_tf(tf.clip_by_value(y_pred_scaled, 0.0, 1.0))
    
    # RMSE component (normalized)
    squared_diff = tf.square(y_true - y_pred)
    rmse = tf.sqrt(tf.reduce_mean(squared_diff) + eps) / 100.0
    
    # Temporal dynamics penalty
    dt_true = y_true[:, 1:] - y_true[:, :-1]
    dt_pred = y_pred[:, 1:] - y_pred[:, :-1]
    temporal_penalty = tf.reduce_mean(tf.square(dt_pred - dt_true)) / 1000.0
    
    # Soft glycemic zone classification
    hypo_threshold = 70.0
    hyper_threshold = 180.0
    
    # Soft probabilities
    p_hypo_true = tf.nn.sigmoid(-(y_true - hypo_threshold) / 10.0)
    p_hyper_true = tf.nn.sigmoid((y_true - hyper_threshold) / 10.0)
    p_normal_true = 1.0 - p_hypo_true - p_hyper_true
    
    p_hypo_pred = tf.nn.sigmoid(-(y_pred - hypo_threshold) / 10.0)
    p_hyper_pred = tf.nn.sigmoid((y_pred - hyper_threshold) / 10.0)
    p_normal_pred = 1.0 - p_hypo_pred - p_hyper_pred
    
    # Zone-wise alignment with safety
    recall_hypo = tf.clip_by_value(tf.reduce_mean(p_hypo_pred * p_hypo_true), eps, 1.0)
    recall_normal = tf.clip_by_value(tf.reduce_mean(p_normal_pred * p_normal_true), eps, 1.0)
    recall_hyper = tf.clip_by_value(tf.reduce_mean(p_hyper_pred * p_hyper_true), eps, 1.0)
    
    # Geometric mean (safe)
    g_mean = tf.exp((tf.math.log(recall_hypo) + tf.math.log(recall_normal) + tf.math.log(recall_hyper)) / 3.0)
    
    # Combine components
    compound_loss = rmse + temporal_penalty + (1.0 - g_mean)
    
    # Final safety check
    return tf.where(tf.math.is_finite(compound_loss), compound_loss, tf.constant(1e6, dtype=tf.float32))

@tf.function
def compute_pinn_losses(X_batch, y_batch, G0_batch, bolus_batch, carbs_batch, 
                        model, lambda_phys, lambda_insulin, data_loss_fn):
    """
    Compute all PINN loss components with NaN protection
    """
    eps = 1e-7
    
    # Neural network predictions
    G_pred_scaled, I_pred = model(X_batch, training=True)
    
    # Data loss (prediction vs ground truth)
    data_loss = data_loss_fn(y_batch, G_pred_scaled)
    
    # Physics loss (prediction vs Bergman simulation)
    G_bergman = model.bergman.simulate(G0_batch, bolus_batch, carbs_batch)
    G_pred_mgdl = inverse_transform_cgm_tf(G_pred_scaled)
    
    cgm_range = float(cgm_scaler.data_max_[0] - cgm_scaler.data_min_[0])
    physics_loss = tf.reduce_mean(tf.square(G_pred_mgdl - G_bergman)) / (cgm_range**2 + eps)
    
    # Insulin consistency loss
    I_expected = model.compute_insulin_effect(bolus_batch)
    insulin_loss = tf.reduce_mean(tf.square(I_pred - I_expected))
    
    # Check each component
    data_loss = tf.where(tf.math.is_finite(data_loss), data_loss, tf.constant(1e6, dtype=tf.float32))
    physics_loss = tf.where(tf.math.is_finite(physics_loss), physics_loss, tf.constant(1e6, dtype=tf.float32))
    insulin_loss = tf.where(tf.math.is_finite(insulin_loss), insulin_loss, tf.constant(1e6, dtype=tf.float32))
    
    # Total loss
    total_loss = data_loss + lambda_phys * physics_loss + lambda_insulin * insulin_loss
    
    return total_loss, data_loss, physics_loss, insulin_loss

# ============================================================================
# PREPARE TRAINING DATA FOR PINN
# ============================================================================

print("\n" + "="*70)
print("STEP 2: Preparing PINN Training Data")
print("="*70)

# Extract bolus, carbs, and initial glucose for physics constraints
pinn_train_data = {
    'X': [], 'y': [], 'G0': [], 'bolus': [], 'carbs': []
}

for sid in train_ids:
    subject_data = train_subjects[sid]
    subject_sequences = train_sequences[sid]
    
    n_windows = subject_sequences['X'].shape[0]
    if n_windows == 0:
        continue
    
    pinn_train_data['X'].append(subject_sequences['X'])
    pinn_train_data['y'].append(subject_sequences['y'])
    
    G_initial = subject_data['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
    pinn_train_data['G0'].append(G_initial)
    
    for w in range(n_windows):
        start_idx = N_IN + w
        bolus_future = subject_data['bolus'].values[start_idx : start_idx + N_OUT]
        carbs_future = subject_data['carbs'].values[start_idx : start_idx + N_OUT]
        pinn_train_data['bolus'].append(bolus_future)
        pinn_train_data['carbs'].append(carbs_future)

# Concatenate all data
X_pinn = np.concatenate(pinn_train_data['X'], axis=0).astype(np.float32)
y_pinn = np.concatenate(pinn_train_data['y'], axis=0).reshape(-1, N_OUT).astype(np.float32)
G0_pinn = np.concatenate(pinn_train_data['G0'], axis=0).astype(np.float32)
bolus_pinn = np.array(pinn_train_data['bolus'], dtype=np.float32)
carbs_pinn = np.array(pinn_train_data['carbs'], dtype=np.float32)

print(f"PINN training data shapes:")
print(f"  X: {X_pinn.shape}")
print(f"  y: {y_pinn.shape}")
print(f"  G0: {G0_pinn.shape}")
print(f"  Bolus: {bolus_pinn.shape}")
print(f"  Carbs: {carbs_pinn.shape}")

# Create TensorFlow dataset
pinn_dataset = tf.data.Dataset.from_tensor_slices((
    X_pinn, y_pinn, G0_pinn, bolus_pinn, carbs_pinn
)).shuffle(10000, seed=RANDOM_SEED).batch(BATCH_SIZE).prefetch(2)

# ============================================================================
# INITIALIZE PINN MODEL
# ============================================================================

n_features = X_train.shape[2]
pinn_model = PINNModel(n_features, bergman, learn_bergman=LEARN_BERGMAN_IN_PINN)

# Build model
pinn_model.build((None, N_IN, n_features))
print(f"\nPINN model initialized with {n_features} input features")

# ============================================================================
# PHASE 1: TRAINING WITH RMSE LOSS
# ============================================================================

print("\n" + "="*70)
print("PHASE 1: Training with RMSE Loss")
print("="*70)

optimizer_phase1 = tf.keras.optimizers.Adam(learning_rate=LR_PHASE_1)
rmse_loss_fn = lambda y_true, y_pred: tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-7)

# Warmup schedule for physics weight
warmup_epochs = max(1, int(0.3 * EPOCHS_PHASE_1))

for epoch in range(EPOCHS_PHASE_1):
    # Gradually increase physics weight during warmup
    physics_weight = LAMBDA_PHYSICS * min(1.0, (epoch + 1) / warmup_epochs)
    
    epoch_loss = 0.0
    epoch_data_loss = 0.0
    epoch_phys_loss = 0.0
    n_batches = 0
    nan_batches = 0
    
    for X_batch, y_batch, G0_batch, bolus_batch, carbs_batch in pinn_dataset:
        with tf.GradientTape() as tape:
            total_loss, data_loss, phys_loss, ins_loss = compute_pinn_losses(
                X_batch, y_batch, G0_batch, bolus_batch, carbs_batch,
                pinn_model, physics_weight, LAMBDA_INSULIN, rmse_loss_fn
            )
            
            # Skip if NaN
            if not tf.math.is_finite(total_loss):
                nan_batches += 1
                continue
        
        grads = tape.gradient(total_loss, pinn_model.trainable_variables)
        
        # Check gradients
        if any(g is not None and tf.reduce_any(~tf.math.is_finite(g)) for g in grads):
            nan_batches += 1
            continue
        
        grads, _ = tf.clip_by_global_norm(grads, 5.0)
        optimizer_phase1.apply_gradients(zip(grads, pinn_model.trainable_variables))
        
        epoch_loss += float(total_loss.numpy())
        epoch_data_loss += float(data_loss.numpy())
        epoch_phys_loss += float(phys_loss.numpy())
        n_batches += 1
    
    if n_batches > 0 and ((epoch + 1) % 10 == 0 or epoch == 0):
        avg_total = epoch_loss / n_batches
        avg_data = epoch_data_loss / n_batches
        avg_phys = epoch_phys_loss / n_batches
        print(f"Epoch {epoch+1}/{EPOCHS_PHASE_1}: Loss={avg_total:.4f} "
              f"(Data={avg_data:.4f}, Physics={avg_phys:.4f}, λ_phys={physics_weight:.3f})" +
              (f" [NaN: {nan_batches}]" if nan_batches > 0 else ""))

# ============================================================================
# PHASE 2: FINE-TUNING WITH COMPOUND LOSS
# ============================================================================

print("\n" + "="*70)
print("PHASE 2: Fine-Tuning with Compound Loss")
print("="*70)

optimizer_phase2 = tf.keras.optimizers.Adam(learning_rate=LR_PHASE_2)

for epoch in range(EPOCHS_PHASE_2):
    epoch_loss = 0.0
    epoch_data_loss = 0.0
    epoch_phys_loss = 0.0
    n_batches = 0
    nan_batches = 0
    
    for X_batch, y_batch, G0_batch, bolus_batch, carbs_batch in pinn_dataset:
        with tf.GradientTape() as tape:
            total_loss, data_loss, phys_loss, ins_loss = compute_pinn_losses(
                X_batch, y_batch, G0_batch, bolus_batch, carbs_batch,
                pinn_model, LAMBDA_PHYSICS, LAMBDA_INSULIN, safe_compound_glucose_loss
            )
            
            if not tf.math.is_finite(total_loss):
                nan_batches += 1
                continue
        
        grads = tape.gradient(total_loss, pinn_model.trainable_variables)
        
        if any(g is not None and tf.reduce_any(~tf.math.is_finite(g)) for g in grads):
            nan_batches += 1
            continue
        
        grads, _ = tf.clip_by_global_norm(grads, 5.0)
        optimizer_phase2.apply_gradients(zip(grads, pinn_model.trainable_variables))
        
        epoch_loss += float(total_loss.numpy())
        epoch_data_loss += float(data_loss.numpy())
        epoch_phys_loss += float(phys_loss.numpy())
        n_batches += 1
    
    if n_batches > 0 and ((epoch + 1) % 5 == 0 or epoch == 0):
        avg_total = epoch_loss / n_batches
        avg_data = epoch_data_loss / n_batches
        avg_phys = epoch_phys_loss / n_batches
        print(f"Epoch {epoch+1}/{EPOCHS_PHASE_2}: Loss={avg_total:.4f} "
              f"(Compound={avg_data:.4f}, Physics={avg_phys:.4f})" +
              (f" [NaN: {nan_batches}]" if nan_batches > 0 else ""))

# Print final Bergman parameters
if LEARN_BERGMAN_IN_PINN:
    berg_p1, berg_p2, berg_p3, berg_ki, berg_Gb = pinn_model.bergman.get_positive_params()
    print(f"\nFinal learned Bergman parameters:")
    print(f"  p1={float(berg_p1):.5f}, p2={float(berg_p2):.5f}, "
          f"p3={float(berg_p3):.5f}, ki={float(berg_ki):.5f}, Gb={float(berg_Gb):.1f}")

# ============================================================================
# VALIDATION EVALUATION
# ============================================================================

print("\n" + "="*70)
print("VALIDATION EVALUATION")
print("="*70)

# Make predictions on validation data
val_predictions_scaled, _ = pinn_model(X_val.astype(np.float32), training=False)
val_predictions_scaled = val_predictions_scaled.numpy()

# Inverse transform to get mg/dL values
val_predictions_mgdl = inverse_transform_cgm_np(val_predictions_scaled)
val_true_mgdl = inverse_transform_cgm_np(y_val.flatten())

# Reshape for proper comparison
val_predictions_mgdl = val_predictions_mgdl.reshape(-1, N_OUT)
val_true_mgdl = val_true_mgdl.reshape(-1, N_OUT)

# Calculate validation metrics
val_rmse = np.sqrt(np.mean((val_predictions_mgdl - val_true_mgdl)**2))
val_mae = np.mean(np.abs(val_predictions_mgdl - val_true_mgdl))

print(f"\nValidation Metrics:")
print(f"  RMSE: {val_rmse:.2f} mg/dL")
print(f"  MAE: {val_mae:.2f} mg/dL")

# Per-horizon metrics
#for h in range(N_OUT):
    #horizon_min = (h + 1) * 5
    #horizon_rmse = np.sqrt(np.mean((val_predictions_mgdl[:, h] - val_true_mgdl[:, h])**2))
    #horizon_mae = np.mean(np.abs(val_predictions_mgdl[:, h] - val_true_mgdl[:, h]))
    #print(f"  Horizon {horizon_min} min: RMSE={horizon_rmse:.2f}, MAE={horizon_mae:.2f}")

# ============================================================================
# VISUALIZATION
# ============================================================================

# Plot some validation predictions
#n_examples = min(5, val_predictions_mgdl.shape[0])
#time_horizons = np.arange(1, N_OUT + 1) * 5

#fig, axes = plt.subplots(n_examples, 1, figsize=(12, 3*n_examples))
#if n_examples == 1:
#    axes = [axes]

#for i in range(n_examples):
    #axes[i].plot(time_horizons, val_true_mgdl[i], 'o-', label='True', color='blue')
    #axes[i].plot(time_horizons, val_predictions_mgdl[i], 's-', label='Predicted', color='red')
    #axes[i].axhline(y=70, color='orange', linestyle='--', alpha=0.5, label='Hypo threshold')
    #axes[i].axhline(y=180, color='orange', linestyle='--', alpha=0.5, label='Hyper threshold')
    #axes[i].set_xlabel('Time Horizon (minutes)')
    #axes[i].set_ylabel('Blood Glucose (mg/dL)')
    #axes[i].set_title(f'Validation Example {i+1}')
    #axes[i].legend()
    #axes[i].grid(True, alpha=0.3)

#plt.tight_layout()
#plt.show()

print("\n" + "="*70)
print("PINN TRAINING COMPLETE")
print("="*70)
print("Model is ready for evaluation on test data (BrisT1D dataset)")

## Hyperparameter tuning - PINN - Optuna:

In [ ]:
"""
Physics-Informed Neural Network (PINN) for Blood Glucose Forecasting
With Nested Cross-Validation and Optuna Hyperparameter Tuning
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import optuna
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

# ============================================================================
# CONFIGURATION
# ============================================================================

RANDOM_SEED = 42
N_OUTER_FOLDS = 3  # Outer CV for model evaluation
N_INNER_FOLDS = 2  # Inner CV for hyperparameter tuning
N_OPTUNA_TRIALS = 20

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("="*70)
print("PINN WITH NESTED CV AND OPTUNA")
print("="*70)
print(f"Features: {MODEL_FEATURES}")
print(f"History: {N_IN} steps, Forecast: {N_OUT} steps")
print(f"Total samples: {X_train.shape[0]}")

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def inverse_transform_cgm_tf(scaled_cgm):
    cgm_min = tf.constant(float(cgm_scaler.data_min_[0]), dtype=tf.float32)
    cgm_max = tf.constant(float(cgm_scaler.data_max_[0]), dtype=tf.float32)
    result = scaled_cgm * (cgm_max - cgm_min) + cgm_min
    return tf.clip_by_value(result, 20.0, 600.0)

def inverse_transform_cgm_np(scaled_cgm):
    result = cgm_scaler.inverse_transform(scaled_cgm.reshape(-1, 1)).flatten()
    return np.clip(result, 20.0, 600.0)

def safe_softplus_inverse(value):
    value = float(value)
    if value < 1e-7:
        return -10.0
    elif value > 20.0:
        return float(np.log(value))
    else:
        result = np.log(np.exp(value) - 1.0)
        return float(result) if np.isfinite(result) else float(np.log(value))

# ============================================================================
# BERGMAN MODEL
# ============================================================================

class BergmanModel:
    def __init__(self, trainable=True, initial_params=None):
        if initial_params is None:
            init_p1, init_p2, init_p3, init_ki, init_Gb = 0.01, 0.01, 0.01, 0.1, 100.0
        else:
            init_p1, init_p2, init_p3, init_ki, init_Gb = initial_params
        
        self.berg_p1_raw = tf.Variable(safe_softplus_inverse(init_p1), dtype=tf.float32, trainable=trainable, name='berg_p1_raw')
        self.berg_p2_raw = tf.Variable(safe_softplus_inverse(init_p2), dtype=tf.float32, trainable=trainable, name='berg_p2_raw')
        self.berg_p3_raw = tf.Variable(safe_softplus_inverse(init_p3), dtype=tf.float32, trainable=trainable, name='berg_p3_raw')
        self.berg_ki_raw = tf.Variable(safe_softplus_inverse(init_ki), dtype=tf.float32, trainable=trainable, name='berg_ki_raw')
        self.berg_Gb_raw = tf.Variable(safe_softplus_inverse(init_Gb), dtype=tf.float32, trainable=trainable, name='berg_Gb_raw')
    
    def get_positive_params(self):
        berg_p1 = tf.clip_by_value(tf.nn.softplus(self.berg_p1_raw), 1e-6, 0.1)
        berg_p2 = tf.clip_by_value(tf.nn.softplus(self.berg_p2_raw), 1e-6, 0.1)
        berg_p3 = tf.clip_by_value(tf.nn.softplus(self.berg_p3_raw), 1e-6, 0.5)
        berg_ki = tf.clip_by_value(tf.nn.softplus(self.berg_ki_raw), 0.01, 1.0)
        berg_Gb = tf.clip_by_value(tf.nn.softplus(self.berg_Gb_raw), 20.0, 500.0)
        return berg_p1, berg_p2, berg_p3, berg_ki, berg_Gb
    
    @property
    def trainable_variables(self):
        return [self.berg_p1_raw, self.berg_p2_raw, self.berg_p3_raw, self.berg_ki_raw, self.berg_Gb_raw]
    
    @tf.function
    def simulate(self, G0, bolus_sequence, carbs_sequence):
        berg_p1, berg_p2, berg_p3, berg_ki, berg_Gb = self.get_positive_params()
        batch_size = tf.shape(G0)[0]
        G = tf.clip_by_value(tf.cast(G0, tf.float32), 50.0, 400.0)
        I = tf.zeros_like(G, dtype=tf.float32)
        glucose_trajectory = tf.TensorArray(tf.float32, size=N_OUT)
        exp_decay = tf.exp(-berg_ki)
        dt = 1.0
        
        for t in tf.range(N_OUT):
            bolus_t = tf.clip_by_value(tf.cast(bolus_sequence[:, t], tf.float32), 0.0, 50.0)
            I = I * exp_decay + bolus_t
            I = tf.clip_by_value(I, 0.0, 100.0)
            carbs_t = tf.clip_by_value(tf.cast(carbs_sequence[:, t], tf.float32), 0.0, 200.0)
            dG = -berg_p1 * (G - berg_Gb) - berg_p2 * I + berg_p3 * carbs_t
            dG = tf.clip_by_value(dG, -50.0, 50.0)
            G = G + dG * dt
            G = tf.clip_by_value(G, 40.0, 500.0)
            glucose_trajectory = glucose_trajectory.write(t, G)
        
        return tf.transpose(glucose_trajectory.stack(), perm=[1, 0])

# ============================================================================
# PINN MODEL
# ============================================================================

class PINNModel(tf.keras.Model):
    def __init__(self, n_features, bergman_model, lstm_units, dense_units, learn_bergman=True):
        super().__init__()
        self.lstm_encoder = layers.LSTM(lstm_units, return_sequences=False)
        self.dense_1 = layers.Dense(dense_units, activation='relu')
        self.output_layer = layers.Dense(N_OUT * 2, activation='linear')
        self.bergman = bergman_model
        self.learn_bergman = learn_bergman
    
    def call(self, inputs, training=False):
        x = tf.cast(inputs, tf.float32)
        encoded = self.lstm_encoder(x)
        hidden = self.dense_1(encoded)
        output = self.output_layer(hidden)
        batch_size = tf.shape(output)[0]
        reshaped = tf.reshape(output, (batch_size, N_OUT, 2))
        glucose_pred = tf.nn.sigmoid(reshaped[:, :, 0])
        insulin_pred = tf.nn.tanh(reshaped[:, :, 1])
        return glucose_pred, insulin_pred
    
    @tf.function
    def compute_insulin_effect(self, bolus_sequence):
        _, _, _, berg_ki, _ = self.bergman.get_positive_params()
        batch_size = tf.shape(bolus_sequence)[0]
        I = tf.zeros((batch_size,), dtype=tf.float32)
        insulin_trajectory = tf.TensorArray(tf.float32, size=N_OUT)
        exp_decay = tf.exp(-berg_ki)
        
        for t in tf.range(N_OUT):
            bolus_t = tf.clip_by_value(tf.cast(bolus_sequence[:, t], tf.float32), 0.0, 50.0)
            I = I * exp_decay + bolus_t
            I = tf.clip_by_value(I, 0.0, 100.0)
            insulin_trajectory = insulin_trajectory.write(t, I)
        
        return tf.transpose(insulin_trajectory.stack(), perm=[1, 0])

# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

def safe_compound_glucose_loss(y_true_scaled, y_pred_scaled):
    eps = 1e-7
    y_true = inverse_transform_cgm_tf(tf.clip_by_value(y_true_scaled, 0.0, 1.0))
    y_pred = inverse_transform_cgm_tf(tf.clip_by_value(y_pred_scaled, 0.0, 1.0))
    
    squared_diff = tf.square(y_true - y_pred)
    rmse = tf.sqrt(tf.reduce_mean(squared_diff) + eps) / 100.0
    
    dt_true = y_true[:, 1:] - y_true[:, :-1]
    dt_pred = y_pred[:, 1:] - y_pred[:, :-1]
    temporal_penalty = tf.reduce_mean(tf.square(dt_pred - dt_true)) / 1000.0
    
    hypo_threshold, hyper_threshold = 70.0, 180.0
    p_hypo_true = tf.nn.sigmoid(-(y_true - hypo_threshold) / 10.0)
    p_hyper_true = tf.nn.sigmoid((y_true - hyper_threshold) / 10.0)
    p_normal_true = 1.0 - p_hypo_true - p_hyper_true
    p_hypo_pred = tf.nn.sigmoid(-(y_pred - hypo_threshold) / 10.0)
    p_hyper_pred = tf.nn.sigmoid((y_pred - hyper_threshold) / 10.0)
    p_normal_pred = 1.0 - p_hypo_pred - p_hyper_pred
    
    recall_hypo = tf.clip_by_value(tf.reduce_mean(p_hypo_pred * p_hypo_true), eps, 1.0)
    recall_normal = tf.clip_by_value(tf.reduce_mean(p_normal_pred * p_normal_true), eps, 1.0)
    recall_hyper = tf.clip_by_value(tf.reduce_mean(p_hyper_pred * p_hyper_true), eps, 1.0)
    g_mean = tf.exp((tf.math.log(recall_hypo) + tf.math.log(recall_normal) + tf.math.log(recall_hyper)) / 3.0)
    
    compound_loss = rmse + temporal_penalty + (1.0 - g_mean)
    return tf.where(tf.math.is_finite(compound_loss), compound_loss, tf.constant(1e6, dtype=tf.float32))

@tf.function
def compute_pinn_losses(X_batch, y_batch, G0_batch, bolus_batch, carbs_batch, 
                        model, lambda_phys, lambda_insulin, data_loss_fn):
    eps = 1e-7
    G_pred_scaled, I_pred = model(X_batch, training=True)
    data_loss = data_loss_fn(y_batch, G_pred_scaled)
    
    G_bergman = model.bergman.simulate(G0_batch, bolus_batch, carbs_batch)
    G_pred_mgdl = inverse_transform_cgm_tf(G_pred_scaled)
    cgm_range = float(cgm_scaler.data_max_[0] - cgm_scaler.data_min_[0])
    physics_loss = tf.reduce_mean(tf.square(G_pred_mgdl - G_bergman)) / (cgm_range**2 + eps)
    
    I_expected = model.compute_insulin_effect(bolus_batch)
    insulin_loss = tf.reduce_mean(tf.square(I_pred - I_expected))
    
    data_loss = tf.where(tf.math.is_finite(data_loss), data_loss, tf.constant(1e6, dtype=tf.float32))
    physics_loss = tf.where(tf.math.is_finite(physics_loss), physics_loss, tf.constant(1e6, dtype=tf.float32))
    insulin_loss = tf.where(tf.math.is_finite(insulin_loss), insulin_loss, tf.constant(1e6, dtype=tf.float32))
    
    total_loss = data_loss + lambda_phys * physics_loss + lambda_insulin * insulin_loss
    return total_loss, data_loss, physics_loss, insulin_loss

# ============================================================================
# DATA PREPARATION
# ============================================================================

def prepare_pinn_data(subject_ids, subjects_dict, sequences_dict):
    pinn_data = {'X': [], 'y': [], 'G0': [], 'bolus': [], 'carbs': []}
    
    for sid in subject_ids:
        subject_data = subjects_dict[sid]
        subject_sequences = sequences_dict[sid]
        n_windows = subject_sequences['X'].shape[0]
        
        if n_windows == 0:
            continue
        
        pinn_data['X'].append(subject_sequences['X'])
        pinn_data['y'].append(subject_sequences['y'])
        G_initial = subject_data['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
        pinn_data['G0'].append(G_initial)
        
        for w in range(n_windows):
            start_idx = N_IN + w
            bolus_future = subject_data['bolus'].values[start_idx : start_idx + N_OUT]
            carbs_future = subject_data['carbs'].values[start_idx : start_idx + N_OUT]
            pinn_data['bolus'].append(bolus_future)
            pinn_data['carbs'].append(carbs_future)
    
    X = np.concatenate(pinn_data['X'], axis=0).astype(np.float32)
    y = np.concatenate(pinn_data['y'], axis=0).reshape(-1, N_OUT).astype(np.float32)
    G0 = np.concatenate(pinn_data['G0'], axis=0).astype(np.float32)
    bolus = np.array(pinn_data['bolus'], dtype=np.float32)
    carbs = np.array(pinn_data['carbs'], dtype=np.float32)
    
    return X, y, G0, bolus, carbs

# ============================================================================
# BERGMAN TRAINING
# ============================================================================

def train_bergman_model(G0_array, bolus_array, carbs_array, y_true_array, epochs=50):
    valid_mask = (np.isfinite(G0_array) & np.all(np.isfinite(y_true_array), axis=1) & 
                  (G0_array > 40) & (G0_array < 400))
    G0_array = G0_array[valid_mask]
    bolus_array = bolus_array[valid_mask]
    carbs_array = carbs_array[valid_mask]
    y_true_array = y_true_array[valid_mask]
    
    median_G = np.clip(np.median(G0_array), 80.0, 150.0)
    initial_params = (0.02, 0.015, 0.03, 0.15, median_G)
    bergman = BergmanModel(trainable=True, initial_params=initial_params)
    bergman_optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
    
    bergman_dataset = tf.data.Dataset.from_tensor_slices((
        G0_array, bolus_array, carbs_array, y_true_array
    )).shuffle(10000, seed=RANDOM_SEED).batch(64).prefetch(2)
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        n_batches = 0
        
        for G0_batch, bolus_batch, carbs_batch, y_batch in bergman_dataset:
            with tf.GradientTape() as tape:
                G_pred = bergman.simulate(G0_batch, bolus_batch, carbs_batch)
                loss = tf.reduce_mean(tf.square(G_pred - y_batch))
                if not tf.math.is_finite(loss):
                    continue
            
            grads = tape.gradient(loss, bergman.trainable_variables)
            if any(tf.reduce_any(~tf.math.is_finite(g)) for g in grads if g is not None):
                continue
            
            grads, _ = tf.clip_by_global_norm(grads, 1.0)
            bergman_optimizer.apply_gradients(zip(grads, bergman.trainable_variables))
            epoch_loss += float(loss.numpy())
            n_batches += 1
    
    return bergman

# ============================================================================
# PINN TRAINING
# ============================================================================

def train_pinn_model(X_train, y_train, G0_train, bolus_train, carbs_train,
                     bergman_model, hyperparams, verbose=False):
    
    pinn_model = PINNModel(X_train.shape[2], bergman_model, 
                           hyperparams['lstm_units'], hyperparams['dense_units'],
                           learn_bergman=hyperparams['learn_bergman'])
    pinn_model.build((None, N_IN, X_train.shape[2]))
    
    # Phase 1: RMSE training
    optimizer_phase1 = tf.keras.optimizers.Adam(learning_rate=hyperparams['lr_phase1'])
    rmse_loss_fn = lambda y_true, y_pred: tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-7)
    
    dataset = tf.data.Dataset.from_tensor_slices((
        X_train, y_train, G0_train, bolus_train, carbs_train
    )).shuffle(10000, seed=RANDOM_SEED).batch(hyperparams['batch_size']).prefetch(2)
    
    warmup_epochs = max(1, int(0.3 * hyperparams['epochs_phase1']))
    
    for epoch in range(hyperparams['epochs_phase1']):
        physics_weight = hyperparams['lambda_physics'] * min(1.0, (epoch + 1) / warmup_epochs)
        
        for X_batch, y_batch, G0_batch, bolus_batch, carbs_batch in dataset:
            with tf.GradientTape() as tape:
                total_loss, _, _, _ = compute_pinn_losses(
                    X_batch, y_batch, G0_batch, bolus_batch, carbs_batch,
                    pinn_model, physics_weight, hyperparams['lambda_insulin'], rmse_loss_fn
                )
                if not tf.math.is_finite(total_loss):
                    continue
            
            grads = tape.gradient(total_loss, pinn_model.trainable_variables)
            if any(g is not None and tf.reduce_any(~tf.math.is_finite(g)) for g in grads):
                continue
            
            grads, _ = tf.clip_by_global_norm(grads, 5.0)
            optimizer_phase1.apply_gradients(zip(grads, pinn_model.trainable_variables))
    
    # Phase 2: Compound loss fine-tuning
    optimizer_phase2 = tf.keras.optimizers.Adam(learning_rate=hyperparams['lr_phase2'])
    
    for epoch in range(hyperparams['epochs_phase2']):
        for X_batch, y_batch, G0_batch, bolus_batch, carbs_batch in dataset:
            with tf.GradientTape() as tape:
                total_loss, _, _, _ = compute_pinn_losses(
                    X_batch, y_batch, G0_batch, bolus_batch, carbs_batch,
                    pinn_model, hyperparams['lambda_physics'], hyperparams['lambda_insulin'],
                    safe_compound_glucose_loss
                )
                if not tf.math.is_finite(total_loss):
                    continue
            
            grads = tape.gradient(total_loss, pinn_model.trainable_variables)
            if any(g is not None and tf.reduce_any(~tf.math.is_finite(g)) for g in grads):
                continue
            
            grads, _ = tf.clip_by_global_norm(grads, 5.0)
            optimizer_phase2.apply_gradients(zip(grads, pinn_model.trainable_variables))
    
    return pinn_model

# ============================================================================
# EVALUATION
# ============================================================================

def evaluate_pinn(pinn_model, X_val, y_val):
    val_predictions_scaled, _ = pinn_model(X_val.astype(np.float32), training=False)
    val_predictions_scaled = val_predictions_scaled.numpy()
    val_predictions_mgdl = inverse_transform_cgm_np(val_predictions_scaled).reshape(-1, N_OUT)
    val_true_mgdl = inverse_transform_cgm_np(y_val.flatten()).reshape(-1, N_OUT)
    
    val_rmse = np.sqrt(np.mean((val_predictions_mgdl - val_true_mgdl)**2))
    val_mae = np.mean(np.abs(val_predictions_mgdl - val_true_mgdl))
    return val_rmse, val_mae

# ============================================================================
# OPTUNA OBJECTIVE
# ============================================================================

def optuna_objective(trial, X_train, y_train, G0_train, bolus_train, carbs_train,
                     X_val, y_val, bergman_model):
    
    hyperparams = {
        'lstm_units': trial.suggest_categorical('lstm_units', [64, 96, 128, 160]),
        'dense_units': trial.suggest_categorical('dense_units', [16, 32, 48, 64]),
        'lr_phase1': trial.suggest_float('lr_phase1', 1e-4, 1e-2, log=True),
        'lr_phase2': trial.suggest_float('lr_phase2', 1e-5, 1e-3, log=True),
        'lambda_physics': trial.suggest_float('lambda_physics', 0.5, 3.0),
        'lambda_insulin': trial.suggest_float('lambda_insulin', 0.001, 0.05, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'epochs_phase1': trial.suggest_int('epochs_phase1', 20, 60),
        'epochs_phase2': trial.suggest_int('epochs_phase2', 10, 30),
        'learn_bergman': trial.suggest_categorical('learn_bergman', [True, False])
    }
    
    pinn_model = train_pinn_model(X_train, y_train, G0_train, bolus_train, carbs_train,
                                  bergman_model, hyperparams)
    val_rmse, _ = evaluate_pinn(pinn_model, X_val, y_val)
    
    return val_rmse

# ============================================================================
# NESTED CROSS-VALIDATION WITH OPTUNA
# ============================================================================

print("\n" + "="*70)
print("NESTED CROSS-VALIDATION WITH OPTUNA")
print("="*70)

# Prepare all data
X_all, y_all, G0_all, bolus_all, carbs_all = prepare_pinn_data(
    train_ids, train_subjects, train_sequences
)

# Prepare Bergman training data
bergman_train_data = []
for sid in train_ids:
    subject_data = train_subjects[sid]
    subject_sequences = train_sequences[sid]
    n_windows = subject_sequences['X'].shape[0]
    if n_windows == 0:
        continue
    G_initial = subject_data['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
    for w in range(n_windows):
        start_idx = N_IN + w
        bolus_future = subject_data['bolus'].values[start_idx : start_idx + N_OUT]
        carbs_future = subject_data['carbs'].values[start_idx : start_idx + N_OUT]
        y_scaled = subject_sequences['y'][w].flatten()
        y_mgdl = cgm_scaler.inverse_transform(y_scaled.reshape(-1, 1)).flatten()
        bergman_train_data.append({
            'G0': G_initial[w], 'bolus': bolus_future,
            'carbs': carbs_future, 'y_true': y_mgdl
        })

G0_berg = np.array([d['G0'] for d in bergman_train_data], dtype=np.float32)
bolus_berg = np.array([d['bolus'] for d in bergman_train_data], dtype=np.float32)
carbs_berg = np.array([d['carbs'] for d in bergman_train_data], dtype=np.float32)
y_berg = np.array([d['y_true'] for d in bergman_train_data], dtype=np.float32)

# Outer CV loop
outer_cv = KFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED)
outer_fold_scores = []
best_hyperparams_per_fold = []

for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_all)):
    print(f"\n{'='*70}")
    print(f"OUTER FOLD {outer_fold + 1}/{N_OUTER_FOLDS}")
    print(f"{'='*70}")
    
    X_train_outer = X_all[train_idx]
    y_train_outer = y_all[train_idx]
    G0_train_outer = G0_all[train_idx]
    bolus_train_outer = bolus_all[train_idx]
    carbs_train_outer = carbs_all[train_idx]
    
    X_test_outer = X_all[test_idx]
    y_test_outer = y_all[test_idx]
    
    # Train Bergman model on outer training data
    print("\nTraining Bergman model for this fold...")
    bergman_outer = train_bergman_model(G0_berg, bolus_berg, carbs_berg, y_berg, epochs=50)
    
    # Inner CV with Optuna for hyperparameter tuning
    print(f"\nRunning Optuna hyperparameter search (Inner CV)...")
    inner_cv = KFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    
    def objective_wrapper(trial):
        inner_scores = []
        for inner_train_idx, inner_val_idx in inner_cv.split(X_train_outer):
            X_train_inner = X_train_outer[inner_train_idx]
            y_train_inner = y_train_outer[inner_train_idx]
            G0_train_inner = G0_train_outer[inner_train_idx]
            bolus_train_inner = bolus_train_outer[inner_train_idx]
            carbs_train_inner = carbs_train_outer[inner_train_idx]
            
            X_val_inner = X_train_outer[inner_val_idx]
            y_val_inner = y_train_outer[inner_val_idx]
            
            score = optuna_objective(trial, X_train_inner, y_train_inner, 
                                   G0_train_inner, bolus_train_inner, carbs_train_inner,
                                   X_val_inner, y_val_inner, bergman_outer)
            inner_scores.append(score)
        
        return np.mean(inner_scores)
    
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective_wrapper, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    
    best_params = study.best_params
    best_hyperparams_per_fold.append(best_params)
    
    print(f"\nBest hyperparameters for fold {outer_fold + 1}:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")
    print(f"Best validation RMSE: {study.best_value:.2f} mg/dL")
    
    # Train final model for this fold with best hyperparameters
    print(f"\nTraining final model for fold {outer_fold + 1} with best hyperparameters...")
    hyperparams_dict = {
        'lstm_units': best_params['lstm_units'],
        'dense_units': best_params['dense_units'],
        'lr_phase1': best_params['lr_phase1'],
        'lr_phase2': best_params['lr_phase2'],
        'lambda_physics': best_params['lambda_physics'],
        'lambda_insulin': best_params['lambda_insulin'],
        'batch_size': best_params['batch_size'],
        'epochs_phase1': best_params['epochs_phase1'],
        'epochs_phase2': best_params['epochs_phase2'],
        'learn_bergman': best_params['learn_bergman']
    }
    
    final_model = train_pinn_model(X_train_outer, y_train_outer, G0_train_outer, 
                                   bolus_train_outer, carbs_train_outer,
                                   bergman_outer, hyperparams_dict)
    
    # Evaluate on outer test fold
    test_rmse, test_mae = evaluate_pinn(final_model, X_test_outer, y_test_outer)
    outer_fold_scores.append({'rmse': test_rmse, 'mae': test_mae})
    
    print(f"\nOuter Fold {outer_fold + 1} Test Results:")
    print(f"  RMSE: {test_rmse:.2f} mg/dL")
    print(f"  MAE: {test_mae:.2f} mg/dL")

# ============================================================================
# AGGREGATE RESULTS
# ============================================================================

print("\n" + "="*70)
print("NESTED CV RESULTS SUMMARY")
print("="*70)

avg_rmse = np.mean([s['rmse'] for s in outer_fold_scores])
std_rmse = np.std([s['rmse'] for s in outer_fold_scores])
avg_mae = np.mean([s['mae'] for s in outer_fold_scores])
std_mae = np.std([s['mae'] for s in outer_fold_scores])

print(f"\nAverage Test RMSE: {avg_rmse:.2f} ± {std_rmse:.2f} mg/dL")
print(f"Average Test MAE: {avg_mae:.2f} ± {std_mae:.2f} mg/dL")

# ============================================================================
# TRAIN FINAL MODEL ON ALL DATA WITH BEST HYPERPARAMETERS
# ============================================================================

print("\n" + "="*70)
print("TRAINING FINAL MODEL ON ALL DATA")
print("="*70)

# Average best hyperparameters across folds (for categorical, use mode)
from collections import Counter

final_hyperparams = {}
for param in best_hyperparams_per_fold[0].keys():
    values = [fold_params[param] for fold_params in best_hyperparams_per_fold]
    if isinstance(values[0], (int, float)):
        final_hyperparams[param] = np.mean(values)
        if param in ['lstm_units', 'dense_units', 'batch_size', 'epochs_phase1', 'epochs_phase2']:
            final_hyperparams[param] = int(final_hyperparams[param])
    else:
        final_hyperparams[param] = Counter(values).most_common(1)[0][0]

print("\nFinal hyperparameters (averaged across folds):")
for param, value in final_hyperparams.items():
    print(f"  {param}: {value}")

# Train Bergman model on all data
print("\nTraining final Bergman model...")
final_bergman = train_bergman_model(G0_berg, bolus_berg, carbs_berg, y_berg, epochs=100)

# Train PINN on all data
print("\nTraining final PINN model...")
final_pinn = train_pinn_model(X_all, y_all, G0_all, bolus_all, carbs_all,
                              final_bergman, final_hyperparams, verbose=True)

# Evaluate on validation set
print("\n" + "="*70)
print("FINAL MODEL VALIDATION")
print("="*70)

val_rmse, val_mae = evaluate_pinn(final_pinn, X_val, y_val)
print(f"\nValidation Metrics:")
print(f"  RMSE: {val_rmse:.2f} mg/dL")
print(f"  MAE: {val_mae:.2f} mg/dL")

# Visualize predictions
val_predictions_scaled, _ = final_pinn(X_val.astype(np.float32), training=False)
val_predictions_mgdl = inverse_transform_cgm_np(val_predictions_scaled.numpy()).reshape(-1, N_OUT)
val_true_mgdl = inverse_transform_cgm_np(y_val.flatten()).reshape(-1, N_OUT)

n_examples = min(3, val_predictions_mgdl.shape[0])
time_horizons = np.arange(1, N_OUT + 1) * 5

fig, axes = plt.subplots(n_examples, 1, figsize=(12, 3*n_examples))
if n_examples == 1:
    axes = [axes]

for i in range(n_examples):
    axes[i].plot(time_horizons, val_true_mgdl[i], 'o-', label='True', color='blue')
    axes[i].plot(time_horizons, val_predictions_mgdl[i], 's-', label='Predicted', color='red')
    axes[i].axhline(y=70, color='orange', linestyle='--', alpha=0.5)
    axes[i].axhline(y=180, color='orange', linestyle='--', alpha=0.5)
    axes[i].set_xlabel('Time Horizon (minutes)')
    axes[i].set_ylabel('Blood Glucose (mg/dL)')
    axes[i].set_title(f'Validation Example {i+1}')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print("Final model ready for evaluation on test data")

In [ ]:
"""
Optuna Hyperparameter Tuning for PINN Model
Uses validation data from OhioT1DM for optimization
Test data (BrisT1D) remains completely untouched
"""

import optuna
from optuna.samplers import TPESampler
import numpy as np
from sklearn.model_selection import train_test_split

# ============================================================================
# CONFIGURATION
# ============================================================================

N_TRIALS = 50  # Number of Optuna trials
N_JOBS = 1  # Parallel jobs (set to -1 for all cores, but TensorFlow may conflict)
TIMEOUT = 3600  # Max time in seconds (1 hour)
PRUNING_ENABLED = True  # Enable early stopping of unpromising trials

# Validation split for hyperparameter tuning
HYPERPARAM_VAL_SIZE = 0.2  # 20% of training data for validation
RANDOM_SEED = 42

print("="*70)
print("OPTUNA HYPERPARAMETER OPTIMIZATION")
print("="*70)
print(f"Number of trials: {N_TRIALS}")
print(f"Validation split: {HYPERPARAM_VAL_SIZE * 100:.0f}% of training data")
print(f"Test data: BrisT1D (completely held out)")

# ============================================================================
# PREPARE DATA FOR HYPERPARAMETER TUNING
# ============================================================================

# Split training subjects into train/val for hyperparameter tuning
all_train_ids = np.array(sorted(train_subjects.keys()))
hp_train_ids, hp_val_ids = train_test_split(
    all_train_ids, 
    test_size=HYPERPARAM_VAL_SIZE, 
    random_state=RANDOM_SEED
)

print(f"\nHyperparameter tuning data split:")
print(f"  Training subjects: {len(hp_train_ids)}")
print(f"  Validation subjects: {len(hp_val_ids)}")

# Prepare data
hp_train_data = prepare_fold_data(hp_train_ids)
hp_val_data = prepare_fold_data(hp_val_ids)

print(f"  Training samples: {hp_train_data['X'].shape[0]}")
print(f"  Validation samples: {hp_val_data['X'].shape[0]}")

# ============================================================================
# OPTUNA OBJECTIVE FUNCTION
# ============================================================================

def objective(trial):
    """
    Optuna objective function for hyperparameter optimization
    
    Args:
        trial: Optuna trial object
    
    Returns:
        validation RMSE (lower is better)
    """
    
    # Sample hyperparameters
    hyperparams = {
        # Learning rates
        'lr_phase_1': trial.suggest_float('lr_phase_1', 1e-4, 1e-2, log=True),
        'lr_phase_2': trial.suggest_float('lr_phase_2', 1e-5, 1e-3, log=True),
        
        # Physics loss weights
        'lambda_physics': trial.suggest_float('lambda_physics', 0.1, 5.0),
        'lambda_insulin': trial.suggest_float('lambda_insulin', 0.001, 0.1, log=True),
        
        # Architecture
        'lstm_units': trial.suggest_categorical('lstm_units', [64, 128, 256]),
        'dense_units': trial.suggest_categorical('dense_units', [16, 32, 64]),
        
        # Training
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'epochs_phase_1': trial.suggest_int('epochs_phase_1', 20, 100),
        'epochs_phase_2': trial.suggest_int('epochs_phase_2', 10, 50),
    }
    
    # Train model with sampled hyperparameters
    try:
        _, val_rmse = train_pinn_model(
            hp_train_data, 
            hp_val_data, 
            hyperparams, 
            verbose=False  # Suppress output for cleaner logs
        )
        
        # Report intermediate value for pruning
        trial.report(val_rmse, step=0)
        
        # Check if trial should be pruned
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        return val_rmse
    
    except Exception as e:
        # Return high error if training fails
        print(f"Trial {trial.number} failed: {e}")
        return 1000.0

# ============================================================================
# RUN OPTUNA OPTIMIZATION
# ============================================================================

print("\n" + "="*70)
print("STARTING HYPERPARAMETER OPTIMIZATION")
print("="*70)

# Create study
sampler = TPESampler(seed=RANDOM_SEED)
pruner = optuna.pruners.MedianPruner() if PRUNING_ENABLED else optuna.pruners.NopPruner()

study = optuna.create_study(
    direction='minimize',  # Minimize RMSE
    sampler=sampler,
    pruner=pruner,
    study_name='pinn_hyperparam_optimization'
)

# Optimize
study.optimize(
    objective, 
    n_trials=N_TRIALS,
    timeout=TIMEOUT,
    n_jobs=N_JOBS,
    show_progress_bar=True
)

# ============================================================================
# RESULTS
# ============================================================================

print("\n" + "="*70)
print("OPTIMIZATION COMPLETE")
print("="*70)

print(f"\nNumber of finished trials: {len(study.trials)}")
print(f"Number of pruned trials: {len(study.get_trials(states=[optuna.trial.TrialState.PRUNED]))}")
print(f"Number of complete trials: {len(study.get_trials(states=[optuna.trial.TrialState.COMPLETE]))}")

print(f"\nBest trial:")
best_trial = study.best_trial
print(f"  Value (RMSE): {best_trial.value:.2f} mg/dL")

print(f"\nBest hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

# ============================================================================
# VISUALIZATION
# ============================================================================

print("\n" + "="*70)
print("GENERATING OPTIMIZATION PLOTS")
print("="*70)

# Plot optimization history
fig1 = optuna.visualization.plot_optimization_history(study)
fig1.update_layout(title="Optimization History")
fig1.show()

# Plot parameter importances
fig2 = optuna.visualization.plot_param_importances(study)
fig2.update_layout(title="Hyperparameter Importances")
fig2.show()

# Plot parallel coordinate
fig3 = optuna.visualization.plot_parallel_coordinate(study)
fig3.update_layout(title="Parallel Coordinate Plot")
fig3.show()

# Plot contour (for top parameters)
if len(best_trial.params) >= 2:
    param_names = list(best_trial.params.keys())[:2]
    fig4 = optuna.visualization.plot_contour(study, params=param_names)
    fig4.update_layout(title=f"Contour Plot: {param_names[0]} vs {param_names[1]}")
    fig4.show()

# ============================================================================
# TRAIN FINAL MODEL WITH BEST HYPERPARAMETERS
# ============================================================================

print("\n" + "="*70)
print("TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
print("="*70)

# Prepare full training data (all OhioT1DM subjects)
full_train_data = prepare_fold_data(all_train_ids)

print(f"Training on all OhioT1DM subjects: {len(all_train_ids)}")
print(f"Total training samples: {full_train_data['X'].shape[0]}")

# Create best hyperparameters dict
best_hyperparams = {
    'lr_phase_1': best_trial.params['lr_phase_1'],
    'lr_phase_2': best_trial.params['lr_phase_2'],
    'lambda_physics': best_trial.params['lambda_physics'],
    'lambda_insulin': best_trial.params['lambda_insulin'],
    'lstm_units': best_trial.params['lstm_units'],
    'dense_units': best_trial.params['dense_units'],
    'batch_size': best_trial.params['batch_size'],
    'epochs_phase_1': best_trial.params['epochs_phase_1'],
    'epochs_phase_2': best_trial.params['epochs_phase_2'],
}

# Prepare test data for final validation
test_data_final = {
    'X': [], 'y': [], 'G0': [], 'bolus': [], 'carbs': []
}

for sid in test_subjects.keys():
    subject_data = test_subjects[sid]
    subject_sequences = test_sequences[sid]
    
    n_windows = subject_sequences['X'].shape[0]
    if n_windows == 0:
        continue
    
    test_data_final['X'].append(subject_sequences['X'])
    test_data_final['y'].append(subject_sequences['y'])
    
    G_initial = subject_data['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
    test_data_final['G0'].append(G_initial)
    
    for w in range(n_windows):
        start_idx = N_IN + w
        bolus_future = subject_data['bolus'].values[start_idx : start_idx + N_OUT]
        carbs_future = subject_data['carbs'].values[start_idx : start_idx + N_OUT]
        test_data_final['bolus'].append(bolus_future)
        test_data_final['carbs'].append(carbs_future)

test_data_final = {
    'X': np.concatenate(test_data_final['X'], axis=0).astype(np.float32),
    'y': np.concatenate(test_data_final['y'], axis=0).reshape(-1, N_OUT).astype(np.float32),
    'G0': np.concatenate(test_data_final['G0'], axis=0).astype(np.float32),
    'bolus': np.array(test_data_final['bolus'], dtype=np.float32),
    'carbs': np.array(test_data_final['carbs'], dtype=np.float32)
}

# Train final model
final_optimized_model, _ = train_pinn_model(
    full_train_data, 
    test_data_final,  # Use test as validation just for monitoring
    best_hyperparams, 
    verbose=True
)

# ============================================================================
# FINAL EVALUATION ON TEST SET
# ============================================================================

print("\n" + "="*70)
print("FINAL EVALUATION ON TEST SET (BrisT1D)")
print("="*70)

# Make predictions
test_predictions_scaled, _ = final_optimized_model(test_data_final['X'], training=False)
test_predictions_scaled = test_predictions_scaled.numpy()

test_predictions_mgdl = inverse_transform_cgm_np(test_predictions_scaled)
test_true_mgdl = inverse_transform_cgm_np(test_data_final['y'].flatten())

test_predictions_mgdl = test_predictions_mgdl.reshape(-1, N_OUT)
test_true_mgdl = test_true_mgdl.reshape(-1, N_OUT)

# Calculate metrics
test_rmse_optimized = np.sqrt(np.mean((test_predictions_mgdl - test_true_mgdl)**2))
test_mae_optimized = np.mean(np.abs(test_predictions_mgdl - test_true_mgdl))

print(f"\nOptimized Model Test Metrics:")
print(f"  RMSE: {test_rmse_optimized:.2f} mg/dL")
print(f"  MAE: {test_mae_optimized:.2f} mg/dL")

# Per-horizon metrics
print(f"\nPer-Horizon Performance:")
for h in range(N_OUT):
    horizon_min = (h + 1) * 5
    horizon_rmse = np.sqrt(np.mean((test_predictions_mgdl[:, h] - test_true_mgdl[:, h])**2))
    horizon_mae = np.mean(np.abs(test_predictions_mgdl[:, h] - test_true_mgdl[:, h]))
    print(f"  {horizon_min:2d} min: RMSE={horizon_rmse:.2f} mg/dL, MAE={horizon_mae:.2f} mg/dL")

print("\n" + "="*70)
print("HYPERPARAMETER OPTIMIZATION COMPLETE")
print("="*70)
print(f"""
Summary:
  ✓ Optimized {len(best_trial.params)} hyperparameters
  ✓ Completed {len(study.trials)} trials
  ✓ Best validation RMSE: {best_trial.value:.2f} mg/dL
  ✓ Test RMSE (BrisT1D): {test_rmse_optimized:.2f} mg/dL
  ✓ No data leakage: Test set never used in optimization
  
Variables available:
  - study: Optuna study object with all results
  - best_trial: Best trial from optimization
  - best_hyperparams: Dictionary of best hyperparameters
  - final_optimized_model: Model trained with best hyperparameters
  - test_predictions_mgdl: Final test predictions
  - test_true_mgdl: True test values
  
Next steps:
  - Use best_hyperparams for nested CV
  - Compare with baseline hyperparameters
  - Analyze parameter importance plots
""")

## Evaluation - Validation Data:

In [ ]:
"""
Comprehensive Model Evaluation on Validation Data
Evaluates LSTM, Bergman, PINN, and ML Corrector models using validation subjects from OhioT1DM
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from collections import Counter
import math
from scipy.signal import correlate
import warnings

# ============================================================================
# SETUP: Create validation subjects dictionary
# ============================================================================

print("="*70)
print("PREPARING VALIDATION DATA")
print("="*70)

# Create val_subjects dictionary from train_subjects using val_ids
val_subjects = {sid: train_subjects[sid] for sid in val_ids}
print(f"Validation subjects: {len(val_subjects)}")
print(f"Validation sequences: {X_val.shape[0]} windows")

# ============================================================================
# HELPER FUNCTIONS (from evaluation code)
# ============================================================================

def inv_cgm_with_scaler(y_scaled):
    """Convert scaled CGM values back to mg/dL"""
    if y_scaled is None or np.asarray(y_scaled).size == 0:
        return np.array([]).reshape(0, N_OUT)
    arr = np.asarray(y_scaled)
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr.reshape(arr.shape[0], arr.shape[1])
    flat = arr.reshape(-1, 1)
    inv = cgm_scaler.inverse_transform(flat).reshape(-1, N_OUT)
    return inv

HYPO = 70.0
HYPER = 180.0

def glycemia_class(x):
    """Classify glucose values: 0=hypo (<70), 1=normo (70-180), 2=hyper (>180)"""
    x = np.asarray(x).reshape(-1)
    cls = np.full(x.shape, -1, dtype=int)
    cls[x < HYPO] = 0
    cls[(x >= HYPO) & (x <= HYPER)] = 1
    cls[x > HYPER] = 2
    return cls

#def temporal_gain(y_true, y_pred, prediction_horizon):
    """
    Temporal Gain (TG): median time gained by prediction before delay degrades accuracy
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    horizon = int(N_OUT)
    dt_minutes = float(prediction_horizon) / float(horizon)
    
    if y_true.ndim == 1 and y_pred.ndim == 1 and (y_true.size % horizon == 0):
        y_true = y_true.reshape(-1, horizon)
        y_pred = y_pred.reshape(-1, horizon)
    
    if y_true.ndim != 2 or y_pred.ndim != 2:
        return int(prediction_horizon)
    
    n_windows, L = y_true.shape
    max_lag_steps = int(prediction_horizon // dt_minutes)
    max_lag_steps = min(max_lag_steps, L - 1)
    
    tg_windows = []
    for i in range(n_windows):
        t = y_true[i].astype(float)
        p = y_pred[i].astype(float)
        
        if np.all(np.isnan(t)) or np.all(np.isnan(p)):
            tg_windows.append(0)
            continue
        
        if np.nanstd(t) < 1e-8 or np.nanstd(p) < 1e-8:
            tg_windows.append(0)
            continue
        
        t0 = t - np.nanmean(t)
        p0 = p - np.nanmean(p)
        
        cc = correlate(t0, p0, mode='full')
        full_lags = np.arange(-L + 1, L)
        
        valid_mask = (full_lags >= 0) & (full_lags <= max_lag_steps)
        valid_lags = full_lags[valid_mask]
        valid_cc = cc[valid_mask]
        
        if valid_cc.size == 0:
            tg_windows.append(0)
            continue
        
        idx = np.argmax(np.abs(valid_cc))
        delay_steps = int(valid_lags[idx])
        delay_minutes = delay_steps * dt_minutes
        tg_val = prediction_horizon - delay_minutes
        tg_val = max(0.0, min(float(prediction_horizon), float(tg_val)))
        tg_windows.append(tg_val)
    
    if len(tg_windows) == 0:
        return int(prediction_horizon)
    return int(round(np.median(tg_windows)))

def temporal_gain(y_true, y_pred, prediction_horizon):
    """
    Constrained Cross-Correlation Temporal Gain (mirrors the pasted implementation).
    
    Measures temporal synchronization by finding optimal lag within constrained range.
    Only searches for negative lags (prediction leading) or zero lag.
    
    NOTE: This has the same limitation as the original - models with good temporal
    alignment will score high regardless of actual predictive value.
    
    Args:
        y_true: Ground truth glucose values (n_windows, n_steps) or (n_samples,)
        y_pred: Predicted glucose values (n_windows, n_steps) or (n_samples,)
        prediction_horizon: Total prediction window in minutes (e.g., 30 or 60)
    
    Returns:
        Temporal gain in minutes (median across windows)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    # Handle 1D vs 2D input
    if y_true.ndim == 1 and y_pred.ndim == 1:
        # Single sequence - compute once
        y_true_arr = y_true.reshape(1, -1)
        y_pred_arr = y_pred.reshape(1, -1)
    elif y_true.ndim == 2 and y_pred.ndim == 2:
        # Multiple windows
        y_true_arr = y_true
        y_pred_arr = y_pred
    else:
        return int(prediction_horizon)
    
    n_windows, n_steps = y_true_arr.shape
    dt_minutes = 5  # Assuming 5-minute timesteps
    
    tg_windows = []
    
    for i in range(n_windows):
        t = y_true_arr[i].astype(float)
        p = y_pred_arr[i].astype(float)
        
        # Skip invalid windows
        if np.all(np.isnan(t)) or np.all(np.isnan(p)):
            continue
        
        # Remove mean for correlation (standardization)
        t_centered = t - np.nanmean(t)
        p_centered = p - np.nanmean(p)
        
        # Compute cross-correlation
        cross_corr = correlate(t_centered, p_centered, mode='full')
        
        # Define constrained lag range
        lag_min = -prediction_horizon // dt_minutes  # e.g., -12 for PH=60
        lag_max = 0  # Only look at prediction leading or synchronized
        
        # Calculate full range of lags
        full_lags = np.arange(-len(t) + 1, len(p))
        
        # Filter valid lags within the prediction horizon
        valid_mask = (full_lags >= lag_min) & (full_lags <= lag_max)
        valid_lags = full_lags[valid_mask]
        valid_cross_corr = cross_corr[valid_mask]
        
        if len(valid_cross_corr) == 0:
            tg_windows.append(0)
            continue
        
        # Find lag with maximum correlation
        max_corr_idx = np.argmax(np.abs(valid_cross_corr))
        optimal_lag_steps = valid_lags[max_corr_idx]
        
        # Convert to temporal gain in minutes
        # If lag = 0 (synchronized), TG = prediction_horizon
        # If lag = -12 (max delay), TG = 0
        tg_minutes = prediction_horizon + (optimal_lag_steps * dt_minutes)
        tg_minutes = max(0.0, min(float(prediction_horizon), float(tg_minutes)))
        
        tg_windows.append(tg_minutes)
    
    if len(tg_windows) == 0:
        return int(prediction_horizon)
    
    return int(round(np.median(tg_windows)))

def g_mean(y_true, y_pred):
    """Geometric mean of per-class recall"""
    from sklearn.metrics import recall_score
    t = glycemia_class(y_true)
    p = glycemia_class(y_pred)
    recalls = recall_score(t, p, average=None, zero_division=0)
    recalls = np.where(recalls == 0, 1e-10, recalls)
    return float(np.exp(np.mean(np.log(recalls))))

def _sigma_ge(x, a, eps):
    """Smooth step function (C2 continuous)"""
    x = np.asarray(x, dtype=float)
    a = np.asarray(a, dtype=float)
    out = np.zeros_like(x, dtype=float)
    
    if np.asarray(eps).size == 1 and eps <= 0:
        out = (x > a).astype(float)
        return out
    
    left = x <= a
    right = x >= (a + eps)
    mid = (~left) & (~right)
    
    out[right] = 1.0
    
    if np.any(mid):
        mid_vals = x[mid]
        a_arr = np.asarray(a)
        if a_arr.shape == ():
            a_mid = a_arr
        else:
            a_b = np.broadcast_to(a_arr, x.shape)
            a_mid = a_b[mid]
        
        xi = 2.0 * (mid_vals - a_mid) / eps - 1.0
        first_half = xi <= 0
        second_half = ~first_half
        
        val = np.empty_like(xi)
        if np.any(first_half):
            xi_f = xi[first_half]
            val[first_half] = (-0.5 * xi_f**4) - (xi_f**3) + xi_f + 0.5
        if np.any(second_half):
            xi_s = xi[second_half]
            val[second_half] = (0.5 * xi_s**4) - (xi_s**3) + xi_s + 0.5
        
        out[mid] = val
    
    return out

def _sigma_le_bar(x, a, eps):
    """Smooth step function (mirror of _sigma_ge)"""
    x = np.asarray(x, dtype=float)
    a = np.asarray(a, dtype=float)
    out = np.zeros_like(x, dtype=float)
    
    if np.asarray(eps).size == 1 and eps <= 0:
        out = (x <= a).astype(float)
        return out
    
    left = x <= (a - eps)
    right = x >= a
    mid = (~left) & (~right)
    
    out[left] = 1.0
    
    if np.any(mid):
        mid_vals = x[mid]
        a_arr = np.asarray(a)
        if a_arr.shape == ():
            a_mid = a_arr
        else:
            a_b = np.broadcast_to(a_arr, x.shape)
            a_mid = a_b[mid]
        
        xi = 2.0 * (mid_vals - a_mid) / eps + 1.0
        first_half = xi <= 0
        second_half = ~first_half
        
        val = np.empty_like(xi)
        if np.any(first_half):
            xi_f = xi[first_half]
            val[first_half] = (0.5 * xi_f**4) - (xi_f**3) + xi_f + 0.5
        if np.any(second_half):
            xi_s = xi[second_half]
            val[second_half] = (-0.5 * xi_s**4) - (xi_s**3) + xi_s + 0.5
        
        out[mid] = val
    
    return out

def pen_function(g, ghat, alphaL=1.5, alphaH=1.0, TL=85.0, TH=155.0,
                 betaL=30.0, betaH=30.0, gammaL=40.0, gammaH=40.0):
    """Penalty function for glucose-weighted metrics"""
    g = np.asarray(g, dtype=float)
    ghat = np.asarray(ghat, dtype=float)
    
    if g.shape == () and ghat.shape != ():
        g = np.full_like(ghat, float(g))
    elif ghat.shape == () and g.shape != ():
        ghat = np.full_like(g, float(ghat))
    else:
        bshape = np.broadcast_shapes(g.shape, ghat.shape)
        g = np.broadcast_to(g, bshape).astype(float)
        ghat = np.broadcast_to(ghat, bshape).astype(float)
    
    partL = _sigma_le_bar(g, TL, betaL) * _sigma_ge(ghat, g, gammaL)
    partH = _sigma_ge(g, TH, betaH) * _sigma_le_bar(ghat, g, gammaH)
    Pen = 1.0 + alphaL * partL + alphaH * partH
    return Pen

def glucose_weighted_rmse(y_true, y_pred, alphaL=1.5, alphaH=1.0, TL=85.0, TH=155.0,
                          betaL=30.0, betaH=30.0, gammaL=40.0, gammaH=40.0):
    """Glucose-weighted RMSE"""
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    if y_true.size == 0:
        return np.nan
    
    Pen = pen_function(y_true, y_pred, alphaL, alphaH, TL, TH, betaL, betaH, gammaL, gammaH)
    gse = (y_true - y_pred)**2 * Pen
    return math.sqrt(np.mean(gse))

def glucose_weighted_mae(y_true, y_pred, alphaL=1.5, alphaH=1.0, TL=85.0, TH=155.0,
                         betaL=30.0, betaH=30.0, gammaL=40.0, gammaH=40.0):
    """Glucose-weighted MAE"""
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    if y_true.size == 0:
        return np.nan
    
    Pen = pen_function(y_true, y_pred, alphaL, alphaH, TL, TH, betaL, betaH, gammaL, gammaH)
    gabs = np.abs(y_true - y_pred) * Pen
    return float(np.mean(gabs))

def clarke_zone_for_point(ref, pred):
    """Clarke Error Grid zone classification"""
    if np.isnan(ref) or np.isnan(pred):
        return None
    if (ref >= 70 and abs(pred - ref) <= 0.2 * ref) or (ref < 70 and abs(pred - ref) <= 20):
        return 'A'
    if (pred >= 330 and ref <= 50) or (pred <= 50 and ref >= 330):
        return 'E'
    if ref < 70 and pred >= 180:
        return 'C'
    if ref >= 180 and pred < 70:
        return 'D'
    return 'B'

def parkes_zone_for_point(ref, pred):
    """Parkes (Consensus) Error Grid zone classification"""
    if np.isnan(ref) or np.isnan(pred):
        return None
    err = pred - ref
    rel_err = abs(err) / max(ref, 1.0)
    if (ref < 70 and abs(err) <= 15) or (ref >= 70 and rel_err <= 0.2):
        return 'A'
    if rel_err <= 0.35:
        return 'B'
    if (ref < 70 and pred > 180) or (ref > 180 and pred < 70):
        return 'D'
    if rel_err > 0.6:
        return 'E'
    return 'C'

def compute_glycemia_confusion(y_true, y_pred):
    """Compute confusion matrix for glycemia detection"""
    y_t = np.asarray(y_true).reshape(-1)
    y_p = np.asarray(y_pred).reshape(-1)
    mask = np.isfinite(y_t) & np.isfinite(y_p)
    y_t = y_t[mask]
    y_p = y_p[mask]
    
    counts = np.zeros((3, 3), dtype=int)
    if y_t.size == 0:
        return counts, np.full_like(counts, np.nan, dtype=float)
    
    tcls = glycemia_class(y_t)
    pcls = glycemia_class(y_p)
    
    for t, p in zip(tcls, pcls):
        if 0 <= t <= 2 and 0 <= p <= 2:
            counts[t, p] += 1
    
    pct = np.full_like(counts, np.nan, dtype=float)
    for i in range(3):
        s = counts[i].sum()
        if s > 0:
            pct[i, :] = counts[i, :] / float(s)
    
    return counts, pct

def plot_confusion_heatmaps(counts, pct, title=""):
    """Plot glycemia detection confusion matrix heatmap"""
    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    im = ax.imshow(pct, vmin=0.0, vmax=1.0, cmap='Blues', interpolation='nearest')
    ax.set_xticks(np.arange(3))
    ax.set_yticks(np.arange(3))
    labels = ['Pred Hypo\n(<70)', 'Pred Normo\n(70-180)', 'Pred Hyper\n(>180)']
    ax.set_xticklabels(labels)
    ax.set_yticklabels(['True Hypo', 'True Normo', 'True Hyper'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    
    for i in range(3):
        for j in range(3):
            c = counts[i, j]
            p = pct[i, j]
            if np.isnan(p):
                txt = "n/a\n(0)"
            else:
                txt = f"{p*100:4.1f}%\n({c})"
            ax.text(j, i, txt, ha='center', va='center', color='black', fontsize=10)
    
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

# ============================================================================
# MODEL PREDICTION FUNCTIONS
# ============================================================================

def persistence_predict_on_val_subject(sid):
    """Persistence baseline: last observed value repeated"""
    info = val_sequences.get(sid)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    
    X_subj = info['X']
    n_windows = X_subj.shape[0]
    
    # Last CGM value from input sequence (already scaled)
    last_cgm = X_subj[:, -1, 0]  # Shape: (n_windows,)
    
    # Repeat for all forecast steps
    preds_scaled = np.tile(last_cgm.reshape(-1, 1), (1, N_OUT))
    return preds_scaled.reshape(n_windows, N_OUT, 1)

def bergman_predict_on_val_subject(sid):
    """Bergman physiological model predictions"""
    info = val_sequences.get(sid)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    
    X_subj = info['X']
    subj_df = val_subjects[sid]
    n_windows = X_subj.shape[0]
    
    # Initial glucose values (mg/dL) for each window
    G_inits = subj_df['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
    
    # Extract insulin and carbs for forecast horizon
    bolus_mat = np.zeros((n_windows, N_OUT))
    carbs_mat = np.zeros((n_windows, N_OUT))
    
    for w in range(n_windows):
        start_idx = N_IN + w
        bolus_mat[w, :] = subj_df['bolus'].values[start_idx:start_idx+N_OUT]
        carbs_mat[w, :] = subj_df['carbs'].values[start_idx:start_idx+N_OUT]
    
    # Bergman parameters (should be defined from bergman_params variable)
    p1, p2, p3, k_i, Gb = bergman_params
    dt_scale = 1.0  # 5min/5min scaling factor
    
    # Simulate forward
    preds_mg = np.zeros((n_windows, N_OUT))
    G = G_inits.copy().astype(float)
    I = np.zeros(n_windows)
    
    for t in range(N_OUT):
        # Update insulin with decay and new dose
        I = I * np.exp(-k_i) + bolus_mat[:, t]
        
        # Meal effect
        meal_effect = p3 * carbs_mat[:, t]
        
        # Glucose dynamics
        dG = -p1 * (G - Gb) - p2 * I + meal_effect
        G = G + dG * dt_scale
        preds_mg[:, t] = G
    
    # Scale to [0, 1]
    preds_scaled = cgm_scaler.transform(preds_mg.reshape(-1, 1)).reshape(n_windows, N_OUT)
    return preds_scaled.reshape(n_windows, N_OUT, 1)

def lstm_predict_on_val_subject(sid):
    """LSTM model predictions"""
    info = val_sequences.get(sid)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    
    X_subj = info['X']
    preds_scaled = lstm_model.predict(X_subj, batch_size=64, verbose=0)
    
    if preds_scaled.ndim == 2:
        preds_scaled = preds_scaled.reshape(-1, N_OUT, 1)
    
    return preds_scaled

def pinn_predict_on_val_subject(sid):
    """Physics-Informed Neural Network predictions"""
    info = val_sequences.get(sid)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    
    X_subj = info['X'].astype(np.float32)
    preds_scaled, _ = pinn_model(X_subj, training=False)
    preds_scaled = preds_scaled.numpy()
    
    if preds_scaled.ndim == 2:
        preds_scaled = preds_scaled.reshape(-1, N_OUT, 1)
    
    return preds_scaled

def ml_corrector_predict_on_val_subject(sid):
    """Hybrid ML corrector: Bergman + learned residual"""
    info = val_sequences.get(sid)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    
    X_subj = info['X']
    
    # Get residual predictions from corrector model
    resid_scaled = lstm_corrector.predict(X_subj, batch_size=128, verbose=0)
    
    # Get Bergman baseline predictions
    bergman_preds = bergman_predict_on_val_subject(sid)
    bergman_preds_scaled = bergman_preds.reshape(-1, N_OUT)
    
    # Correct Bergman with residual
    corrected_scaled = bergman_preds_scaled + resid_scaled
    return corrected_scaled.reshape(-1, N_OUT, 1)

# ============================================================================
# COLLECT PREDICTIONS FROM ALL MODELS
# ============================================================================

print("\n" + "="*70)
print("GENERATING PREDICTIONS ON VALIDATION DATA")
print("="*70)

models = {
    'persistence': persistence_predict_on_val_subject,
    'bergman': bergman_predict_on_val_subject,
    'lstm': lstm_predict_on_val_subject,
    'pinn': pinn_predict_on_val_subject,
    'ml_corrector': ml_corrector_predict_on_val_subject
}

# Collect predictions for all validation subjects
all_results = {}
for mname, mpredict in models.items():
    print(f"\nModel: {mname}")
    Ys = []
    for sid in val_ids:
        yhat_scaled = mpredict(sid)
        yhat_scaled = np.asarray(yhat_scaled)
        
        if yhat_scaled.ndim == 3 and yhat_scaled.shape[-1] == 1:
            yhat_scaled = yhat_scaled.reshape(yhat_scaled.shape[0], yhat_scaled.shape[1])
        
        yhat_mg = inv_cgm_with_scaler(yhat_scaled)
        Ys.append(yhat_mg)
        print(f"  Subject {sid}: {yhat_mg.shape[0]} windows")
    
    all_results[mname] = np.vstack(Ys) if len(Ys) else np.zeros((0, N_OUT))
    print(f"  Total: {all_results[mname].shape[0]} windows")

# Build ground truth (all validation windows)
y_true_all_windows = []
for sid in val_ids:
    info = val_sequences.get(sid)
    y_val_subj = info['y']
    if y_val_subj.shape[0] == 0:
        continue
    y_true_mg = inv_cgm_with_scaler(y_val_subj.reshape(-1, N_OUT))
    y_true_all_windows.append(y_true_mg)

if len(y_true_all_windows) == 0:
    raise RuntimeError("No validation windows found.")

y_true_all = np.vstack(y_true_all_windows)
print(f"\nGround truth: {y_true_all.shape[0]} windows")

# ============================================================================
# EVALUATION METRICS
# ============================================================================

print("\n" + "="*70)
print("EVALUATION METRICS")
print("="*70)

# Per-horizon RMSE/MAE
horiz_steps = np.arange(1, N_OUT+1) * 5
print("\n=== Per-horizon RMSE / MAE for each model ===")

per_horizon = {}
for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        print(f"\nModel {mname}: no predictions")
        continue
    
    rmse_h = []
    mae_h = []
    
    for h in range(N_OUT):
        y_t = y_true_all[:, h]
        y_p = ypred_all[:, h]
        rmse_h.append(np.sqrt(mean_squared_error(y_t, y_p)))
        mae_h.append(mean_absolute_error(y_t, y_p))
    
    per_horizon[mname] = {'rmse': rmse_h, 'mae': mae_h}
    
    print(f"\nModel: {mname}")
    print("Horizon (min):", list(horiz_steps))
    print("RMSE (mg/dL): ", [round(v, 2) for v in rmse_h])
    print("MAE  (mg/dL): ", [round(v, 2) for v in mae_h])

# Final-step aggregated metrics
print("\n=== Final-step aggregated metrics (30 min) ===")

for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        continue
    
    ref = y_true_all[:, -1]
    pred = ypred_all[:, -1]
    
    rmse_val = np.sqrt(mean_squared_error(ref, pred))
    mae_val = mean_absolute_error(ref, pred)
    gw_rmse = glucose_weighted_rmse(ref, pred)
    gw_mae = glucose_weighted_mae(ref, pred)
    tg = temporal_gain(y_true_all, ypred_all, prediction_horizon=5*N_OUT)
    gm = g_mean(ref, pred)
    
    print(f"\nModel: {mname}")
    print(f"  Windows: {len(ref)}")
    print(f"  RMSE: {rmse_val:.3f}  MAE: {mae_val:.3f}")
    print(f"  Glucose-weighted RMSE: {gw_rmse:.3f}  Glucose-weighted MAE: {gw_mae:.3f}")
    print(f"  Temporal gain (min): {tg}   G-Mean: {gm:.4f}")

# ============================================================================
# CLARKE & PARKES ERROR GRIDS + SCATTER PLOTS
# ============================================================================

print("\n" + "="*70)
print("CLARKE & PARKES ERROR GRIDS")
print("="*70)

for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        continue
    
    ref = y_true_all[:, -1]
    pred = ypred_all[:, -1]
    
    # Compute zones
    cz = [clarke_zone_for_point(r, p) for r, p in zip(ref, pred)]
    pz = [parkes_zone_for_point(r, p) for r, p in zip(ref, pred)]
    
    c_counts = dict(Counter(cz))
    p_counts = dict(Counter(pz))
    total = len(ref)
    
    print(f"\n{mname} Clarke zones: ", {k: (c_counts[k], round(100*c_counts[k]/total, 2)) 
                                         for k in c_counts})
    print(f"{mname} Parkes zones : ", {k: (p_counts[k], round(100*p_counts[k]/total, 2)) 
                                        for k in p_counts})
    
    # Scatter plots
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
    # Clarke grid
    ax0 = ax[0]
    ax0.scatter(ref, pred, s=6, alpha=0.5)
    ax0.plot([0, 400], [0, 400], '--', linewidth=0.8, color='gray')
    xs = np.linspace(0, 400, 100)
    ax0.plot(xs, 1.2*xs, linewidth=0.6, color='gray')
    ax0.plot(xs, 0.8*xs, linewidth=0.6, color='gray')
    ax0.axvline(70, linestyle=':', color='gray')
    ax0.axhline(70, linestyle=':', color='gray')
    ax0.set_xlim(0, 400)
    ax0.set_ylim(0, 400)
    ax0.set_xlabel('Reference (mg/dL)')
    ax0.set_ylabel('Predicted (mg/dL)')
    ax0.set_title(f"{mname} — Clarke (final step)")
    ax0.grid(True, alpha=0.3)
    
    # Parkes grid
    ax1 = ax[1]
    ax1.scatter(ref, pred, s=6, alpha=0.5)
    ax1.plot([0, 400], [0, 400], '--', linewidth=0.8, color='gray')
    ax1.set_xlim(0, 400)
    ax1.set_ylim(0, 400)
    ax1.set_xlabel('Reference (mg/dL)')
    ax1.set_ylabel('Predicted (mg/dL)')
    ax1.set_title(f"{mname} — Parkes (visual)")
    ax1.grid(True, alpha=0.3)
    
    plt.suptitle(f"{mname} Pred vs Ref (final step, validation) — N={total}")
    plt.tight_layout()
    plt.show()

# ============================================================================
# GLYCEMIA DETECTION CONFUSION MATRICES
# ============================================================================

print("\n" + "="*70)
print("GLYCEMIA DETECTION CONFUSION MATRICES")
print("="*70)

for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        continue
    
    counts, pct = compute_glycemia_confusion(y_true_all[:, -1], ypred_all[:, -1])
    print(f"\nGlycemia detection — {mname} (final step):")
    plot_confusion_heatmaps(counts, pct, title=f"{mname} — Glycemia detection (validation)")

# ============================================================================
# MULTI-STEP TRAJECTORY PLOTS (Selected Subjects, 24h view)
# ============================================================================

print("\n" + "="*70)
print("MULTI-STEP TRAJECTORY PLOTS (24 hour view)")
print("="*70)

dt = 5  # minutes
prediction_horizon_min = 30
ph_steps = int(prediction_horizon_min // dt)
total_minutes = 24 * 60
time_bins = np.arange(dt, total_minutes + dt, dt)
nbins = len(time_bins)

model_order = ['persistence', 'bergman', 'lstm', 'ml_corrector', 'pinn']
color_map = {
    'persistence': 'tab:orange',
    'bergman': 'tab:green',
    'lstm': 'tab:blue',
    'ml_corrector': 'tab:red',
    'pinn': 'purple'
}

selected_sids = val_ids[:min(3, len(val_ids))]  # First 3 validation subjects

for sid in selected_sids:
    info = val_sequences.get(sid)
    if info is None:
        continue
    
    y_val_subj = info['y']
    if y_val_subj.shape[0] == 0:
        continue
    
    y_test_arr = np.asarray(y_val_subj)
    if y_test_arr.ndim == 3 and y_test_arr.shape[-1] == 1:
        y_test_flat = y_test_arr.reshape(y_test_arr.shape[0], y_test_arr.shape[1])
    else:
        y_test_flat = y_test_arr.copy()
    
    if y_test_flat.shape[1] < ph_steps:
        continue
    
    y_test_mg = inv_cgm_with_scaler(y_test_flat[:, :ph_steps])
    
    n_models = len(model_order)
    fig, axs = plt.subplots(2, n_models, figsize=(5*n_models, 9), sharex=True)
    if n_models == 1:
        axs = axs.reshape(2, 1)
    
    fig.suptitle(f"Subject {sid} — 30-min forecasts (validation, 24h view)", fontsize=14)
    
    for i_m, mname in enumerate(model_order):
        ax_top = axs[0, i_m]
        ax_bot = axs[1, i_m]
        
        if mname not in models:
            ax_top.text(0.5, 0.5, f"model '{mname}' not found", ha='center', va='center')
            ax_top.set_title(mname)
            ax_bot.set_visible(False)
            continue
        
        yhat_scaled_all = models[mname](sid)
        yhat_scaled_all = np.asarray(yhat_scaled_all)
        
        if yhat_scaled_all.size == 0:
            ax_top.text(0.5, 0.5, "no preds", ha='center', va='center')
            ax_top.set_title(mname)
            ax_bot.set_visible(False)
            continue
        
        if yhat_scaled_all.ndim == 3 and yhat_scaled_all.shape[-1] == 1:
            yhat_scaled_all = yhat_scaled_all.reshape(yhat_scaled_all.shape[0], 
                                                       yhat_scaled_all.shape[1])
        
        if yhat_scaled_all.shape[1] < ph_steps:
            continue
        
        yhat_mg_all = inv_cgm_with_scaler(yhat_scaled_all[:, :ph_steps])
        n_windows = yhat_mg_all.shape[0]
        
        # Compute predicted times for each window
        win_idx = np.arange(n_windows).reshape(-1, 1)
        step_idx = np.arange(1, ph_steps+1).reshape(1, -1)
        pred_times = (win_idx * dt) + (step_idx * dt)
        
        pred_times_flat = pred_times.ravel().astype(int)
        preds_flat = yhat_mg_all.ravel()
        actuals_flat = y_test_mg.ravel()
        
        valid_mask = (pred_times_flat >= dt) & (pred_times_flat <= total_minutes)
        bin_idx = (pred_times_flat // dt) - 1
        
        agg_pred = np.full(nbins, np.nan)
        agg_actual = np.full(nbins, np.nan)
        agg_rmse = np.full(nbins, np.nan)
        agg_counts = np.zeros(nbins, dtype=int)
        
        for b in range(nbins):
            idxs = np.where((bin_idx == b) & valid_mask)[0]
            c = idxs.size
            agg_counts[b] = c
            if c > 0:
                vals_pred = preds_flat[idxs].astype(float)
                vals_act = actuals_flat[idxs].astype(float)
                agg_pred[b] = np.nanmean(vals_pred)
                agg_actual[b] = np.nanmean(vals_act)
                agg_rmse[b] = np.sqrt(np.nanmean((vals_pred - vals_act)**2))
        
        # Interpolate NaN gaps
        def interp_nan(arr):
            arr = arr.astype(float)
            mask = ~np.isnan(arr)
            if mask.sum() == 0:
                return arr
            first, last = np.where(mask)[0][0], np.where(mask)[0][-1]
            x = np.arange(first, last+1)
            y = arr[first:last+1]
            valid = ~np.isnan(y)
            if valid.sum() <= 1:
                return arr
            xi = np.arange(first, last+1)
            arr[first:last+1] = np.interp(xi, x[valid], y[valid])
            return arr
        
        agg_pred_f = interp_nan(agg_pred)
        agg_actual_f = interp_nan(agg_actual)
        
        t_minutes = time_bins
        t_hours = t_minutes / 60.0
        color = color_map.get(mname, 'gray')
        
        # TOP: Continuous trajectories
        if np.all(np.isnan(agg_actual_f)) and np.all(np.isnan(agg_pred_f)):
            ax_top.text(0.5, 0.5, "no data in 24h window", ha='center', va='center')
        else:
            ax_top.plot(t_hours, agg_actual_f, '-', color='black', linewidth=1.6, 
                       label='Actual (avg)')
            ax_top.plot(t_hours, agg_pred_f, '-', color=color, linewidth=2.0, 
                       label='Predicted (avg, 30-min PH)')
            
            # Overlay last window's short-term forecast
            last_pred = yhat_mg_all[-1, :ph_steps]
            last_times = (np.arange(1, ph_steps+1) * dt) / 60.0
            ax_top.plot(last_times, last_pred, '--', color=color, linewidth=1.4, 
                       label='Last-window 30-min')
        
        ax_top.set_title(mname)
        if i_m == 0:
            ax_top.set_ylabel('Blood Glucose (mg/dL)')
        
        ax_top.set_xlim(0, total_minutes/60.0)
        xticks = np.arange(0, (total_minutes/60.0) + 0.1, 2)
        ax_top.set_xticks(xticks)
        
        if HYPO is not None:
            ax_top.axhline(HYPO, color='gray', linestyle=':', linewidth=0.8)
        
        all_vals = np.concatenate([
            agg_actual_f[~np.isnan(agg_actual_f)] if agg_actual_f is not None else np.array([]),
            agg_pred_f[~np.isnan(agg_pred_f)] if agg_pred_f is not None else np.array([])
        ])
        
        if all_vals.size > 0:
            lo, hi = np.nanpercentile(all_vals, [1, 99])
            pad = max(10, 0.05 * (hi - lo))
            ax_top.set_ylim(max(30, lo - pad), min(400, hi + pad))
        else:
            ax_top.set_ylim(40, 300)
        
        ax_top.grid(True, linestyle=':', alpha=0.3)
        ax_top.legend(fontsize=8)
        
        # BOTTOM: Just xlabel for consistency
        ax_bot.set_xlabel('Hours (since window start)')
        ax_bot.set_xlim(0, total_minutes/60.0)
        ax_bot.set_xticks(xticks)
        ax_bot.set_visible(False)  # Hide bottom panel for cleaner display
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# ============================================================================
# COMBINED PER-HORIZON RMSE PLOT
# ============================================================================

print("\n" + "="*70)
print("COMBINED PER-HORIZON RMSE PLOT")
print("="*70)

model_names = list(all_results.keys())
per_subject_rmse = {m: [] for m in model_names}

for sid in val_ids:
    info = val_sequences.get(sid)
    if info is None:
        continue
    
    y_val_subj = info['y']
    if y_val_subj.shape[0] == 0:
        continue
    
    y_true_mg_full = inv_cgm_with_scaler(y_val_subj.reshape(-1, N_OUT))
    
    if y_true_mg_full.shape[1] < ph_steps:
        continue
    
    y_true_mg = y_true_mg_full[:, :ph_steps]
    
    for m in model_names:
        yhat_scaled = models[m](sid)
        if np.asarray(yhat_scaled).size == 0:
            continue
        
        yhat_scaled = np.asarray(yhat_scaled)
        if yhat_scaled.ndim == 3 and yhat_scaled.shape[-1] == 1:
            yhat_scaled = yhat_scaled.reshape(yhat_scaled.shape[0], yhat_scaled.shape[1])
        
        if yhat_scaled.shape[1] < ph_steps:
            continue
        
        yhat_mg = inv_cgm_with_scaler(yhat_scaled)[:, :ph_steps]
        rmse_per_h = [np.sqrt(mean_squared_error(y_true_mg[:, h], yhat_mg[:, h])) 
                      for h in range(ph_steps)]
        per_subject_rmse[m].append(rmse_per_h)

plt.figure(figsize=(8, 5))
for m in model_names:
    subj_rmses = np.array(per_subject_rmse[m]) if len(per_subject_rmse[m]) > 0 else np.zeros((0, ph_steps))
    if subj_rmses.size == 0:
        continue
    
    mean_rmse = np.nanmean(subj_rmses, axis=0)
    plt.plot(np.arange(1, ph_steps+1)*5, mean_rmse, '-o', label=m, linewidth=2)

plt.xticks(np.arange(1, ph_steps+1)*5)
plt.xlabel('Minutes ahead')
plt.ylabel('RMSE (mg/dL)')
plt.title('Per-horizon RMSE averaged across validation subjects (up to 30 min)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("EVALUATION COMPLETE")
print("="*70)
print(f"""
Summary:
  ✓ Evaluated on validation data from OhioT1DM
  ✓ {len(val_ids)} validation subjects
  ✓ {y_true_all.shape[0]} total validation windows
  ✓ All metrics computed: RMSE, MAE, g-RMSE, g-MAE, TG, G-Mean
  ✓ Clarke & Parkes Error Grid analysis
  ✓ Glycemia detection confusion matrices
  ✓ Multi-step trajectory visualizations
  ✓ Per-horizon performance comparison
""")

## Evaluation - Testing - BrisT1D Dataset:

In [ ]:
"""
Comprehensive Model Evaluation on Test Data
Evaluates LSTM, Bergman, PINN, and ML Corrector models using test subjects from BrisT1D
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from collections import Counter
import math
from scipy.signal import correlate
import warnings

# ============================================================================
# SETUP: Prepare test data
# ============================================================================

print("="*70)
print("PREPARING TEST DATA (BrisT1D)")
print("="*70)

# Test subjects and sequences from BrisT1D dataset
test_subject_ids = sorted(test_subjects.keys())
print(f"Test subjects: {len(test_subject_ids)}")
print(f"Test sequences: {X_test.shape[0]} windows")

# ============================================================================
# HELPER FUNCTIONS (from evaluation code)
# ============================================================================

def inv_cgm_with_scaler(y_scaled):
    """Convert scaled CGM values back to mg/dL"""
    if y_scaled is None or np.asarray(y_scaled).size == 0:
        return np.array([]).reshape(0, N_OUT)
    arr = np.asarray(y_scaled)
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr.reshape(arr.shape[0], arr.shape[1])
    flat = arr.reshape(-1, 1)
    inv = cgm_scaler.inverse_transform(flat).reshape(-1, N_OUT)
    return inv

HYPO = 70.0
HYPER = 180.0

def glycemia_class(x):
    """Classify glucose values: 0=hypo (<70), 1=normo (70-180), 2=hyper (>180)"""
    x = np.asarray(x).reshape(-1)
    cls = np.full(x.shape, -1, dtype=int)
    cls[x < HYPO] = 0
    cls[(x >= HYPO) & (x <= HYPER)] = 1
    cls[x > HYPER] = 2
    return cls

def temporal_gain(y_true, y_pred, prediction_horizon):
    """
    Temporal Gain (TG): median time gained by prediction before delay degrades accuracy
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    horizon = int(N_OUT)
    dt_minutes = float(prediction_horizon) / float(horizon)
    
    if y_true.ndim == 1 and y_pred.ndim == 1 and (y_true.size % horizon == 0):
        y_true = y_true.reshape(-1, horizon)
        y_pred = y_pred.reshape(-1, horizon)
    
    if y_true.ndim != 2 or y_pred.ndim != 2:
        return int(prediction_horizon)
    
    n_windows, L = y_true.shape
    max_lag_steps = int(prediction_horizon // dt_minutes)
    max_lag_steps = min(max_lag_steps, L - 1)
    
    tg_windows = []
    for i in range(n_windows):
        t = y_true[i].astype(float)
        p = y_pred[i].astype(float)
        
        if np.all(np.isnan(t)) or np.all(np.isnan(p)):
            tg_windows.append(0)
            continue
        
        if np.nanstd(t) < 1e-8 or np.nanstd(p) < 1e-8:
            tg_windows.append(0)
            continue
        
        t0 = t - np.nanmean(t)
        p0 = p - np.nanmean(p)
        
        cc = correlate(t0, p0, mode='full')
        full_lags = np.arange(-L + 1, L)
        
        valid_mask = (full_lags >= 0) & (full_lags <= max_lag_steps)
        valid_lags = full_lags[valid_mask]
        valid_cc = cc[valid_mask]
        
        if valid_cc.size == 0:
            tg_windows.append(0)
            continue
        
        idx = np.argmax(np.abs(valid_cc))
        delay_steps = int(valid_lags[idx])
        delay_minutes = delay_steps * dt_minutes
        tg_val = prediction_horizon - delay_minutes
        tg_val = max(0.0, min(float(prediction_horizon), float(tg_val)))
        tg_windows.append(tg_val)
    
    if len(tg_windows) == 0:
        return int(prediction_horizon)
    return int(round(np.median(tg_windows)))

def g_mean(y_true, y_pred):
    """Geometric mean of per-class recall"""
    from sklearn.metrics import recall_score
    t = glycemia_class(y_true)
    p = glycemia_class(y_pred)
    recalls = recall_score(t, p, average=None, zero_division=0)
    recalls = np.where(recalls == 0, 1e-10, recalls)
    return float(np.exp(np.mean(np.log(recalls))))

def _sigma_ge(x, a, eps):
    """Smooth step function (C2 continuous)"""
    x = np.asarray(x, dtype=float)
    a = np.asarray(a, dtype=float)
    out = np.zeros_like(x, dtype=float)
    
    if np.asarray(eps).size == 1 and eps <= 0:
        out = (x > a).astype(float)
        return out
    
    left = x <= a
    right = x >= (a + eps)
    mid = (~left) & (~right)
    
    out[right] = 1.0
    
    if np.any(mid):
        mid_vals = x[mid]
        a_arr = np.asarray(a)
        if a_arr.shape == ():
            a_mid = a_arr
        else:
            a_b = np.broadcast_to(a_arr, x.shape)
            a_mid = a_b[mid]
        
        xi = 2.0 * (mid_vals - a_mid) / eps - 1.0
        first_half = xi <= 0
        second_half = ~first_half
        
        val = np.empty_like(xi)
        if np.any(first_half):
            xi_f = xi[first_half]
            val[first_half] = (-0.5 * xi_f**4) - (xi_f**3) + xi_f + 0.5
        if np.any(second_half):
            xi_s = xi[second_half]
            val[second_half] = (0.5 * xi_s**4) - (xi_s**3) + xi_s + 0.5
        
        out[mid] = val
    
    return out

def _sigma_le_bar(x, a, eps):
    """Smooth step function (mirror of _sigma_ge)"""
    x = np.asarray(x, dtype=float)
    a = np.asarray(a, dtype=float)
    out = np.zeros_like(x, dtype=float)
    
    if np.asarray(eps).size == 1 and eps <= 0:
        out = (x <= a).astype(float)
        return out
    
    left = x <= (a - eps)
    right = x >= a
    mid = (~left) & (~right)
    
    out[left] = 1.0
    
    if np.any(mid):
        mid_vals = x[mid]
        a_arr = np.asarray(a)
        if a_arr.shape == ():
            a_mid = a_arr
        else:
            a_b = np.broadcast_to(a_arr, x.shape)
            a_mid = a_b[mid]
        
        xi = 2.0 * (mid_vals - a_mid) / eps + 1.0
        first_half = xi <= 0
        second_half = ~first_half
        
        val = np.empty_like(xi)
        if np.any(first_half):
            xi_f = xi[first_half]
            val[first_half] = (0.5 * xi_f**4) - (xi_f**3) + xi_f + 0.5
        if np.any(second_half):
            xi_s = xi[second_half]
            val[second_half] = (-0.5 * xi_s**4) - (xi_s**3) + xi_s + 0.5
        
        out[mid] = val
    
    return out

def pen_function(g, ghat, alphaL=1.5, alphaH=1.0, TL=85.0, TH=155.0,
                 betaL=30.0, betaH=30.0, gammaL=40.0, gammaH=40.0):
    """Penalty function for glucose-weighted metrics"""
    g = np.asarray(g, dtype=float)
    ghat = np.asarray(ghat, dtype=float)
    
    if g.shape == () and ghat.shape != ():
        g = np.full_like(ghat, float(g))
    elif ghat.shape == () and g.shape != ():
        ghat = np.full_like(g, float(ghat))
    else:
        bshape = np.broadcast_shapes(g.shape, ghat.shape)
        g = np.broadcast_to(g, bshape).astype(float)
        ghat = np.broadcast_to(ghat, bshape).astype(float)
    
    partL = _sigma_le_bar(g, TL, betaL) * _sigma_ge(ghat, g, gammaL)
    partH = _sigma_ge(g, TH, betaH) * _sigma_le_bar(ghat, g, gammaH)
    Pen = 1.0 + alphaL * partL + alphaH * partH
    return Pen

def glucose_weighted_rmse(y_true, y_pred, alphaL=1.5, alphaH=1.0, TL=85.0, TH=155.0,
                          betaL=30.0, betaH=30.0, gammaL=40.0, gammaH=40.0):
    """Glucose-weighted RMSE"""
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    if y_true.size == 0:
        return np.nan
    
    Pen = pen_function(y_true, y_pred, alphaL, alphaH, TL, TH, betaL, betaH, gammaL, gammaH)
    gse = (y_true - y_pred)**2 * Pen
    return math.sqrt(np.mean(gse))

def glucose_weighted_mae(y_true, y_pred, alphaL=1.5, alphaH=1.0, TL=85.0, TH=155.0,
                         betaL=30.0, betaH=30.0, gammaL=40.0, gammaH=40.0):
    """Glucose-weighted MAE"""
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    if y_true.size == 0:
        return np.nan
    
    Pen = pen_function(y_true, y_pred, alphaL, alphaH, TL, TH, betaL, betaH, gammaL, gammaH)
    gabs = np.abs(y_true - y_pred) * Pen
    return float(np.mean(gabs))

def clarke_zone_for_point(ref, pred):
    """Clarke Error Grid zone classification"""
    if np.isnan(ref) or np.isnan(pred):
        return None
    if (ref >= 70 and abs(pred - ref) <= 0.2 * ref) or (ref < 70 and abs(pred - ref) <= 20):
        return 'A'
    if (pred >= 330 and ref <= 50) or (pred <= 50 and ref >= 330):
        return 'E'
    if ref < 70 and pred >= 180:
        return 'C'
    if ref >= 180 and pred < 70:
        return 'D'
    return 'B'

def parkes_zone_for_point(ref, pred):
    """Parkes (Consensus) Error Grid zone classification"""
    if np.isnan(ref) or np.isnan(pred):
        return None
    err = pred - ref
    rel_err = abs(err) / max(ref, 1.0)
    if (ref < 70 and abs(err) <= 15) or (ref >= 70 and rel_err <= 0.2):
        return 'A'
    if rel_err <= 0.35:
        return 'B'
    if (ref < 70 and pred > 180) or (ref > 180 and pred < 70):
        return 'D'
    if rel_err > 0.6:
        return 'E'
    return 'C'

def compute_glycemia_confusion(y_true, y_pred):
    """Compute confusion matrix for glycemia detection"""
    y_t = np.asarray(y_true).reshape(-1)
    y_p = np.asarray(y_pred).reshape(-1)
    mask = np.isfinite(y_t) & np.isfinite(y_p)
    y_t = y_t[mask]
    y_p = y_p[mask]
    
    counts = np.zeros((3, 3), dtype=int)
    if y_t.size == 0:
        return counts, np.full_like(counts, np.nan, dtype=float)
    
    tcls = glycemia_class(y_t)
    pcls = glycemia_class(y_p)
    
    for t, p in zip(tcls, pcls):
        if 0 <= t <= 2 and 0 <= p <= 2:
            counts[t, p] += 1
    
    pct = np.full_like(counts, np.nan, dtype=float)
    for i in range(3):
        s = counts[i].sum()
        if s > 0:
            pct[i, :] = counts[i, :] / float(s)
    
    return counts, pct

def plot_confusion_heatmaps(counts, pct, title=""):
    """Plot glycemia detection confusion matrix heatmap"""
    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    im = ax.imshow(pct, vmin=0.0, vmax=1.0, cmap='Blues', interpolation='nearest')
    ax.set_xticks(np.arange(3))
    ax.set_yticks(np.arange(3))
    labels = ['Pred Hypo\n(<70)', 'Pred Normo\n(70-180)', 'Pred Hyper\n(>180)']
    ax.set_xticklabels(labels)
    ax.set_yticklabels(['True Hypo', 'True Normo', 'True Hyper'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    
    for i in range(3):
        for j in range(3):
            c = counts[i, j]
            p = pct[i, j]
            if np.isnan(p):
                txt = "n/a\n(0)"
            else:
                txt = f"{p*100:4.1f}%\n({c})"
            ax.text(j, i, txt, ha='center', va='center', color='black', fontsize=10)
    
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

# ============================================================================
# MODEL PREDICTION FUNCTIONS FOR TEST DATA
# ============================================================================

def persistence_predict_on_test_subject(sid):
    """Persistence baseline: last observed value repeated"""
    info = test_sequences.get(sid)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    
    X_subj = info['X']
    n_windows = X_subj.shape[0]
    
    # Last CGM value from input sequence (already scaled)
    last_cgm = X_subj[:, -1, 0]  # Shape: (n_windows,)
    
    # Repeat for all forecast steps
    preds_scaled = np.tile(last_cgm.reshape(-1, 1), (1, N_OUT))
    return preds_scaled.reshape(n_windows, N_OUT, 1)

def bergman_predict_on_test_subject(sid):
    """Bergman physiological model predictions"""
    info = test_sequences.get(sid)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    
    X_subj = info['X']
    subj_df = test_subjects[sid]
    n_windows = X_subj.shape[0]
    
    # Initial glucose values (mg/dL) for each window
    G_inits = subj_df['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
    
    # Extract insulin and carbs for forecast horizon
    bolus_mat = np.zeros((n_windows, N_OUT))
    carbs_mat = np.zeros((n_windows, N_OUT))
    
    for w in range(n_windows):
        start_idx = N_IN + w
        bolus_mat[w, :] = subj_df['bolus'].values[start_idx:start_idx+N_OUT]
        carbs_mat[w, :] = subj_df['carbs'].values[start_idx:start_idx+N_OUT]
    
    # Bergman parameters (should be defined from bergman_params variable)
    p1, p2, p3, k_i, Gb = bergman_params
    dt_scale = 1.0  # 5min/5min scaling factor
    
    # Simulate forward
    preds_mg = np.zeros((n_windows, N_OUT))
    G = G_inits.copy().astype(float)
    I = np.zeros(n_windows)
    
    for t in range(N_OUT):
        # Update insulin with decay and new dose
        I = I * np.exp(-k_i) + bolus_mat[:, t]
        
        # Meal effect
        meal_effect = p3 * carbs_mat[:, t]
        
        # Glucose dynamics
        dG = -p1 * (G - Gb) - p2 * I + meal_effect
        G = G + dG * dt_scale
        preds_mg[:, t] = G
    
    # Scale to [0, 1]
    preds_scaled = cgm_scaler.transform(preds_mg.reshape(-1, 1)).reshape(n_windows, N_OUT)
    return preds_scaled.reshape(n_windows, N_OUT, 1)

def lstm_predict_on_test_subject(sid):
    """LSTM model predictions"""
    info = test_sequences.get(sid)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    
    X_subj = info['X']
    preds_scaled = lstm_model.predict(X_subj, batch_size=64, verbose=0)
    
    if preds_scaled.ndim == 2:
        preds_scaled = preds_scaled.reshape(-1, N_OUT, 1)
    
    return preds_scaled

def pinn_predict_on_test_subject(sid):
    """Physics-Informed Neural Network predictions"""
    info = test_sequences.get(sid)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    
    X_subj = info['X'].astype(np.float32)
    preds_scaled, _ = pinn_model(X_subj, training=False)
    preds_scaled = preds_scaled.numpy()
    
    if preds_scaled.ndim == 2:
        preds_scaled = preds_scaled.reshape(-1, N_OUT, 1)
    
    return preds_scaled

def ml_corrector_predict_on_test_subject(sid):
    """Hybrid ML corrector: Bergman + learned residual"""
    info = test_sequences.get(sid)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    
    X_subj = info['X']
    
    # Get residual predictions from corrector model
    resid_scaled = lstm_corrector.predict(X_subj, batch_size=128, verbose=0)
    
    # Get Bergman baseline predictions
    bergman_preds = bergman_predict_on_test_subject(sid)
    bergman_preds_scaled = bergman_preds.reshape(-1, N_OUT)
    
    # Correct Bergman with residual
    corrected_scaled = bergman_preds_scaled + resid_scaled
    return corrected_scaled.reshape(-1, N_OUT, 1)

# ============================================================================
# COLLECT PREDICTIONS FROM ALL MODELS
# ============================================================================

print("\n" + "="*70)
print("GENERATING PREDICTIONS ON TEST DATA (BrisT1D)")
print("="*70)

models = {
    'persistence': persistence_predict_on_test_subject,
    'bergman': bergman_predict_on_test_subject,
    'lstm': lstm_predict_on_test_subject,
    'pinn': pinn_predict_on_test_subject,
    'ml_corrector': ml_corrector_predict_on_test_subject
}

# Collect predictions for all test subjects
all_results = {}
for mname, mpredict in models.items():
    print(f"\nModel: {mname}")
    Ys = []
    for sid in test_subject_ids:
        yhat_scaled = mpredict(sid)
        yhat_scaled = np.asarray(yhat_scaled)
        
        if yhat_scaled.ndim == 3 and yhat_scaled.shape[-1] == 1:
            yhat_scaled = yhat_scaled.reshape(yhat_scaled.shape[0], yhat_scaled.shape[1])
        
        yhat_mg = inv_cgm_with_scaler(yhat_scaled)
        Ys.append(yhat_mg)
        print(f"  Subject {sid}: {yhat_mg.shape[0]} windows")
    
    all_results[mname] = np.vstack(Ys) if len(Ys) else np.zeros((0, N_OUT))
    print(f"  Total: {all_results[mname].shape[0]} windows")

# Build ground truth (all test windows)
y_true_all_windows = []
for sid in test_subject_ids:
    info = test_sequences.get(sid)
    y_test_subj = info['y']
    if y_test_subj.shape[0] == 0:
        continue
    y_true_mg = inv_cgm_with_scaler(y_test_subj.reshape(-1, N_OUT))
    y_true_all_windows.append(y_true_mg)

if len(y_true_all_windows) == 0:
    raise RuntimeError("No test windows found.")

y_true_all = np.vstack(y_true_all_windows)
print(f"\nGround truth: {y_true_all.shape[0]} windows")

# ============================================================================
# EVALUATION METRICS
# ============================================================================

print("\n" + "="*70)
print("EVALUATION METRICS (TEST DATA)")
print("="*70)

# Per-horizon RMSE/MAE
horiz_steps = np.arange(1, N_OUT+1) * 5
print("\n=== Per-horizon RMSE / MAE for each model ===")

per_horizon = {}
for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        print(f"\nModel {mname}: no predictions")
        continue
    
    rmse_h = []
    mae_h = []
    
    for h in range(N_OUT):
        y_t = y_true_all[:, h]
        y_p = ypred_all[:, h]
        rmse_h.append(np.sqrt(mean_squared_error(y_t, y_p)))
        mae_h.append(mean_absolute_error(y_t, y_p))
    
    per_horizon[mname] = {'rmse': rmse_h, 'mae': mae_h}
    
    print(f"\nModel: {mname}")
    print("Horizon (min):", list(horiz_steps))
    print("RMSE (mg/dL): ", [round(v, 2) for v in rmse_h])
    print("MAE  (mg/dL): ", [round(v, 2) for v in mae_h])

# Final-step aggregated metrics
print("\n=== Final-step aggregated metrics (30 min) ===")

for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        continue
    
    ref = y_true_all[:, -1]
    pred = ypred_all[:, -1]
    
    rmse_val = np.sqrt(mean_squared_error(ref, pred))
    mae_val = mean_absolute_error(ref, pred)
    gw_rmse = glucose_weighted_rmse(ref, pred)
    gw_mae = glucose_weighted_mae(ref, pred)
    tg = temporal_gain(y_true_all, ypred_all, prediction_horizon=5*N_OUT)
    gm = g_mean(ref, pred)
    
    print(f"\nModel: {mname}")
    print(f"  Windows: {len(ref)}")
    print(f"  RMSE: {rmse_val:.3f}  MAE: {mae_val:.3f}")
    print(f"  Glucose-weighted RMSE: {gw_rmse:.3f}  Glucose-weighted MAE: {gw_mae:.3f}")
    print(f"  Temporal gain (min): {tg}   G-Mean: {gm:.4f}")

# ============================================================================
# CLARKE & PARKES ERROR GRIDS + SCATTER PLOTS
# ============================================================================

print("\n" + "="*70)
print("CLARKE & PARKES ERROR GRIDS")
print("="*70)

for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        continue
    
    ref = y_true_all[:, -1]
    pred = ypred_all[:, -1]
    
    # Compute zones
    cz = [clarke_zone_for_point(r, p) for r, p in zip(ref, pred)]
    pz = [parkes_zone_for_point(r, p) for r, p in zip(ref, pred)]
    
    c_counts = dict(Counter(cz))
    p_counts = dict(Counter(pz))
    total = len(ref)
    
    print(f"\n{mname} Clarke zones: ", {k: (c_counts[k], round(100*c_counts[k]/total, 2)) 
                                         for k in c_counts})
    print(f"{mname} Parkes zones : ", {k: (p_counts[k], round(100*p_counts[k]/total, 2)) 
                                        for k in p_counts})
    
    # Scatter plots
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
    # Clarke grid
    ax0 = ax[0]
    ax0.scatter(ref, pred, s=6, alpha=0.5)
    ax0.plot([0, 400], [0, 400], '--', linewidth=0.8, color='gray')
    xs = np.linspace(0, 400, 100)
    ax0.plot(xs, 1.2*xs, linewidth=0.6, color='gray')
    ax0.plot(xs, 0.8*xs, linewidth=0.6, color='gray')
    ax0.axvline(70, linestyle=':', color='gray')
    ax0.axhline(70, linestyle=':', color='gray')
    ax0.set_xlim(0, 400)
    ax0.set_ylim(0, 400)
    ax0.set_xlabel('Reference (mg/dL)')
    ax0.set_ylabel('Predicted (mg/dL)')
    ax0.set_title(f"{mname} — Clarke (final step)")
    ax0.grid(True, alpha=0.3)
    
    # Parkes grid
    ax1 = ax[1]
    ax1.scatter(ref, pred, s=6, alpha=0.5)
    ax1.plot([0, 400], [0, 400], '--', linewidth=0.8, color='gray')
    ax1.set_xlim(0, 400)
    ax1.set_ylim(0, 400)
    ax1.set_xlabel('Reference (mg/dL)')
    ax1.set_ylabel('Predicted (mg/dL)')
    ax1.set_title(f"{mname} — Parkes (visual)")
    ax1.grid(True, alpha=0.3)
    
    plt.suptitle(f"{mname} Pred vs Ref (final step, test) — N={total}")
    plt.tight_layout()
    plt.show()

# ============================================================================
# GLYCEMIA DETECTION CONFUSION MATRICES
# ============================================================================

print("\n" + "="*70)
print("GLYCEMIA DETECTION CONFUSION MATRICES")
print("="*70)

for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        continue
    
    counts, pct = compute_glycemia_confusion(y_true_all[:, -1], ypred_all[:, -1])
    print(f"\nGlycemia detection — {mname} (final step):")
    plot_confusion_heatmaps(counts, pct, title=f"{mname} — Glycemia detection (test)")

# ============================================================================
# MULTI-STEP TRAJECTORY PLOTS (Selected Subjects, 24h view)
# ============================================================================

print("\n" + "="*70)
print("MULTI-STEP TRAJECTORY PLOTS (24 hour view)")
print("="*70)

dt = 5  # minutes
prediction_horizon_min = 30
ph_steps = int(prediction_horizon_min // dt)
total_minutes = 24 * 60
time_bins = np.arange(dt, total_minutes + dt, dt)
nbins = len(time_bins)

model_order = ['persistence', 'bergman', 'lstm', 'ml_corrector', 'pinn']
color_map = {
    'persistence': 'tab:orange',
    'bergman': 'tab:green',
    'lstm': 'tab:blue',
    'ml_corrector': 'tab:red',
    'pinn': 'purple'
}

selected_sids = test_subject_ids[:min(3, len(test_subject_ids))]  # First 3 test subjects

for sid in selected_sids:
    info = test_sequences.get(sid)
    if info is None:
        continue
    
    y_test_subj = info['y']
    if y_test_subj.shape[0] == 0:
        continue
    
    y_test_arr = np.asarray(y_test_subj)
    if y_test_arr.ndim == 3 and y_test_arr.shape[-1] == 1:
        y_test_flat = y_test_arr.reshape(y_test_arr.shape[0], y_test_arr.shape[1])
    else:
        y_test_flat = y_test_arr.copy()
    
    if y_test_flat.shape[1] < ph_steps:
        continue
    
    y_test_mg = inv_cgm_with_scaler(y_test_flat[:, :ph_steps])
    
    n_models = len(model_order)
    fig, axs = plt.subplots(2, n_models, figsize=(5*n_models, 9), sharex=True)
    if n_models == 1:
        axs = axs.reshape(2, 1)
    
    fig.suptitle(f"Subject {sid} — 30-min forecasts (test, 24h view)", fontsize=14)
    
    for i_m, mname in enumerate(model_order):
        ax_top = axs[0, i_m]
        ax_bot = axs[1, i_m]
        
        if mname not in models:
            ax_top.text(0.5, 0.5, f"model '{mname}' not found", ha='center', va='center')
            ax_top.set_title(mname)
            ax_bot.set_visible(False)
            continue
        
        yhat_scaled_all = models[mname](sid)
        yhat_scaled_all = np.asarray(yhat_scaled_all)
        
        if yhat_scaled_all.size == 0:
            ax_top.text(0.5, 0.5, "no preds", ha='center', va='center')
            ax_top.set_title(mname)
            ax_bot.set_visible(False)
            continue
        
        if yhat_scaled_all.ndim == 3 and yhat_scaled_all.shape[-1] == 1:
            yhat_scaled_all = yhat_scaled_all.reshape(yhat_scaled_all.shape[0], 
                                                       yhat_scaled_all.shape[1])
        
        if yhat_scaled_all.shape[1] < ph_steps:
            continue
        
        yhat_mg_all = inv_cgm_with_scaler(yhat_scaled_all[:, :ph_steps])
        n_windows = yhat_mg_all.shape[0]
        
        # Compute predicted times for each window
        win_idx = np.arange(n_windows).reshape(-1, 1)
        step_idx = np.arange(1, ph_steps+1).reshape(1, -1)
        pred_times = (win_idx * dt) + (step_idx * dt)
        
        pred_times_flat = pred_times.ravel().astype(int)
        preds_flat = yhat_mg_all.ravel()
        actuals_flat = y_test_mg.ravel()
        
        valid_mask = (pred_times_flat >= dt) & (pred_times_flat <= total_minutes)
        bin_idx = (pred_times_flat // dt) - 1
        
        agg_pred = np.full(nbins, np.nan)
        agg_actual = np.full(nbins, np.nan)
        agg_rmse = np.full(nbins, np.nan)
        agg_counts = np.zeros(nbins, dtype=int)
        
        for b in range(nbins):
            idxs = np.where((bin_idx == b) & valid_mask)[0]
            c = idxs.size
            agg_counts[b] = c
            if c > 0:
                vals_pred = preds_flat[idxs].astype(float)
                vals_act = actuals_flat[idxs].astype(float)
                agg_pred[b] = np.nanmean(vals_pred)
                agg_actual[b] = np.nanmean(vals_act)
                agg_rmse[b] = np.sqrt(np.nanmean((vals_pred - vals_act)**2))
        
        # Interpolate NaN gaps
        def interp_nan(arr):
            arr = arr.astype(float)
            mask = ~np.isnan(arr)
            if mask.sum() == 0:
                return arr
            first, last = np.where(mask)[0][0], np.where(mask)[0][-1]
            x = np.arange(first, last+1)
            y = arr[first:last+1]
            valid = ~np.isnan(y)
            if valid.sum() <= 1:
                return arr
            xi = np.arange(first, last+1)
            arr[first:last+1] = np.interp(xi, x[valid], y[valid])
            return arr
        
        agg_pred_f = interp_nan(agg_pred)
        agg_actual_f = interp_nan(agg_actual)
        
        t_minutes = time_bins
        t_hours = t_minutes / 60.0
        color = color_map.get(mname, 'gray')
        
        # TOP: Continuous trajectories
        if np.all(np.isnan(agg_actual_f)) and np.all(np.isnan(agg_pred_f)):
            ax_top.text(0.5, 0.5, "no data in 24h window", ha='center', va='center')
        else:
            ax_top.plot(t_hours, agg_actual_f, '-', color='black', linewidth=1.6, 
                       label='Actual (avg)')
            ax_top.plot(t_hours, agg_pred_f, '-', color=color, linewidth=2.0, 
                       label='Predicted (avg, 30-min PH)')
            
            # Overlay last window's short-term forecast
            last_pred = yhat_mg_all[-1, :ph_steps]
            last_times = (np.arange(1, ph_steps+1) * dt) / 60.0
            ax_top.plot(last_times, last_pred, '--', color=color, linewidth=1.4, 
                       label='Last-window 30-min')
        
        ax_top.set_title(mname)
        if i_m == 0:
            ax_top.set_ylabel('Blood Glucose (mg/dL)')
        
        ax_top.set_xlim(0, total_minutes/60.0)
        xticks = np.arange(0, (total_minutes/60.0) + 0.1, 2)
        ax_top.set_xticks(xticks)
        
        if HYPO is not None:
            ax_top.axhline(HYPO, color='gray', linestyle=':', linewidth=0.8)
        
        all_vals = np.concatenate([
            agg_actual_f[~np.isnan(agg_actual_f)] if agg_actual_f is not None else np.array([]),
            agg_pred_f[~np.isnan(agg_pred_f)] if agg_pred_f is not None else np.array([])
        ])
        
        if all_vals.size > 0:
            lo, hi = np.nanpercentile(all_vals, [1, 99])
            pad = max(10, 0.05 * (hi - lo))
            ax_top.set_ylim(max(30, lo - pad), min(400, hi + pad))
        else:
            ax_top.set_ylim(40, 300)
        
        ax_top.grid(True, linestyle=':', alpha=0.3)
        ax_top.legend(fontsize=8)
        
        # BOTTOM: Just xlabel for consistency
        ax_bot.set_xlabel('Hours (since window start)')
        ax_bot.set_xlim(0, total_minutes/60.0)
        ax_bot.set_xticks(xticks)
        ax_bot.set_visible(False)  # Hide bottom panel for cleaner display
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# ============================================================================
# COMBINED PER-HORIZON RMSE PLOT
# ============================================================================

print("\n" + "="*70)
print("COMBINED PER-HORIZON RMSE PLOT")
print("="*70)

model_names = list(all_results.keys())
per_subject_rmse = {m: [] for m in model_names}

for sid in test_subject_ids:
    info = test_sequences.get(sid)
    if info is None:
        continue
    
    y_test_subj = info['y']
    if y_test_subj.shape[0] == 0:
        continue
    
    y_true_mg_full = inv_cgm_with_scaler(y_test_subj.reshape(-1, N_OUT))
    
    if y_true_mg_full.shape[1] < ph_steps:
        continue
    
    y_true_mg = y_true_mg_full[:, :ph_steps]
    
    for m in model_names:
        yhat_scaled = models[m](sid)
        if np.asarray(yhat_scaled).size == 0:
            continue
        
        yhat_scaled = np.asarray(yhat_scaled)
        if yhat_scaled.ndim == 3 and yhat_scaled.shape[-1] == 1:
            yhat_scaled = yhat_scaled.reshape(yhat_scaled.shape[0], yhat_scaled.shape[1])
        
        if yhat_scaled.shape[1] < ph_steps:
            continue
        
        yhat_mg = inv_cgm_with_scaler(yhat_scaled)[:, :ph_steps]
        rmse_per_h = [np.sqrt(mean_squared_error(y_true_mg[:, h], yhat_mg[:, h])) 
                      for h in range(ph_steps)]
        per_subject_rmse[m].append(rmse_per_h)

plt.figure(figsize=(8, 5))
for m in model_names:
    subj_rmses = np.array(per_subject_rmse[m]) if len(per_subject_rmse[m]) > 0 else np.zeros((0, ph_steps))
    if subj_rmses.size == 0:
        continue
    
    mean_rmse = np.nanmean(subj_rmses, axis=0)
    plt.plot(np.arange(1, ph_steps+1)*5, mean_rmse, '-o', label=m, linewidth=2)

plt.xticks(np.arange(1, ph_steps+1)*5)
plt.xlabel('Minutes ahead')
plt.ylabel('RMSE (mg/dL)')
plt.title('Per-horizon RMSE averaged across test subjects (up to 30 min)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("EVALUATION COMPLETE (TEST DATA)")
print("="*70)
print(f"""
Summary:
  ✓ Evaluated on test data from BrisT1D
  ✓ {len(test_subject_ids)} test subjects
  ✓ {y_true_all.shape[0]} total test windows
  ✓ All metrics computed: RMSE, MAE, g-RMSE, g-MAE, TG, G-Mean
  ✓ Clarke & Parkes Error Grid analysis
  ✓ Glycemia detection confusion matrices
  ✓ Multi-step trajectory visualizations
  ✓ Per-horizon performance comparison
  
Note: Models trained on OhioT1DM, evaluated on BrisT1D (cross-dataset generalization)
""")

## Feature Importance - SHAP Analysis (Only For machine learning based models):

In [ ]:
"""
SHAP Feature Importance Analysis for Blood Glucose Models
Analyzes three models: LSTM, Hybrid Residual Corrector, PINN
Features: CGM, carbs, bolus, heartrate, steps, CGM_derivative, carbs_derivative
"""

import numpy as np
import matplotlib.pyplot as plt
import shap
import tensorflow as tf # ADDED: Required for Keras model cloning
import os # ADDED: Required for Keras model cloning (though not directly used with clone_model)
import copy # ADDED: Used for safety, though clone_model is preferred for Keras

# ============================================================================
# CONFIGURATION
# ============================================================================

# Define feature names based on data processing
feature_names = [
    'CGM',
    'Carbs',
    'Bolus Insulin',
    'Heart Rate',
    'Steps',
    'CGM Derivative',
    'Carbs Derivative'
]

# Models to analyze (set to True for models you want to analyze)
ANALYZE_LSTM = True
ANALYZE_HYBRID = True
ANALYZE_PINN = True

# Model variable names (update these based on your variable names)
# For Hybrid: the Bergman model should be from BergmanMinimalModel (pure Python/NumPy)
# For PINN: uses a different BergmanModel with TensorFlow
HYBRID_BERGMAN_VAR = 'bergman'   # Change if your hybrid bergman has a different name
PINN_MODEL_VAR = 'pinn_model'    # Change if your PINN model has a different name

# SHAP settings
BACKGROUND_SIZE = 3
EXPLAIN_SIZE = 5
NSAMPLES = 10

print("="*70)
print("SHAP FEATURE IMPORTANCE ANALYSIS")
print("="*70)
print(f"Features: {feature_names}")
print(f"Background samples: {BACKGROUND_SIZE}")
print(f"Samples to explain: {EXPLAIN_SIZE}")

# ============================================================================
# MODEL AVAILABILITY CHECK
# ============================================================================

print("\n" + "="*70)
print("CHECKING MODEL AVAILABILITY")
print("="*70)

available_models = {}

# Check LSTM
if 'lstm_model' in globals():
    available_models['LSTM'] = True
    print("✓ LSTM model found (lstm_model)")
else:
    available_models['LSTM'] = False
    print("✗ LSTM model not found")

# Check Hybrid components
hybrid_ok = False
if 'bergman' in globals() and 'lstm_corrector' in globals():
    bergman_obj = globals()['bergman']
    # Check if it's the correct type (NumPy-based, not TensorFlow)
    if hasattr(bergman_obj, 'p1_bg'):
        hybrid_ok = True
        available_models['Hybrid'] = True
        print("✓ Hybrid model components found (bergman + lstm_corrector)")
    else:
        available_models['Hybrid'] = False
        print("✗ Hybrid: bergman variable is TensorFlow-based (PINN version)")
        print("  The Hybrid model needs BergmanMinimalModel")
        print("  Solution: Re-run the Hybrid training code to recreate it")
else:
    available_models['Hybrid'] = False
    if 'bergman' not in globals():
        print("✗ Hybrid: 'bergman' variable not found")
    if 'lstm_corrector' not in globals():
        print("✗ Hybrid: 'lstm_corrector' variable not found")

# Check PINN
if 'pinn_model' in globals():
    available_models['PINN'] = True
    print("✓ PINN model found (pinn_model)")
else:
    available_models['PINN'] = False
    print("✗ PINN model not found")

print("\n" + "="*70)

# Update analysis flags based on availability
if not available_models.get('LSTM', False):
    ANALYZE_LSTM = False
    print("! Skipping LSTM analysis (model not available)")

if not available_models.get('Hybrid', False):
    ANALYZE_HYBRID = False
    print("! Skipping Hybrid analysis (model not available)")

if not available_models.get('PINN', False):
    ANALYZE_PINN = False
    print("! Skipping PINN analysis (model not available)")

if not any([ANALYZE_LSTM, ANALYZE_HYBRID, ANALYZE_PINN]):
    raise RuntimeError("No models available for analysis! Please train models first.")

print(f"\nWill analyze: {[k for k, v in [('LSTM', ANALYZE_LSTM), ('Hybrid', ANALYZE_HYBRID), ('PINN', ANALYZE_PINN)] if v]}")
print("="*70)

# ============================================================================
# SELECT DATA FOR SHAP ANALYSIS
# ============================================================================

# Select background dataset
# Assuming X_train is available globally
background_indices = np.random.choice(X_train.shape[0], BACKGROUND_SIZE, replace=False)
background_data = X_train[background_indices]

# Select samples to explain
# Assuming X_val is available globally
explain_indices = np.random.choice(X_val.shape[0], EXPLAIN_SIZE, replace=False)
explain_data = X_val[explain_indices]

print(f"Input shape: {explain_data.shape}")

# Store original dimensions
n_timesteps = explain_data.shape[1]
n_features = explain_data.shape[2]
# Assuming N_IN and N_OUT are globally defined

# ============================================================================
# PREDICTION FUNCTIONS FOR EACH MODEL
# ============================================================================

def create_lstm_predict_fn(model):
    """Create prediction function for vanilla LSTM"""
    def predict_fn(X):
        if len(X.shape) == 2:
            n_samples = X.shape[0]
            X = X.reshape(n_samples, n_timesteps, n_features)
        preds = model.predict(X, verbose=0, batch_size=32)
        return np.mean(preds, axis=1, keepdims=True)
    return predict_fn


def create_hybrid_predict_fn(bergman, lstm_corrector, val_subjects_dict):
    """
    Create prediction function for Hybrid Residual Corrector
    Maps validation samples to their corresponding subject data
    """
    # Create mapping from global validation index to (subject_id, window_index)
    sample_mapping = []
    global_idx = 0
    
    # Assuming val_sequences is available globally
    for sid in sorted(val_subjects_dict.keys()):
        subj_data = val_subjects_dict[sid]
        seq_dict = val_sequences[sid]
        n_samples = seq_dict['X'].shape[0]
        
        if n_samples == 0:
            continue
        
        for window_idx in range(n_samples):
            sample_mapping.append((sid, window_idx))
            global_idx += 1
    
    # Pre-compute sequences for explain_indices
    G0_array = []
    bolus_array = []
    carbs_array = []
    
    for val_idx in explain_indices:
        if val_idx >= len(sample_mapping):
            # Handle edge case - use last valid sample
            val_idx = len(sample_mapping) - 1
        
        sid, window_idx = sample_mapping[val_idx]
        # Assuming train_subjects is available globally to get data
        # WARNING: If val_subjects_dict is derived from train_subjects, use val_subjects_dict[sid]
        subj_data = val_subjects_dict[sid] 
        
        # Assuming N_IN and N_OUT are globally defined
        window_start = N_IN + window_idx
        G_init = subj_data['CGM_smoothed'].values[window_start - 1]
        bolus = subj_data['bolus'].values[window_start:window_start + N_OUT]
        carbs = subj_data['carbs'].values[window_start:window_start + N_OUT]
        
        G0_array.append(G_init)
        bolus_array.append(bolus)
        carbs_array.append(carbs)
    
    G0_array = np.array(G0_array)
    bolus_array = np.array(bolus_array)
    carbs_array = np.array(carbs_array)
    
    print(f"  Hybrid: Prepared {len(G0_array)} sample mappings")
    
    def predict_fn(X):
        if len(X.shape) == 2:
            n_samples = X.shape[0]
            X = X.reshape(n_samples, n_timesteps, n_features)
        
        # Assuming cgm_scaler is available globally
        
        # Get Bergman predictions
        predictions = []
        for i in range(X.shape[0]):
            # Use modulo to handle any size mismatches
            idx = i % len(G0_array)
            
            berg_pred = bergman.simulate(G0_array[idx], bolus_array[idx], carbs_array[idx])
            berg_pred_scaled = cgm_scaler.transform(berg_pred.reshape(-1, 1)).flatten()
            
            # Get LSTM residual correction
            x_input = X[i:i+1]
            residual_pred = lstm_corrector.predict(x_input, verbose=0)[0]
            
            # Combine
            y_hybrid = berg_pred_scaled + residual_pred
            predictions.append(np.mean(y_hybrid))
        
        return np.array(predictions).reshape(-1, 1)
    
    return predict_fn


def create_pinn_predict_fn(pinn_model):
    """Create prediction function for PINN model"""
    def predict_fn(X):
        if len(X.shape) == 2:
            n_samples = X.shape[0]
            X = X.reshape(n_samples, n_timesteps, n_features)
        
        X_tf = X.astype(np.float32)
        # Assuming pinn_model returns (predictions_scaled, something_else)
        preds_scaled, _ = pinn_model(X_tf, training=False) 
        preds_scaled = preds_scaled.numpy()
        return np.mean(preds_scaled, axis=1, keepdims=True)
    
    return predict_fn

# ============================================================================
# SHAP ANALYSIS FUNCTION
# ============================================================================

def compute_shap_analysis(predict_fn, model_name):
    """
    Compute SHAP analysis for a given model
    Returns dictionary with SHAP results
    """
    print(f"\n{'='*70}")
    print(f"Analyzing {model_name}")
    print(f"{'='*70}")
    
    # Flatten data
    background_flat = background_data.reshape(BACKGROUND_SIZE, -1)
    explain_flat = explain_data.reshape(EXPLAIN_SIZE, -1)
    
    # Create explainer
    print("Initializing KernelExplainer...")
    explainer = shap.KernelExplainer(predict_fn, background_flat)
    
    # Compute SHAP values
    print("Computing SHAP values...")
    shap_values_flat = explainer.shap_values(explain_flat, nsamples=NSAMPLES, silent=False)
    
    # Reshape back
    # Assuming shap_values_flat is a list with one element (for single output model)
    if isinstance(shap_values_flat, list):
        shap_values_flat = shap_values_flat[0]
        
    shap_values = shap_values_flat.reshape(EXPLAIN_SIZE, n_timesteps, n_features)
    print(f"SHAP values shape: {shap_values.shape}")
    
    # Aggregate SHAP values
    mean_abs_shap = np.mean(np.abs(shap_values), axis=(0, 1))
    sum_abs_shap_time = np.sum(np.abs(shap_values), axis=1)
    mean_sum_abs_shap = np.mean(sum_abs_shap_time, axis=0)
    
    # Normalize
    mean_abs_shap_norm = mean_abs_shap / np.sum(mean_abs_shap)
    mean_sum_abs_shap_norm = mean_sum_abs_shap / np.sum(mean_sum_abs_shap)
    
    # Temporal importance
    temporal_importance = np.mean(np.abs(shap_values), axis=0)
    
    # Time window analysis
    segment_size = n_timesteps // 3
    recent_importance = np.mean(temporal_importance[-segment_size:], axis=0)
    middle_importance = np.mean(temporal_importance[segment_size:-segment_size], axis=0)
    distant_importance = np.mean(temporal_importance[:segment_size], axis=0)
    
    recent_norm = recent_importance / np.sum(recent_importance)
    middle_norm = middle_importance / np.sum(middle_importance)
    distant_norm = distant_importance / np.sum(distant_importance)
    
    print(f"✓ SHAP analysis complete for {model_name}")
    
    return {
        'shap_values': shap_values,
        'mean_abs_shap': mean_abs_shap_norm,
        'mean_sum_abs_shap': mean_sum_abs_shap_norm,
        'temporal_importance': temporal_importance,
        'temporal_windows': {
            'distant': distant_norm,
            'middle': middle_norm,
            'recent': recent_norm
        }
    }

# ============================================================================
# RUN SHAP ANALYSIS FOR EACH MODEL
# ============================================================================

results = {}

# Analyze LSTM model
if ANALYZE_LSTM:
    try:
        # --- START: Deep Copy for Safety ---
        # Create a deep copy of the Keras model to prevent graph corruption by SHAP
        lstm_model_safe = tf.keras.models.clone_model(lstm_model)
        lstm_model_safe.set_weights(lstm_model.get_weights())
        # --- END: Deep Copy for Safety ---
        
        lstm_predict_fn = create_lstm_predict_fn(lstm_model_safe)
        results['LSTM'] = compute_shap_analysis(lstm_predict_fn, "Vanilla LSTM")
    except Exception as e:
        print(f"Error analyzing LSTM: {e}")

# Analyze Hybrid Residual Corrector
if ANALYZE_HYBRID:
    try:
        # Check if the correct Bergman model exists
        hybrid_bergman = None
        hybrid_corrector = None
        
        # Try multiple possible variable names
        bergman_candidates = ['hybrid_bergman', 'bergman_hybrid', 'bergman']
        corrector_candidates = ['hybrid_lstm', 'lstm_corrector', 'hybrid_corrector']
        
        # Find Bergman model
        for var_name in bergman_candidates:
            if var_name in globals():
                bergman_obj = globals()[var_name]
                # Check if it's the NumPy-based version (has p1_bg attribute)
                if hasattr(bergman_obj, 'p1_bg'):
                    hybrid_bergman = bergman_obj
                    print(f"  Found NumPy-based Bergman model: '{var_name}'")
                    break
        
        # Find LSTM corrector (Keras model)
        for var_name in corrector_candidates:
            if var_name in globals():
                hybrid_corrector = globals()[var_name]
                print(f"  Found LSTM corrector: '{var_name}'")
                break
        
        if hybrid_bergman is None:
            raise ValueError(
                "Hybrid Bergman model not found. Tried: " + ", ".join(bergman_candidates) +
                "\nPlease ensure BergmanMinimalModel exists (not PINN's TensorFlow version)"
            )
        
        if hybrid_corrector is None:
            raise ValueError(
                "Hybrid LSTM corrector not found. Tried: " + ", ".join(corrector_candidates)
            )

        # --- START: Deep Copy for Safety ---
        # Create a deep copy of the LSTM corrector model before passing to SHAP
        hybrid_corrector_safe = tf.keras.models.clone_model(hybrid_corrector)
        hybrid_corrector_safe.set_weights(hybrid_corrector.get_weights())
        # --- END: Deep Copy for Safety ---
        
        # Assuming train_subjects and val_ids are available globally
        val_subjects_dict = {sid: train_subjects[sid] for sid in val_ids}
        
        # Pass the safe copy of the corrector model
        hybrid_predict_fn = create_hybrid_predict_fn(hybrid_bergman, hybrid_corrector_safe, val_subjects_dict)
        results['Hybrid'] = compute_shap_analysis(hybrid_predict_fn, "Hybrid Residual Corrector")
    except Exception as e:
        print(f"Error analyzing Hybrid model: {e}")
        import traceback
        traceback.print_exc()
        print("\nTroubleshooting tips:")
        print("  1. Make sure you've run the Hybrid Residual Corrector training code")
        print("  2. The 'bergman' variable should be BergmanMinimalModel (not PINN's BergmanModel)")
        print("  3. The 'lstm_corrector' variable should exist")
        print("  4. If you trained PINN after Hybrid, the bergman variable may have been overwritten")
        print("    -> Re-run the Hybrid training to recreate the correct bergman model")

# Analyze PINN model
if ANALYZE_PINN:
    try:
        # --- START: Deep Copy for Safety ---
        # Create a deep copy of the Keras model to prevent graph corruption by SHAP
        pinn_model_safe = tf.keras.models.clone_model(pinn_model)
        pinn_model_safe.set_weights(pinn_model.get_weights())
        # --- END: Deep Copy for Safety ---
        
        pinn_predict_fn = create_pinn_predict_fn(pinn_model_safe)
        results['PINN'] = compute_shap_analysis(pinn_predict_fn, "PINN")
    except Exception as e:
        print(f"Error analyzing PINN: {e}")
        print("Make sure pinn_model is trained and available")

# ============================================================================
# VISUALIZATION 1: COMPARISON ACROSS MODELS
# ============================================================================

if len(results) > 0:
    print(f"\n{'='*70}")
    print("GENERATING VISUALIZATIONS")
    print(f"{'='*70}")
    
    # Feature importance comparison
    fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 7))
    if len(results) == 1:
        axes = [axes]
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
    y_pos = np.arange(len(feature_names))
    
    for idx, (model_name, data) in enumerate(results.items()):
        ax = axes[idx]
        bars = ax.barh(y_pos, data['mean_abs_shap'], color=colors, 
                         alpha=0.8, edgecolor='black', linewidth=0.5)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(feature_names, fontsize=10)
        ax.set_xlabel('Importance Score', fontsize=11, fontweight='bold')
        ax.set_title(f'{model_name}\nFeature Importance', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        
        # Add values
        for i, bar in enumerate(bars):
            width = bar.get_width()
            ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
                   f'{data["mean_abs_shap"][i]:.3f}',
                   ha='left', va='center', fontsize=9)
    
    plt.suptitle('Feature Importance Comparison Across Models', 
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig('shap_models_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: shap_models_comparison.png")

# ============================================================================
# VISUALIZATION 2: INDIVIDUAL MODEL PLOTS
# ============================================================================

for model_name, data in results.items():
    # Bar plot for each model
    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(y_pos, data['mean_abs_shap'], color=colors, 
                    alpha=0.8, edgecolor='black', linewidth=0.5)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feature_names, fontsize=11)
    ax.set_xlabel('Mean |SHAP| Value (Normalized)', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name}: Feature Importance\nBlood Glucose Forecasting', 
                  fontsize=14, fontweight='bold', pad=15)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
               f'{data["mean_abs_shap"][i]:.3f}',
               ha='left', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    filename = f'shap_{model_name.lower().replace(" ", "_")}_importance.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")
    
    # Temporal heatmap
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(data['temporal_importance'].T, aspect='auto', 
                    cmap='YlOrRd', interpolation='nearest')
    
    ax.set_yticks(range(len(feature_names)))
    ax.set_yticklabels(feature_names, fontsize=11)
    ax.set_xlabel('Timestep (5-min intervals)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name}: Temporal Feature Importance\n(60-min history for 30-min forecast)', 
                  fontsize=13, fontweight='bold', pad=15)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Mean |SHAP| Value', fontsize=11, fontweight='bold')
    
    tick_positions = range(0, n_timesteps, 3)
    tick_labels = [f'-{(n_timesteps-i)*5}min' for i in tick_positions]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=9)
    
    plt.tight_layout()
    filename = f'shap_{model_name.lower().replace(" ", "_")}_temporal.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")

# ============================================================================
# VISUALIZATION 3: TEMPORAL WINDOW COMPARISON
# ============================================================================

if len(results) > 1:
    fig, axes = plt.subplots(len(results), 1, figsize=(14, 5*len(results)))
    if len(results) == 1:
        axes = [axes]
    
    x = np.arange(len(feature_names))
    width = 0.25
    
    for idx, (model_name, data) in enumerate(results.items()):
        ax = axes[idx]
        
        bars1 = ax.bar(x - width, data['temporal_windows']['distant'], width, 
                       label='Distant (0-20 min)', color='lightblue', 
                       edgecolor='black', linewidth=0.5)
        bars2 = ax.bar(x, data['temporal_windows']['middle'], width, 
                       label='Middle (20-40 min)', color='steelblue', 
                       edgecolor='black', linewidth=0.5)
        bars3 = ax.bar(x + width, data['temporal_windows']['recent'], width, 
                       label='Recent (40-60 min)', color='darkblue', 
                       edgecolor='black', linewidth=0.5)
        
        ax.set_ylabel('Normalized Importance', fontsize=11, fontweight='bold')
        ax.set_xlabel('Features', fontsize=11, fontweight='bold')
        ax.set_title(f'{model_name}: Importance by Time Window', 
                     fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=10)
        ax.legend(fontsize=10, loc='upper right')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.suptitle('Temporal Window Analysis Across Models', 
                 fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('shap_temporal_windows_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: shap_temporal_windows_comparison.png")

# ============================================================================
# VISUALIZATION 4: SUMMARY PLOTS FOR EACH MODEL
# ============================================================================

for model_name, data in results.items():
    shap_values_reshaped = data['shap_values'].reshape(-1, len(feature_names))
    explain_data_reshaped = explain_data.reshape(-1, len(feature_names))
    
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values_reshaped, explain_data_reshaped, 
                      feature_names=feature_names, show=False, 
                      max_display=len(feature_names))
    plt.title(f'{model_name}: SHAP Summary Plot\n(Red = High Value, Blue = Low Value)', 
              fontsize=13, fontweight='bold', pad=20)
    plt.tight_layout()
    filename = f'shap_{model_name.lower().replace(" ", "_")}_summary.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")

# ============================================================================
# FEATURE IMPORTANCE RANKINGS
# ============================================================================

print(f"\n{'='*70}")
print("FEATURE IMPORTANCE RANKINGS")
print(f"{'='*70}")

for model_name, data in results.items():
    print(f"\n{model_name}:")
    print("-" * 70)
    
    indices = np.argsort(data['mean_abs_shap'])[::-1]
    for rank, idx in enumerate(indices, 1):
        print(f"  {rank}. {feature_names[idx]:20s} {data['mean_abs_shap'][idx]:.4f} "
              f"({data['mean_abs_shap'][idx]*100:.2f}%)")

# ============================================================================
# CROSS-MODEL COMPARISON TABLE
# ============================================================================

if len(results) > 1:
    print(f"\n{'='*70}")
    print("CROSS-MODEL FEATURE IMPORTANCE COMPARISON")
    print(f"{'='*70}")
    print(f"\n{'Feature':<20}", end="")
    for model_name in results.keys():
        print(f"{model_name:>15}", end="")
    print()
    print("-" * (20 + 15 * len(results)))
    
    for i, fname in enumerate(feature_names):
        print(f"{fname:<20}", end="")
        for model_name in results.keys():
            importance = results[model_name]['mean_abs_shap'][i]
            print(f"{importance:>15.4f}", end="")
        print()
    
    # Feature group analysis
    print(f"\n{'='*70}")
    print("FEATURE GROUP ANALYSIS")
    print(f"{'='*70}")
    
    direct_features = ['CGM', 'Carbs', 'Bolus Insulin', 'Heart Rate', 'Steps']
    derivative_features = ['CGM Derivative', 'Carbs Derivative']
    
    for model_name, data in results.items():
        print(f"\n{model_name}:")
        
        direct_indices = [i for i, name in enumerate(feature_names) if name in direct_features]
        derivative_indices = [i for i, name in enumerate(feature_names) if name in derivative_features]
        
        direct_importance = sum(data['mean_abs_shap'][i] for i in direct_indices)
        derivative_importance = sum(data['mean_abs_shap'][i] for i in derivative_indices)
        
        print(f"  Direct Measurements:  {direct_importance:.4f} ({direct_importance*100:.2f}%)")
        print(f"  Derivatives:          {derivative_importance:.4f} ({derivative_importance*100:.2f}%)")

# ============================================================================
# SAVE RESULTS
# ============================================================================

shap_results = {
    'feature_names': feature_names,
    'models': results,
    'explain_data': explain_data,
    'background_data': background_data
}

print(f"\n{'='*70}")
print("SHAP ANALYSIS COMPLETE!")
print(f"{'='*70}")
print(f"""
Analyzed models: {', '.join(results.keys())}

Generated visualizations:
  - shap_models_comparison.png (comparison across all models)
  - shap_[model]_importance.png (individual importance plots)
  - shap_[model]_temporal.png (temporal heatmaps)
  - shap_[model]_summary.png (SHAP summary plots)
  - shap_temporal_windows_comparison.png (time window analysis)

Results stored in 'shap_results' dictionary.

Key Insights:
  - Compare which features matter most across different architectures
  - Identify if physics-informed models rely more on certain features
  - Understand temporal dynamics for each model type
  - See how hybrid approaches balance data-driven vs physics-based features
""")

# ============================================================================
# TROUBLESHOOTING GUIDE
# ============================================================================

print(f"{'='*70}")
print("TROUBLESHOOTING GUIDE (UPDATED)")
print(f"{'='*70}")

print("""
**IMPORTANT CHANGE:**
The code now creates a **safe deep copy** of all Keras/TensorFlow models 
(`lstm_model`, `hybrid_corrector`, `pinn_model`) using `tf.keras.models.clone_model` 
before running SHAP analysis. This prevents the previous graph corruption 
(`LookupError: gradient registry has no entry for: shap_AddV2`).
""")

if 'Hybrid' not in results and ANALYZE_HYBRID:
    print("""
HYBRID MODEL ISSUE:
If the Hybrid model analysis failed, it's likely because the 'bergman' 
variable was overwritten by PINN training.

SOLUTION - Add this code RIGHT AFTER training the Hybrid model:
    
    # Save Hybrid model components before PINN training
    hybrid_bergman = bergman  # Save the NumPy-based Bergman
    hybrid_lstm = lstm_corrector  # Save the LSTM corrector
    
Then, when running SHAP analysis, the code will use 'hybrid_bergman' 
instead of 'bergman'.

Alternatively, just re-run the Hybrid training code to recreate the 
'bergman' and 'lstm_corrector' variables before running this analysis.
""")

print("""
GENERAL TIPS:
1. Train models in any order, but be aware that variable names may overlap
2. For Hybrid: Ensure 'bergman' and 'lstm_corrector' exist and are correct types
3. For PINN: Ensure 'pinn_model' exists
4. For LSTM: Ensure 'lstm_model' exists
5. If analyzing multiple models, consider saving each with unique names

RECOMMENDED WORKFLOW (For Clean Variable Names):
# After training each model, save with unique names:
lstm_trained = lstm_model
hybrid_bergman_model = bergman
hybrid_lstm_model = lstm_corrector  
pinn_trained = pinn_model

# Then update variable names in SHAP analysis configuration section (not strictly necessary with the variable search, but safer)
""")

In [ ]:
"""
SHAP Feature Importance Analysis for Blood Glucose Models
Analyzes three models: LSTM, Hybrid Residual Corrector, PINN
Features: CGM, carbs, bolus, heartrate, steps, CGM_derivative, carbs_derivative
"""

import numpy as np
import matplotlib.pyplot as plt
import shap

# ============================================================================
# CONFIGURATION
# ============================================================================

# Define feature names based on data processing
feature_names = [
    'CGM',
    'Carbs',
    'Bolus Insulin',
    'Heart Rate',
    'Steps',
    'CGM Derivative',
    'Carbs Derivative'
]

# Models to analyze (set to True for models you want to analyze)
ANALYZE_LSTM = True
ANALYZE_HYBRID = True
ANALYZE_PINN = True

# Model variable names (update these based on your variable names)
# For Hybrid: the Bergman model should be from BergmanMinimalModel (pure Python/NumPy)
# For PINN: uses a different BergmanModel with TensorFlow
HYBRID_BERGMAN_VAR = 'bergman'  # Change if your hybrid bergman has a different name
PINN_MODEL_VAR = 'pinn_model'   # Change if your PINN model has a different name

# SHAP settings
BACKGROUND_SIZE = 3
EXPLAIN_SIZE = 5
NSAMPLES = 10

print("="*70)
print("SHAP FEATURE IMPORTANCE ANALYSIS")
print("="*70)
print(f"Features: {feature_names}")
print(f"Background samples: {BACKGROUND_SIZE}")
print(f"Samples to explain: {EXPLAIN_SIZE}")

# ============================================================================
# MODEL AVAILABILITY CHECK
# ============================================================================

print("\n" + "="*70)
print("CHECKING MODEL AVAILABILITY")
print("="*70)

available_models = {}

# Check LSTM
if 'lstm_model' in globals():
    available_models['LSTM'] = True
    print("✓ LSTM model found (lstm_model)")
else:
    available_models['LSTM'] = False
    print("✗ LSTM model not found")

# Check Hybrid components
hybrid_ok = False
if 'bergman' in globals() and 'lstm_corrector' in globals():
    bergman_obj = globals()['bergman']
    # Check if it's the correct type (NumPy-based, not TensorFlow)
    if hasattr(bergman_obj, 'p1_bg'):
        hybrid_ok = True
        available_models['Hybrid'] = True
        print("✓ Hybrid model components found (bergman + lstm_corrector)")
    else:
        available_models['Hybrid'] = False
        print("✗ Hybrid: bergman variable is TensorFlow-based (PINN version)")
        print("  The Hybrid model needs BergmanMinimalModel")
        print("  Solution: Re-run the Hybrid training code to recreate it")
else:
    available_models['Hybrid'] = False
    if 'bergman' not in globals():
        print("✗ Hybrid: 'bergman' variable not found")
    if 'lstm_corrector' not in globals():
        print("✗ Hybrid: 'lstm_corrector' variable not found")

# Check PINN
if 'pinn_model' in globals():
    available_models['PINN'] = True
    print("✓ PINN model found (pinn_model)")
else:
    available_models['PINN'] = False
    print("✗ PINN model not found")

print("\n" + "="*70)

# Update analysis flags based on availability
if not available_models.get('LSTM', False):
    ANALYZE_LSTM = False
    print("! Skipping LSTM analysis (model not available)")

if not available_models.get('Hybrid', False):
    ANALYZE_HYBRID = False
    print("! Skipping Hybrid analysis (model not available)")

if not available_models.get('PINN', False):
    ANALYZE_PINN = False
    print("! Skipping PINN analysis (model not available)")

if not any([ANALYZE_LSTM, ANALYZE_HYBRID, ANALYZE_PINN]):
    raise RuntimeError("No models available for analysis! Please train models first.")

print(f"\nWill analyze: {[k for k, v in [('LSTM', ANALYZE_LSTM), ('Hybrid', ANALYZE_HYBRID), ('PINN', ANALYZE_PINN)] if v]}")
print("="*70)

# ============================================================================
# SELECT DATA FOR SHAP ANALYSIS
# ============================================================================

# Select background dataset
background_indices = np.random.choice(X_train.shape[0], BACKGROUND_SIZE, replace=False)
background_data = X_train[background_indices]

# Select samples to explain
explain_indices = np.random.choice(X_val.shape[0], EXPLAIN_SIZE, replace=False)
explain_data = X_val[explain_indices]

print(f"Input shape: {explain_data.shape}")

# Store original dimensions
n_timesteps = explain_data.shape[1]
n_features = explain_data.shape[2]

# ============================================================================
# PREDICTION FUNCTIONS FOR EACH MODEL
# ============================================================================

def create_lstm_predict_fn(model):
    """Create prediction function for vanilla LSTM"""
    def predict_fn(X):
        if len(X.shape) == 2:
            n_samples = X.shape[0]
            X = X.reshape(n_samples, n_timesteps, n_features)
        preds = model.predict(X, verbose=0, batch_size=32)
        return np.mean(preds, axis=1, keepdims=True)
    return predict_fn


def create_hybrid_predict_fn(bergman, lstm_corrector, val_subjects_dict):
    """
    Create prediction function for Hybrid Residual Corrector
    Maps validation samples to their corresponding subject data
    """
    # Create mapping from global validation index to (subject_id, window_index)
    sample_mapping = []
    global_idx = 0
    
    for sid in sorted(val_subjects_dict.keys()):
        subj_data = val_subjects_dict[sid]
        seq_dict = val_sequences[sid]
        n_samples = seq_dict['X'].shape[0]
        
        if n_samples == 0:
            continue
        
        for window_idx in range(n_samples):
            sample_mapping.append((sid, window_idx))
            global_idx += 1
    
    # Pre-compute sequences for explain_indices
    G0_array = []
    bolus_array = []
    carbs_array = []
    
    for val_idx in explain_indices:
        if val_idx >= len(sample_mapping):
            # Handle edge case - use last valid sample
            val_idx = len(sample_mapping) - 1
        
        sid, window_idx = sample_mapping[val_idx]
        subj_data = val_subjects_dict[sid]
        
        window_start = N_IN + window_idx
        G_init = subj_data['CGM_smoothed'].values[window_start - 1]
        bolus = subj_data['bolus'].values[window_start:window_start + N_OUT]
        carbs = subj_data['carbs'].values[window_start:window_start + N_OUT]
        
        G0_array.append(G_init)
        bolus_array.append(bolus)
        carbs_array.append(carbs)
    
    G0_array = np.array(G0_array)
    bolus_array = np.array(bolus_array)
    carbs_array = np.array(carbs_array)
    
    print(f"  Hybrid: Prepared {len(G0_array)} sample mappings")
    
    def predict_fn(X):
        if len(X.shape) == 2:
            n_samples = X.shape[0]
            X = X.reshape(n_samples, n_timesteps, n_features)
        
        # Get Bergman predictions
        predictions = []
        for i in range(X.shape[0]):
            # Use modulo to handle any size mismatches
            idx = i % len(G0_array)
            
            berg_pred = bergman.simulate(G0_array[idx], bolus_array[idx], carbs_array[idx])
            berg_pred_scaled = cgm_scaler.transform(berg_pred.reshape(-1, 1)).flatten()
            
            # Get LSTM residual correction
            x_input = X[i:i+1]
            residual_pred = lstm_corrector.predict(x_input, verbose=0)[0]
            
            # Combine
            y_hybrid = berg_pred_scaled + residual_pred
            predictions.append(np.mean(y_hybrid))
        
        return np.array(predictions).reshape(-1, 1)
    
    return predict_fn


def create_pinn_predict_fn(pinn_model):
    """Create prediction function for PINN model"""
    def predict_fn(X):
        if len(X.shape) == 2:
            n_samples = X.shape[0]
            X = X.reshape(n_samples, n_timesteps, n_features)
        
        X_tf = X.astype(np.float32)
        preds_scaled, _ = pinn_model(X_tf, training=False)
        preds_scaled = preds_scaled.numpy()
        return np.mean(preds_scaled, axis=1, keepdims=True)
    
    return predict_fn

# ============================================================================
# SHAP ANALYSIS FUNCTION
# ============================================================================

def compute_shap_analysis(predict_fn, model_name):
    """
    Compute SHAP analysis for a given model
    Returns dictionary with SHAP results
    """
    print(f"\n{'='*70}")
    print(f"Analyzing {model_name}")
    print(f"{'='*70}")
    
    # Flatten data
    background_flat = background_data.reshape(BACKGROUND_SIZE, -1)
    explain_flat = explain_data.reshape(EXPLAIN_SIZE, -1)
    
    # Create explainer
    print("Initializing KernelExplainer...")
    explainer = shap.KernelExplainer(predict_fn, background_flat)
    
    # Compute SHAP values
    print("Computing SHAP values...")
    shap_values_flat = explainer.shap_values(explain_flat, nsamples=NSAMPLES, silent=False)
    
    # Reshape back
    shap_values = shap_values_flat.reshape(EXPLAIN_SIZE, n_timesteps, n_features)
    print(f"SHAP values shape: {shap_values.shape}")
    
    # Aggregate SHAP values
    mean_abs_shap = np.mean(np.abs(shap_values), axis=(0, 1))
    sum_abs_shap_time = np.sum(np.abs(shap_values), axis=1)
    mean_sum_abs_shap = np.mean(sum_abs_shap_time, axis=0)
    
    # Normalize
    mean_abs_shap_norm = mean_abs_shap / np.sum(mean_abs_shap)
    mean_sum_abs_shap_norm = mean_sum_abs_shap / np.sum(mean_sum_abs_shap)
    
    # Temporal importance
    temporal_importance = np.mean(np.abs(shap_values), axis=0)
    
    # Time window analysis
    segment_size = n_timesteps // 3
    recent_importance = np.mean(temporal_importance[-segment_size:], axis=0)
    middle_importance = np.mean(temporal_importance[segment_size:-segment_size], axis=0)
    distant_importance = np.mean(temporal_importance[:segment_size], axis=0)
    
    recent_norm = recent_importance / np.sum(recent_importance)
    middle_norm = middle_importance / np.sum(middle_importance)
    distant_norm = distant_importance / np.sum(distant_importance)
    
    print(f"✓ SHAP analysis complete for {model_name}")
    
    return {
        'shap_values': shap_values,
        'mean_abs_shap': mean_abs_shap_norm,
        'mean_sum_abs_shap': mean_sum_abs_shap_norm,
        'temporal_importance': temporal_importance,
        'temporal_windows': {
            'distant': distant_norm,
            'middle': middle_norm,
            'recent': recent_norm
        }
    }

# ============================================================================
# RUN SHAP ANALYSIS FOR EACH MODEL
# ============================================================================

results = {}

# Analyze LSTM model
if ANALYZE_LSTM:
    try:
        lstm_predict_fn = create_lstm_predict_fn(lstm_model)
        results['LSTM'] = compute_shap_analysis(lstm_predict_fn, "Vanilla LSTM")
    except Exception as e:
        print(f"Error analyzing LSTM: {e}")

# Analyze Hybrid Residual Corrector
if ANALYZE_HYBRID:
    try:
        # Check if the correct Bergman model exists
        # The Hybrid model needs BergmanMinimalModel (pure Python/NumPy)
        # NOT the TensorFlow-based BergmanModel from PINN
        
        # Look for the hybrid-specific bergman model
        hybrid_bergman = None
        hybrid_corrector = None
        
        # Try multiple possible variable names
        bergman_candidates = ['hybrid_bergman', 'bergman_hybrid', 'bergman']
        corrector_candidates = ['hybrid_lstm', 'lstm_corrector', 'hybrid_corrector']
        
        # Find Bergman model
        for var_name in bergman_candidates:
            if var_name in globals():
                bergman_obj = globals()[var_name]
                # Check if it's the NumPy-based version (has p1_bg attribute)
                if hasattr(bergman_obj, 'p1_bg'):
                    hybrid_bergman = bergman_obj
                    print(f"  Found NumPy-based Bergman model: '{var_name}'")
                    break
        
        # Find LSTM corrector
        for var_name in corrector_candidates:
            if var_name in globals():
                hybrid_corrector = globals()[var_name]
                print(f"  Found LSTM corrector: '{var_name}'")
                break
        
        if hybrid_bergman is None:
            raise ValueError(
                "Hybrid Bergman model not found. Tried: " + ", ".join(bergman_candidates) +
                "\nPlease ensure BergmanMinimalModel exists (not PINN's TensorFlow version)"
            )
        
        if hybrid_corrector is None:
            raise ValueError(
                "Hybrid LSTM corrector not found. Tried: " + ", ".join(corrector_candidates)
            )
        
        # Create validation subjects dict
        val_subjects_dict = {sid: train_subjects[sid] for sid in val_ids}
        hybrid_predict_fn = create_hybrid_predict_fn(hybrid_bergman, hybrid_corrector, val_subjects_dict)
        results['Hybrid'] = compute_shap_analysis(hybrid_predict_fn, "Hybrid Residual Corrector")
    except Exception as e:
        print(f"Error analyzing Hybrid model: {e}")
        import traceback
        traceback.print_exc()
        print("\nTroubleshooting tips:")
        print("  1. Make sure you've run the Hybrid Residual Corrector training code")
        print("  2. The 'bergman' variable should be BergmanMinimalModel (not PINN's BergmanModel)")
        print("  3. The 'lstm_corrector' variable should exist")
        print("  4. If you trained PINN after Hybrid, the bergman variable may have been overwritten")
        print("     -> Re-run the Hybrid training to recreate the correct bergman model")

# Analyze PINN model
if ANALYZE_PINN:
    try:
        pinn_predict_fn = create_pinn_predict_fn(pinn_model)
        results['PINN'] = compute_shap_analysis(pinn_predict_fn, "PINN")
    except Exception as e:
        print(f"Error analyzing PINN: {e}")
        print("Make sure pinn_model is trained and available")

# ============================================================================
# VISUALIZATION 1: COMPARISON ACROSS MODELS
# ============================================================================

if len(results) > 0:
    print(f"\n{'='*70}")
    print("GENERATING VISUALIZATIONS")
    print(f"{'='*70}")
    
    # Feature importance comparison
    fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 7))
    if len(results) == 1:
        axes = [axes]
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
    y_pos = np.arange(len(feature_names))
    
    for idx, (model_name, data) in enumerate(results.items()):
        ax = axes[idx]
        bars = ax.barh(y_pos, data['mean_abs_shap'], color=colors, 
                       alpha=0.8, edgecolor='black', linewidth=0.5)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(feature_names, fontsize=10)
        ax.set_xlabel('Importance Score', fontsize=11, fontweight='bold')
        ax.set_title(f'{model_name}\nFeature Importance', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        
        # Add values
        for i, bar in enumerate(bars):
            width = bar.get_width()
            ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
                   f'{data["mean_abs_shap"][i]:.3f}',
                   ha='left', va='center', fontsize=9)
    
    plt.suptitle('Feature Importance Comparison Across Models', 
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig('shap_models_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: shap_models_comparison.png")

# ============================================================================
# VISUALIZATION 2: INDIVIDUAL MODEL PLOTS
# ============================================================================

for model_name, data in results.items():
    # Bar plot for each model
    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(y_pos, data['mean_abs_shap'], color=colors, 
                   alpha=0.8, edgecolor='black', linewidth=0.5)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feature_names, fontsize=11)
    ax.set_xlabel('Mean |SHAP| Value (Normalized)', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name}: Feature Importance\nBlood Glucose Forecasting', 
                 fontsize=14, fontweight='bold', pad=15)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
               f'{data["mean_abs_shap"][i]:.3f}',
               ha='left', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    filename = f'shap_{model_name.lower().replace(" ", "_")}_importance.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")
    
    # Temporal heatmap
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(data['temporal_importance'].T, aspect='auto', 
                   cmap='YlOrRd', interpolation='nearest')
    
    ax.set_yticks(range(len(feature_names)))
    ax.set_yticklabels(feature_names, fontsize=11)
    ax.set_xlabel('Timestep (5-min intervals)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name}: Temporal Feature Importance\n(60-min history for 30-min forecast)', 
                 fontsize=13, fontweight='bold', pad=15)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Mean |SHAP| Value', fontsize=11, fontweight='bold')
    
    tick_positions = range(0, n_timesteps, 3)
    tick_labels = [f'-{(n_timesteps-i)*5}min' for i in tick_positions]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=9)
    
    plt.tight_layout()
    filename = f'shap_{model_name.lower().replace(" ", "_")}_temporal.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")

# ============================================================================
# VISUALIZATION 3: TEMPORAL WINDOW COMPARISON
# ============================================================================

if len(results) > 1:
    fig, axes = plt.subplots(len(results), 1, figsize=(14, 5*len(results)))
    if len(results) == 1:
        axes = [axes]
    
    x = np.arange(len(feature_names))
    width = 0.25
    
    for idx, (model_name, data) in enumerate(results.items()):
        ax = axes[idx]
        
        bars1 = ax.bar(x - width, data['temporal_windows']['distant'], width, 
                      label='Distant (0-20 min)', color='lightblue', 
                      edgecolor='black', linewidth=0.5)
        bars2 = ax.bar(x, data['temporal_windows']['middle'], width, 
                      label='Middle (20-40 min)', color='steelblue', 
                      edgecolor='black', linewidth=0.5)
        bars3 = ax.bar(x + width, data['temporal_windows']['recent'], width, 
                      label='Recent (40-60 min)', color='darkblue', 
                      edgecolor='black', linewidth=0.5)
        
        ax.set_ylabel('Normalized Importance', fontsize=11, fontweight='bold')
        ax.set_xlabel('Features', fontsize=11, fontweight='bold')
        ax.set_title(f'{model_name}: Importance by Time Window', 
                    fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=10)
        ax.legend(fontsize=10, loc='upper right')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.suptitle('Temporal Window Analysis Across Models', 
                fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('shap_temporal_windows_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: shap_temporal_windows_comparison.png")

# ============================================================================
# VISUALIZATION 4: SUMMARY PLOTS FOR EACH MODEL
# ============================================================================

for model_name, data in results.items():
    shap_values_reshaped = data['shap_values'].reshape(-1, len(feature_names))
    explain_data_reshaped = explain_data.reshape(-1, len(feature_names))
    
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values_reshaped, explain_data_reshaped, 
                      feature_names=feature_names, show=False, 
                      max_display=len(feature_names))
    plt.title(f'{model_name}: SHAP Summary Plot\n(Red = High Value, Blue = Low Value)', 
              fontsize=13, fontweight='bold', pad=20)
    plt.tight_layout()
    filename = f'shap_{model_name.lower().replace(" ", "_")}_summary.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")

# ============================================================================
# FEATURE IMPORTANCE RANKINGS
# ============================================================================

print(f"\n{'='*70}")
print("FEATURE IMPORTANCE RANKINGS")
print(f"{'='*70}")

for model_name, data in results.items():
    print(f"\n{model_name}:")
    print("-" * 70)
    
    indices = np.argsort(data['mean_abs_shap'])[::-1]
    for rank, idx in enumerate(indices, 1):
        print(f"  {rank}. {feature_names[idx]:20s} {data['mean_abs_shap'][idx]:.4f} "
              f"({data['mean_abs_shap'][idx]*100:.2f}%)")

# ============================================================================
# CROSS-MODEL COMPARISON TABLE
# ============================================================================

if len(results) > 1:
    print(f"\n{'='*70}")
    print("CROSS-MODEL FEATURE IMPORTANCE COMPARISON")
    print(f"{'='*70}")
    print(f"\n{'Feature':<20}", end="")
    for model_name in results.keys():
        print(f"{model_name:>15}", end="")
    print()
    print("-" * (20 + 15 * len(results)))
    
    for i, fname in enumerate(feature_names):
        print(f"{fname:<20}", end="")
        for model_name in results.keys():
            importance = results[model_name]['mean_abs_shap'][i]
            print(f"{importance:>15.4f}", end="")
        print()
    
    # Feature group analysis
    print(f"\n{'='*70}")
    print("FEATURE GROUP ANALYSIS")
    print(f"{'='*70}")
    
    direct_features = ['CGM', 'Carbs', 'Bolus Insulin', 'Heart Rate', 'Steps']
    derivative_features = ['CGM Derivative', 'Carbs Derivative']
    
    for model_name, data in results.items():
        print(f"\n{model_name}:")
        
        direct_indices = [i for i, name in enumerate(feature_names) if name in direct_features]
        derivative_indices = [i for i, name in enumerate(feature_names) if name in derivative_features]
        
        direct_importance = sum(data['mean_abs_shap'][i] for i in direct_indices)
        derivative_importance = sum(data['mean_abs_shap'][i] for i in derivative_indices)
        
        print(f"  Direct Measurements:  {direct_importance:.4f} ({direct_importance*100:.2f}%)")
        print(f"  Derivatives:          {derivative_importance:.4f} ({derivative_importance*100:.2f}%)")

# ============================================================================
# SAVE RESULTS
# ============================================================================

shap_results = {
    'feature_names': feature_names,
    'models': results,
    'explain_data': explain_data,
    'background_data': background_data
}

print(f"\n{'='*70}")
print("SHAP ANALYSIS COMPLETE!")
print(f"{'='*70}")
print(f"""
Analyzed models: {', '.join(results.keys())}

Generated visualizations:
  - shap_models_comparison.png (comparison across all models)
  - shap_[model]_importance.png (individual importance plots)
  - shap_[model]_temporal.png (temporal heatmaps)
  - shap_[model]_summary.png (SHAP summary plots)
  - shap_temporal_windows_comparison.png (time window analysis)

Results stored in 'shap_results' dictionary.

Key Insights:
  - Compare which features matter most across different architectures
  - Identify if physics-informed models rely more on certain features
  - Understand temporal dynamics for each model type
  - See how hybrid approaches balance data-driven vs physics-based features
""")

# ============================================================================
# TROUBLESHOOTING GUIDE
# ============================================================================

print(f"{'='*70}")
print("TROUBLESHOOTING GUIDE")
print(f"{'='*70}")

if 'Hybrid' not in results and ANALYZE_HYBRID:
    print("""
HYBRID MODEL ISSUE:
If the Hybrid model analysis failed, it's likely because the 'bergman' 
variable was overwritten by PINN training.

SOLUTION - Add this code RIGHT AFTER training the Hybrid model:
    
    # Save Hybrid model components before PINN training
    hybrid_bergman = bergman  # Save the NumPy-based Bergman
    hybrid_lstm = lstm_corrector  # Save the LSTM corrector
    
Then, when running SHAP analysis, the code will use 'hybrid_bergman' 
instead of 'bergman'.

Alternatively, just re-run the Hybrid training code to recreate the 
'bergman' and 'lstm_corrector' variables before running this analysis.
""")

print("""
GENERAL TIPS:
1. Train models in any order, but be aware that variable names may overlap
2. For Hybrid: Ensure 'bergman' and 'lstm_corrector' exist and are correct types
3. For PINN: Ensure 'pinn_model' exists
4. For LSTM: Ensure 'lstm_model' exists
5. If analyzing multiple models, consider saving each with unique names

RECOMMENDED WORKFLOW:
# After training each model, save with unique names:
lstm_trained = lstm_model
hybrid_bergman_model = bergman
hybrid_lstm_model = lstm_corrector  
pinn_trained = pinn_model

# Then update variable names in SHAP analysis configuration section
""")

In [ ]:
"""
SHAP Feature Importance Analysis for Blood Glucose Models
Analyzes three models: LSTM, Hybrid Residual Corrector, PINN
Features: CGM, carbs, bolus, heartrate, steps, CGM_derivative, carbs_derivative
"""

import numpy as np
import matplotlib.pyplot as plt
import shap

# ============================================================================
# CONFIGURATION
# ============================================================================

# Define feature names based on data processing
feature_names = [
    'CGM',
    'Carbs',
    'Bolus Insulin',
    'Heart Rate',
    'Steps',
    'CGM Derivative',
    'Carbs Derivative'
]

# Models to analyze (set to True for models you want to analyze)
ANALYZE_LSTM = True
ANALYZE_HYBRID = True
ANALYZE_PINN = True

# SHAP settings
BACKGROUND_SIZE = 30
EXPLAIN_SIZE = 1
NSAMPLES = 10

print("="*70)
print("SHAP FEATURE IMPORTANCE ANALYSIS")
print("="*70)
print(f"Features: {feature_names}")
print(f"Background samples: {BACKGROUND_SIZE}")
print(f"Samples to explain: {EXPLAIN_SIZE}")

# ============================================================================
# SELECT DATA FOR SHAP ANALYSIS
# ============================================================================

# Select background dataset
background_indices = np.random.choice(X_train.shape[0], BACKGROUND_SIZE, replace=False)
background_data = X_train[background_indices]

# Select samples to explain
explain_indices = np.random.choice(X_val.shape[0], EXPLAIN_SIZE, replace=False)
explain_data = X_val[explain_indices]

print(f"Input shape: {explain_data.shape}")

# Store original dimensions
n_timesteps = explain_data.shape[1]
n_features = explain_data.shape[2]

# ============================================================================
# PREDICTION FUNCTIONS FOR EACH MODEL
# ============================================================================

def create_lstm_predict_fn(model):
    """Create prediction function for vanilla LSTM"""
    def predict_fn(X):
        if len(X.shape) == 2:
            n_samples = X.shape[0]
            X = X.reshape(n_samples, n_timesteps, n_features)
        preds = model.predict(X, verbose=0, batch_size=32)
        return np.mean(preds, axis=1, keepdims=True)
    return predict_fn


def create_hybrid_predict_fn(bergman, lstm_corrector, val_subjects_dict):
    """
    Create prediction function for Hybrid Residual Corrector
    Note: This requires access to subject data for bolus/carbs sequences
    """
    # Pre-compute all needed sequences from validation data
    all_G0 = []
    all_bolus = []
    all_carbs = []
    
    for sid in val_subjects_dict.keys():
        subj_data = val_subjects_dict[sid]
        seq_dict = val_sequences[sid]
        n_samples = seq_dict['X'].shape[0]
        
        if n_samples == 0:
            continue
        
        for idx in range(n_samples):
            window_start = N_IN + idx
            G_init = subj_data['CGM_smoothed'].values[window_start - 1]
            bolus = subj_data['bolus'].values[window_start:window_start + N_OUT]
            carbs = subj_data['carbs'].values[window_start:window_start + N_OUT]
            
            all_G0.append(G_init)
            all_bolus.append(bolus)
            all_carbs.append(carbs)
    
    # Select only the ones corresponding to explain_indices
    G0_array = np.array([all_G0[i] for i in explain_indices])
    bolus_array = np.array([all_bolus[i] for i in explain_indices])
    carbs_array = np.array([all_carbs[i] for i in explain_indices])
    
    def predict_fn(X):
        if len(X.shape) == 2:
            n_samples = X.shape[0]
            X = X.reshape(n_samples, n_timesteps, n_features)
        
        # Get Bergman predictions
        predictions = []
        for i in range(X.shape[0]):
            berg_pred = bergman.simulate(G0_array[i], bolus_array[i], carbs_array[i])
            berg_pred_scaled = cgm_scaler.transform(berg_pred.reshape(-1, 1)).flatten()
            
            # Get LSTM residual correction
            x_input = X[i:i+1]
            residual_pred = lstm_corrector.predict(x_input, verbose=0)[0]
            
            # Combine
            y_hybrid = berg_pred_scaled + residual_pred
            predictions.append(np.mean(y_hybrid))
        
        return np.array(predictions).reshape(-1, 1)
    
    return predict_fn


def create_pinn_predict_fn(pinn_model):
    """Create prediction function for PINN model"""
    def predict_fn(X):
        if len(X.shape) == 2:
            n_samples = X.shape[0]
            X = X.reshape(n_samples, n_timesteps, n_features)
        
        X_tf = X.astype(np.float32)
        preds_scaled, _ = pinn_model(X_tf, training=False)
        preds_scaled = preds_scaled.numpy()
        return np.mean(preds_scaled, axis=1, keepdims=True)
    
    return predict_fn

# ============================================================================
# SHAP ANALYSIS FUNCTION
# ============================================================================

def compute_shap_analysis(predict_fn, model_name):
    """
    Compute SHAP analysis for a given model
    Returns dictionary with SHAP results
    """
    print(f"\n{'='*70}")
    print(f"Analyzing {model_name}")
    print(f"{'='*70}")
    
    # Flatten data
    background_flat = background_data.reshape(BACKGROUND_SIZE, -1)
    explain_flat = explain_data.reshape(EXPLAIN_SIZE, -1)
    
    # Create explainer
    print("Initializing KernelExplainer...")
    explainer = shap.KernelExplainer(predict_fn, background_flat)
    
    # Compute SHAP values
    print("Computing SHAP values...")
    shap_values_flat = explainer.shap_values(explain_flat, nsamples=NSAMPLES, silent=False)
    
    # Reshape back
    shap_values = shap_values_flat.reshape(EXPLAIN_SIZE, n_timesteps, n_features)
    print(f"SHAP values shape: {shap_values.shape}")
    
    # Aggregate SHAP values
    mean_abs_shap = np.mean(np.abs(shap_values), axis=(0, 1))
    sum_abs_shap_time = np.sum(np.abs(shap_values), axis=1)
    mean_sum_abs_shap = np.mean(sum_abs_shap_time, axis=0)
    
    # Normalize
    mean_abs_shap_norm = mean_abs_shap / np.sum(mean_abs_shap)
    mean_sum_abs_shap_norm = mean_sum_abs_shap / np.sum(mean_sum_abs_shap)
    
    # Temporal importance
    temporal_importance = np.mean(np.abs(shap_values), axis=0)
    
    # Time window analysis
    segment_size = n_timesteps // 3
    recent_importance = np.mean(temporal_importance[-segment_size:], axis=0)
    middle_importance = np.mean(temporal_importance[segment_size:-segment_size], axis=0)
    distant_importance = np.mean(temporal_importance[:segment_size], axis=0)
    
    recent_norm = recent_importance / np.sum(recent_importance)
    middle_norm = middle_importance / np.sum(middle_importance)
    distant_norm = distant_importance / np.sum(distant_importance)
    
    print(f"✓ SHAP analysis complete for {model_name}")
    
    return {
        'shap_values': shap_values,
        'mean_abs_shap': mean_abs_shap_norm,
        'mean_sum_abs_shap': mean_sum_abs_shap_norm,
        'temporal_importance': temporal_importance,
        'temporal_windows': {
            'distant': distant_norm,
            'middle': middle_norm,
            'recent': recent_norm
        }
    }

# ============================================================================
# RUN SHAP ANALYSIS FOR EACH MODEL
# ============================================================================

results = {}

# Analyze LSTM model
if ANALYZE_LSTM:
    try:
        lstm_predict_fn = create_lstm_predict_fn(lstm_model)
        results['LSTM'] = compute_shap_analysis(lstm_predict_fn, "Vanilla LSTM")
    except Exception as e:
        print(f"Error analyzing LSTM: {e}")

# Analyze Hybrid Residual Corrector
if ANALYZE_HYBRID:
    try:
        # Create validation subjects dict
        val_subjects_dict = {sid: train_subjects[sid] for sid in val_ids}
        hybrid_predict_fn = create_hybrid_predict_fn(bergman, lstm_corrector, val_subjects_dict)
        results['Hybrid'] = compute_shap_analysis(hybrid_predict_fn, "Hybrid Residual Corrector")
    except Exception as e:
        print(f"Error analyzing Hybrid model: {e}")
        print("Make sure bergman and lstm_corrector models are trained and available")

# Analyze PINN model
if ANALYZE_PINN:
    try:
        pinn_predict_fn = create_pinn_predict_fn(pinn_model)
        results['PINN'] = compute_shap_analysis(pinn_predict_fn, "PINN")
    except Exception as e:
        print(f"Error analyzing PINN: {e}")
        print("Make sure pinn_model is trained and available")

# ============================================================================
# VISUALIZATION 1: COMPARISON ACROSS MODELS
# ============================================================================

if len(results) > 0:
    print(f"\n{'='*70}")
    print("GENERATING VISUALIZATIONS")
    print(f"{'='*70}")
    
    # Feature importance comparison
    fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 7))
    if len(results) == 1:
        axes = [axes]
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
    y_pos = np.arange(len(feature_names))
    
    for idx, (model_name, data) in enumerate(results.items()):
        ax = axes[idx]
        bars = ax.barh(y_pos, data['mean_abs_shap'], color=colors, 
                       alpha=0.8, edgecolor='black', linewidth=0.5)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(feature_names, fontsize=10)
        ax.set_xlabel('Importance Score', fontsize=11, fontweight='bold')
        ax.set_title(f'{model_name}\nFeature Importance', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        
        # Add values
        for i, bar in enumerate(bars):
            width = bar.get_width()
            ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
                   f'{data["mean_abs_shap"][i]:.3f}',
                   ha='left', va='center', fontsize=9)
    
    plt.suptitle('Feature Importance Comparison Across Models', 
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig('shap_models_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: shap_models_comparison.png")

# ============================================================================
# VISUALIZATION 2: INDIVIDUAL MODEL PLOTS
# ============================================================================

for model_name, data in results.items():
    # Bar plot for each model
    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(y_pos, data['mean_abs_shap'], color=colors, 
                   alpha=0.8, edgecolor='black', linewidth=0.5)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feature_names, fontsize=11)
    ax.set_xlabel('Mean |SHAP| Value (Normalized)', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name}: Feature Importance\nBlood Glucose Forecasting', 
                 fontsize=14, fontweight='bold', pad=15)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
               f'{data["mean_abs_shap"][i]:.3f}',
               ha='left', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    filename = f'shap_{model_name.lower().replace(" ", "_")}_importance.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")
    
    # Temporal heatmap
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(data['temporal_importance'].T, aspect='auto', 
                   cmap='YlOrRd', interpolation='nearest')
    
    ax.set_yticks(range(len(feature_names)))
    ax.set_yticklabels(feature_names, fontsize=11)
    ax.set_xlabel('Timestep (5-min intervals)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name}: Temporal Feature Importance\n(60-min history for 30-min forecast)', 
                 fontsize=13, fontweight='bold', pad=15)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Mean |SHAP| Value', fontsize=11, fontweight='bold')
    
    tick_positions = range(0, n_timesteps, 3)
    tick_labels = [f'-{(n_timesteps-i)*5}min' for i in tick_positions]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=9)
    
    plt.tight_layout()
    filename = f'shap_{model_name.lower().replace(" ", "_")}_temporal.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")

# ============================================================================
# VISUALIZATION 3: TEMPORAL WINDOW COMPARISON
# ============================================================================

if len(results) > 1:
    fig, axes = plt.subplots(len(results), 1, figsize=(14, 5*len(results)))
    if len(results) == 1:
        axes = [axes]
    
    x = np.arange(len(feature_names))
    width = 0.25
    
    for idx, (model_name, data) in enumerate(results.items()):
        ax = axes[idx]
        
        bars1 = ax.bar(x - width, data['temporal_windows']['distant'], width, 
                      label='Distant (0-20 min)', color='lightblue', 
                      edgecolor='black', linewidth=0.5)
        bars2 = ax.bar(x, data['temporal_windows']['middle'], width, 
                      label='Middle (20-40 min)', color='steelblue', 
                      edgecolor='black', linewidth=0.5)
        bars3 = ax.bar(x + width, data['temporal_windows']['recent'], width, 
                      label='Recent (40-60 min)', color='darkblue', 
                      edgecolor='black', linewidth=0.5)
        
        ax.set_ylabel('Normalized Importance', fontsize=11, fontweight='bold')
        ax.set_xlabel('Features', fontsize=11, fontweight='bold')
        ax.set_title(f'{model_name}: Importance by Time Window', 
                    fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=10)
        ax.legend(fontsize=10, loc='upper right')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.suptitle('Temporal Window Analysis Across Models', 
                fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('shap_temporal_windows_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: shap_temporal_windows_comparison.png")

# ============================================================================
# VISUALIZATION 4: SUMMARY PLOTS FOR EACH MODEL
# ============================================================================

for model_name, data in results.items():
    shap_values_reshaped = data['shap_values'].reshape(-1, len(feature_names))
    explain_data_reshaped = explain_data.reshape(-1, len(feature_names))
    
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values_reshaped, explain_data_reshaped, 
                      feature_names=feature_names, show=False, 
                      max_display=len(feature_names))
    plt.title(f'{model_name}: SHAP Summary Plot\n(Red = High Value, Blue = Low Value)', 
              fontsize=13, fontweight='bold', pad=20)
    plt.tight_layout()
    filename = f'shap_{model_name.lower().replace(" ", "_")}_summary.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")

# ============================================================================
# FEATURE IMPORTANCE RANKINGS
# ============================================================================

print(f"\n{'='*70}")
print("FEATURE IMPORTANCE RANKINGS")
print(f"{'='*70}")

for model_name, data in results.items():
    print(f"\n{model_name}:")
    print("-" * 70)
    
    indices = np.argsort(data['mean_abs_shap'])[::-1]
    for rank, idx in enumerate(indices, 1):
        print(f"  {rank}. {feature_names[idx]:20s} {data['mean_abs_shap'][idx]:.4f} "
              f"({data['mean_abs_shap'][idx]*100:.2f}%)")

# ============================================================================
# CROSS-MODEL COMPARISON TABLE
# ============================================================================

if len(results) > 1:
    print(f"\n{'='*70}")
    print("CROSS-MODEL FEATURE IMPORTANCE COMPARISON")
    print(f"{'='*70}")
    print(f"\n{'Feature':<20}", end="")
    for model_name in results.keys():
        print(f"{model_name:>15}", end="")
    print()
    print("-" * (20 + 15 * len(results)))
    
    for i, fname in enumerate(feature_names):
        print(f"{fname:<20}", end="")
        for model_name in results.keys():
            importance = results[model_name]['mean_abs_shap'][i]
            print(f"{importance:>15.4f}", end="")
        print()
    
    # Feature group analysis
    print(f"\n{'='*70}")
    print("FEATURE GROUP ANALYSIS")
    print(f"{'='*70}")
    
    direct_features = ['CGM', 'Carbs', 'Bolus Insulin', 'Heart Rate', 'Steps']
    derivative_features = ['CGM Derivative', 'Carbs Derivative']
    
    for model_name, data in results.items():
        print(f"\n{model_name}:")
        
        direct_indices = [i for i, name in enumerate(feature_names) if name in direct_features]
        derivative_indices = [i for i, name in enumerate(feature_names) if name in derivative_features]
        
        direct_importance = sum(data['mean_abs_shap'][i] for i in direct_indices)
        derivative_importance = sum(data['mean_abs_shap'][i] for i in derivative_indices)
        
        print(f"  Direct Measurements:  {direct_importance:.4f} ({direct_importance*100:.2f}%)")
        print(f"  Derivatives:          {derivative_importance:.4f} ({derivative_importance*100:.2f}%)")

# ============================================================================
# SAVE RESULTS
# ============================================================================

shap_results = {
    'feature_names': feature_names,
    'models': results,
    'explain_data': explain_data,
    'background_data': background_data
}

print(f"\n{'='*70}")
print("SHAP ANALYSIS COMPLETE!")
print(f"{'='*70}")
print(f"""
Analyzed models: {', '.join(results.keys())}

Generated visualizations:
  - shap_models_comparison.png (comparison across all models)
  - shap_[model]_importance.png (individual importance plots)
  - shap_[model]_temporal.png (temporal heatmaps)
  - shap_[model]_summary.png (SHAP summary plots)
  - shap_temporal_windows_comparison.png (time window analysis)

Results stored in 'shap_results' dictionary.

Key Insights:
  - Compare which features matter most across different architectures
  - Identify if physics-informed models rely more on certain features
  - Understand temporal dynamics for each model type
  - See how hybrid approaches balance data-driven vs physics-based features
""")